# Notebook 03+04 — Candidate Universe + Feature Engineering (v11)

**v11 changes:** Barcode auto-derivation (Cell 5B), mac path support, p24 threshold 5000, vectorized `make_detection_df`, batched `bilinear_sample_batch`, gc moved from per-candidate to per-100-candidates, OFFTRAIN/OFFAUG cells removed.

Self-contained. No external file dependencies beyond the raw TIFF data root and
the NB02 annotations parquet. All proposers run on every crop.

## Proposer inventory

### Mentor methods (FROZEN — verbatim from Yang's NB03 v9)
`mentor_v1`, `mentor_v2`, `mentor_v1_k2`, `mentor_v1_k4`, `mentor_v1_tight`
All run on all 5 barcode panels using LoG + panel-consensus logic.

### Starfish BlobDetector (SELF-CONTAINED — v13 configs frozen)
Six variants tuned in barcode_decoding_v13. Runs directly here without reading
any external parquet. Requires `starfish` package.

### Spotiflow (fine-tuned on NB02 TRUE_SPOT annotations)
Dominguez Mantes et al. 2025 *Nature Methods* 22:1495-1504.

### New raw-ADU proposers (P03–P29, tuned, self-contained)
All thresholds in raw ADU. No per-crop percentile normalization.

| ID | Method | R | P |
|---|---|---|---|
| p03_perturb | ON-mean − α·OFF-mean | 0.944 | 0.710 |
| p04_dog_1_2 | DoG σ=(1,2) | 0.977 | 0.622 |
| p04_dog_1_3 | DoG σ=(1,3) | 0.993 | 0.588 |
| p04_dog_1_4 | DoG σ=(1,4) | 0.997 | 0.572 |
| p05_log | LoG multiscale σ=[1.5,2,3] | 0.990 | 0.571 |
| p06_highpass | Gaussian hp σ=2 | 0.997 | 0.526 |
| p10_bright_rescue | Raw local max | 0.977 | 0.608 |
| p24_onmax_raw | ON-max projection | 0.977 | 0.670 |
| p27_perch_min | Per-channel MIN gate | 0.835 | 0.764 |
| p29_dog_multiscale | DoG scale-space max | 0.993 | 0.579 |
| p01_spotiflow | Spotiflow fine-tuned | 0.961 | 0.598 |

## Outputs written to `{REPO_ROOT}/tables/`
- `{RUN_ID}_candidate_raw.parquet`
- `{RUN_ID}_candidate_union.parquet`
- `{RUN_ID}_candidate_union_membership.parquet`
- `{RUN_ID}_candidate_clusters.parquet`
- `{RUN_ID}_candidate_features.parquet`  ← NB04 features, joined to union_id
- `{RUN_ID}_detector_run_summary.parquet`
- `{RUN_ID}_run_manifest.json`


In [ ]:
# ── Cell 1B: STALE OUTPUT CLEANUP ───────────────────────────────────────────
# Deletes NB03/04-derived output TABLES, manifests, and downstream NB05/06/07
# outputs so a fresh rerun cannot accidentally mix artifacts from prior runs.
#
# PRESERVES: detector cache, feature-family cache, and preprocessing cache
# (these are expensive to rebuild and are reused via short-circuit logic).
# ──────────────────────────────────────────────────────────────────────────────
import glob, os, shutil
from pathlib import Path

_REPO = Path.home() / "Desktop" / "spot-detection-meta"
_TABLES = _REPO / "tables"
_MANIFESTS = _REPO / "artifacts" / "manifests"
_OVERLAYS = _REPO / "artifacts" / "reports" / "nb03_candidate_universe" / "overlays"

# Patterns for NB03/04 output tables (NOT caches)
_NB0304_PATTERNS = [
    "nb03_*",
    "nb04_*",
]
# Patterns for downstream NB05/06/07 outputs (stale from prior runs)
_DOWNSTREAM_PATTERNS = [
    "nb05_*",
    "nb06_*",
    "nb07_*",
]

_deleted = []

# Delete matching table files
for pat in _NB0304_PATTERNS + _DOWNSTREAM_PATTERNS:
    if _TABLES.exists():
        for f in _TABLES.glob(pat):
            if f.is_file():
                f.unlink()
                _deleted.append(str(f.relative_to(_REPO)))

# Delete matching manifest files
for pat in _NB0304_PATTERNS + _DOWNSTREAM_PATTERNS:
    if _MANIFESTS.exists():
        for f in _MANIFESTS.glob(pat):
            if f.is_file():
                f.unlink()
                _deleted.append(str(f.relative_to(_REPO)))

# Clear overlay PNGs
if _OVERLAYS.exists():
    for f in _OVERLAYS.glob("*.png"):
        f.unlink()
        _deleted.append(str(f.relative_to(_REPO)))

if _deleted:
    print(f"Deleted {len(_deleted)} stale files:")
    for d in sorted(_deleted):
        print(f"  {d}")
else:
    print("No stale files found — clean slate.")

print()
print("NOTE: Detector cache, feature-family cache, and preprocessing cache are PRESERVED.")
print("      These will be reused via short-circuit logic in downstream cells.")


In [ ]:
# ── Cell 2: CONFIG — edit only this cell ────────────────────────────────────
from pathlib import Path
from datetime import datetime, timezone
import os

# ── Mac: REPO_ROOT is spot-detection-meta on Desktop ─────────────────────────
# No .git required. All output subdirs are created on first run if missing.
REPO_ROOT = Path.home() / "Desktop" / "spot-detection-meta"
for _subdir in [
    "tables",
    "artifacts/manifests",
    "artifacts/reports/nb03_candidate_universe/overlays",
    "artifacts/detector_cache/fullwell_parity_v1",
    "artifacts/feature_family_cache/fullwell_parity_v1",
    "artifacts/preprocessing_cache/fullwell_parity_v1",
    "annotations/crop_registry",
    "annotations/clicked_truth",
]:
    (REPO_ROOT / _subdir).mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)
print("Subdirs created/verified.")



CFG = {
    # ── roots ────────────────────────────────────────────────────────────────
    "repo_root": str(REPO_ROOT),
    "data_root_candidates": [
        # Mac primary
        str(Path.home() / "Desktop" / "spot_detection"),
        str(Path.home() / "Desktop" / "spot_detection" / "spot_detection"),
        # fallbacks
        str(REPO_ROOT / "data"),
        str(Path.home() / "Desktop"),
    ],

    # ── crop/annotation registry auto-discovery (used for QA + downstream NB05) ──
    "crop_registry_search_dirs": [
        str(REPO_ROOT/"annotations"/"crop_registry"),
        str(REPO_ROOT/"tables"),
        str(REPO_ROOT/"artifacts"),
        str(REPO_ROOT),
    ],

    # ── execution mode ────────────────────────────────────────────────────────
    # Main candidate/proposer/feature pipeline runs on ACTUAL full wells.
    # Crop windows are loaded only for QA provenance and for downstream NB05/NB06
    # supervision masking after full-well candidates/features already exist.
    "execution_mode": "full_well",
    "well_allowlist":    ["C8","D8"],
    "crop_id_allowlist": None,
    "max_crops_total":   None,

    # ── panel / barcode definitions ───────────────────────────────────────────
    "panel_to_folder_wl": {
        "C1_638": ("Cycle1_view","638"),
        "C4_638": ("Cycle4_view","638"),
        "C4_561": ("Cycle4_view","561"),
        "C5_638": ("Cycle5_view","638"),
        "C5_561": ("Cycle5_view","561"),
    },
    "mentor_required_panels": ["C1_638","C4_561","C4_638","C5_561","C5_638"],
    "barcode_panels":          ["C1_638","C4_638","C4_561","C5_638","C5_561"],
    "barcode_on_panels":  {"C8":["C1_638","C4_638","C4_561"],
                           "D8":["C5_638","C5_561","C1_638"]},
    "barcode_off_panels": {"C8":["C5_638","C5_561"],
                           "D8":["C4_638","C4_561"]},

    # ── preprocessing (mentor methods + feature maps) ─────────────────────────
    "norm_percentiles":          (1,99),
    "white_local_sigma_px":      6.0,
    "peak_bg_sigma_small_px":    1.2,
    "peak_bg_sigma_large_px":    6.0,
    "z_local_sigma_px":          6.0,
    "highpass_sigma_px":         3.0,
    "rolling_ball_radius_px":    9,
    "opening_recon_radius_px":   7,
    "wavelet_sigmas_px":         [1.0,2.0,4.0],
    "matched_filter_sigma_px":   2.0,
    "radial_symmetry_radius_px": [1,2,3,4],
    "hessian_sigma_px":          2.0,

    # ── shape gate (applied ONLY to rolling_ball + morph_tophat) ─────────────
    "shape_gate": {
        "enabled":        True,
        "patch_radius":   5,
        "sigma_gradient": 1.0,
        "sigma_tensor":   2.0,
        "min_blobness":   0.15,
        "min_peak_snr":   2.0,
    },

    # ── budget ────────────────────────────────────────────────────────────────
    "per_method_panel_budget": 800,
    "per_crop_total_budget":   5000,

    # ── union / cluster ───────────────────────────────────────────────────────
    "merge_radius_px":   5.0,
    "cluster_radius_px": 12.0,

    # ── FROZEN mentor configs ─────────────────────────────────────────────────
    "mentor_v1": {
        "sigma":8.0,"min_distance":1,"threshold_space":"normalized",
        "threshold_561":0.999,"threshold_638":0.999,
        "log_threshold":None,"log_threshold_percentile":70,
        "min_area":1,"consensus_k":3,"consensus_radius":5,
    },
    "mentor_v2": {
        "sigma":8.0,"min_distance":1,"threshold_space":"normalized",
        "threshold_561":0.999,"threshold_638":0.999,
        "consensus_k":3,"consensus_radius":5,
        "exclude_border":True,"auto_norm_percentiles":(1,99),
    },
    "mentor_v1_k2": {
        "sigma":8.0,"min_distance":1,"threshold_space":"normalized",
        "threshold_561":0.999,"threshold_638":0.999,
        "log_threshold":None,"log_threshold_percentile":70,
        "min_area":1,"consensus_k":2,"consensus_radius":5,
    },
    "mentor_v1_k4": {
        "sigma":8.0,"min_distance":1,"threshold_space":"normalized",
        "threshold_561":0.999,"threshold_638":0.999,
        "log_threshold":None,"log_threshold_percentile":70,
        "min_area":1,"consensus_k":4,"consensus_radius":5,
    },
    "mentor_v1_tight": {
        "sigma":8.0,"min_distance":1,"threshold_space":"normalized",
        "threshold_561":0.999,"threshold_638":0.999,
        "log_threshold":None,"log_threshold_percentile":70,
        "min_area":1,"consensus_k":3,"consensus_radius":3,
    },

    # ── Joint-panel Starfish proposers ────────────────────────────────────────
    "starfish_blob_cfg": {
        "min_sigma":1.0,"max_sigma":5.0,"num_sigma":5,
        "threshold":1.5,"overlap":0.5,"log_scale":True,
    },
    "sf_method_A": {
        "name":"sf_A_sumnorm_snr","method":"A","snr_k":8,"min_panels":2,
    },
    "sf_method_D": {
        "name":"sf_D_crosscorr","method":"D",
        "ncc_thresh":0.05,"patch_r":3,"search_r":2,"snr_k":0,
    },
    "sf_method_G": {
        "name":"sf_G_union","method":"G",
        "A_snr":8,"D_ncc":0.05,"D_patch_r":3,"D_search_r":2,
        "regate_snr":5,"merge_r":5,
    },

    # ── Cross-panel registration offsets (for GT QA only) ─────────────────────
    "panel_registration_offsets": {
        "C8": {
            "C1_638": (0.00,  0.00),
            "C4_638": (-0.10, 0.00),
            "C4_561": (-0.10, 0.00),
            "C5_638": (+0.05,-0.15),
            "C5_561": ( 0.00,-0.10),
        },
        "D8": {
            "C1_638": (0.00,  0.00),
            "C4_638": (-0.15,-0.10),
            "C4_561": (-0.30,-0.10),
            "C5_638": (+0.30,-2.55),
            "C5_561": (+0.20,-2.25),
        },
    },

    # ── new raw-ADU proposers ─────────────────────────────────────────────────
    "new_proposers": {
        "p03_perturb":    {"alpha":1.5,"smooth_sigma":1.0,
                           "abs_threshold":2450.0,"min_distance":4},
        "p04_dog_1_2":    {"sigma1":1.0,"sigma2":2.0,
                           "abs_threshold":2500.0,"min_distance":4},
        "p04_dog_1_3":    {"sigma1":1.0,"sigma2":3.0,
                           "abs_threshold":2500.0,"min_distance":4},
        "p04_dog_1_4":    {"sigma1":1.0,"sigma2":4.0,
                           "abs_threshold":2500.0,"min_distance":4},
        "p05_log":        {"sigmas":[1.5,2.0,3.0],
                           "abs_threshold":2300.0,"min_distance":4},
        "p06_highpass":   {"bg_sigma":2.0,
                           "abs_threshold":2500.0,"min_distance":4},
        "p10_bright_rescue":{"use_log1p":False,
                             "abs_threshold":10000.0,"min_distance":4},
        "p24_onmax_raw":  {"abs_threshold":5000.0,"min_distance":4},
        "p27_perch_min":  {"bg_sigma":5.0,
                           "abs_threshold":2300.0,"min_distance":4},
        "p29_dog_multiscale":{
            "dog_pairs":[(1.0,2.0),(1.0,3.0),(1.0,4.0),(1.5,5.0),(2.0,7.0)],
            "abs_threshold":2800.0,"min_distance":4},
    },

    # ── Spotiflow ─────────────────────────────────────────────────────────────
    "spotiflow_pretrained":    "hybiss",
    "spotiflow_prob_threshold": 0.41,
    "spotiflow_min_distance":   3,
    "spotiflow_n_epochs":    50,
    "spotiflow_force_retrain": True,
    "spotiflow_lr":             3e-4,

    # ── feature engineering (NB04) ────────────────────────────────────────────
    "patch_inner_radius_px":  3,
    "patch_ring1_inner_px":   3,
    "patch_ring1_outer_px":   7,
    "patch_ring2_inner_px":   7,
    "patch_ring2_outer_px":   12,
    "log_sigmas_feat":        [2.0,4.0,8.0],
    "psf_fit_radius_px":      6,
    "neighbor_radii_px":      [5.0,10.0,20.0],
    "nn_k":                   3,
    "struct_tensor_sigma":    2.0,
    "hu_patch_radius_px":     5,
    "n_hu_moments":           4,

    # ── outputs ───────────────────────────────────────────────────────────────
    "tables_subdir":    "tables",
    "manifest_subdir":  "artifacts/manifests",
    "write_overlays":   True,

    # ── provenance ────────────────────────────────────────────────────────────
    "project_config_version":         "nb03v11_fullwell_nb04v3",
    "preprocessing_registry_version": "v11_fullwell",
    "detector_registry_version":      "v11_fullwell_rawadu",
    "feature_registry_version":       "v3_fullwell_nb04",
    "annotation_registry_version":    "v6_nb02",
    "matching_registry_version":      "v6_frozen",
}

print("REPO_ROOT:", REPO_ROOT)
print("Execution mode:", CFG["execution_mode"])
print("Wells:", CFG["well_allowlist"])
print("New proposers:", list(CFG["new_proposers"].keys()))


In [ ]:
# ── Cell 3: Imports ──────────────────────────────────────────────────────────
import gc, json, math, os, re, time, hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from typing import Optional

import numpy as np
import pandas as pd
import tifffile
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.ndimage import gaussian_filter, gaussian_laplace
from scipy.signal import correlate2d
from scipy.stats import skew as scipy_skew

from skimage import feature, measure, morphology
from skimage.feature import peak_local_max, canny
from skimage.morphology import disk, binary_dilation, reconstruction
from skimage.restoration import rolling_ball

# ── optional deep-learning ────────────────────────────────────────────────────
try:
    from spotiflow.model import Spotiflow
    HAS_SPOTIFLOW = True; print("spotiflow OK")
except ImportError:
    HAS_SPOTIFLOW = False; print("[warn] spotiflow not installed")

print("Core imports OK")

In [ ]:
# ── Cell 4: Logging / provenance / IO helpers ────────────────────────────────
import pickle

def ts_utc()->str: return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
def log(msg:str)->None: print(f"[{ts_utc()}] {msg}",flush=True)
def ensure_dir(p)->Path: p=Path(p); p.mkdir(parents=True,exist_ok=True); return p

def make_run_id(prefix="nb03")->str:
    t=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    h=hashlib.sha1(f"{t}|{os.getpid()}".encode()).hexdigest()[:8]
    return f"{prefix}_{t}_{h}"

def sha1_text(x:str)->str:
    return hashlib.sha1(x.encode("utf-8")).hexdigest()

def sha1_file(path,chunk=1<<20)->str:
    h=hashlib.sha1()
    with open(path,"rb") as f:
        while True:
            c=f.read(chunk)
            if not c: break
            h.update(c)
    return h.hexdigest()

def safe_parquet(df:pd.DataFrame,path)->Path:
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    try: df.to_parquet(path,index=False); return path
    except Exception as e:
        csv=path.with_suffix(".csv"); df.to_csv(csv,index=False)
        log(f"[warn] parquet failed ({e}), wrote CSV: {csv.name}"); return csv

def write_json(obj,path)->Path:
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2),encoding="utf-8"); return path

def get_git_commit(root:Path)->Optional[str]:
    gd=root/".git"
    if not gd.exists(): return None
    head=gd/"HEAD"
    if not head.exists(): return None
    c=head.read_text().strip()
    if c.startswith("ref:"):
        rp=gd/c.split(" ",1)[1].strip()
        return rp.read_text().strip() if rp.exists() else None
    return c

RUN_ID        = make_run_id("nb03")
TABLES_DIR    = ensure_dir(REPO_ROOT/CFG["tables_subdir"])
MANIFEST_DIR  = ensure_dir(REPO_ROOT/CFG["manifest_subdir"])
OVERLAY_DIR   = ensure_dir(REPO_ROOT/"artifacts"/"reports"/"nb03_candidate_universe"/"overlays")
GIT_COMMIT    = get_git_commit(REPO_ROOT)

DETECTOR_CACHE_DIR = ensure_dir(REPO_ROOT/"artifacts"/"detector_cache"/"fullwell_parity_v1")
FEATURE_CACHE_DIR  = ensure_dir(REPO_ROOT/"artifacts"/"feature_family_cache"/"fullwell_parity_v1")

def _cache_token(x)->str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))

def table_cache_stem(cache_dir:Path, *parts)->Path:
    name="__".join(_cache_token(p) for p in parts if p is not None and str(p)!="")
    return Path(cache_dir)/name

def has_table_cache(cache_dir:Path, *parts)->bool:
    stem=table_cache_stem(cache_dir, *parts)
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()

def read_table_cache(cache_dir:Path, *parts)->pd.DataFrame:
    stem=table_cache_stem(cache_dir, *parts)
    pq=stem.with_suffix(".parquet")
    csv=stem.with_suffix(".csv")
    if pq.exists():
        return pd.read_parquet(pq)
    if csv.exists():
        return pd.read_csv(csv)
    raise FileNotFoundError(f"Missing cache table for {stem.name}")

def write_table_cache(df:pd.DataFrame, cache_dir:Path, *parts)->Path:
    stem=table_cache_stem(cache_dir, *parts)
    return safe_parquet(df, stem.with_suffix(".parquet"))

def _is_box_path(p:Path)->bool:
    parts={str(x).lower() for x in Path(p).parts}
    return "box" in parts

# Panel preprocessing cache directory.
# On Box (Windows): written to Desktop to avoid sync overhead.
# On Mac/local: written alongside repo artifacts.
_BOX_CACHE_DIR = REPO_ROOT/"artifacts"/"preprocessing_cache"/"fullwell_parity_v1"
_DESKTOP_CACHE_DIR = Path.home()/"Desktop"/"spot-detection-meta_runtime_cache"/"preprocessing_cache"/"fullwell_parity_v1"

if _is_box_path(REPO_ROOT):
    CACHE_DIR = ensure_dir(_DESKTOP_CACHE_DIR)
    _PANEL_CACHE_READ_DIRS = [CACHE_DIR]
    if _BOX_CACHE_DIR != CACHE_DIR:
        _PANEL_CACHE_READ_DIRS.append(_BOX_CACHE_DIR)
else:
    # Mac / local: use repo-relative artifacts dir (no Box sync concern)
    CACHE_DIR = ensure_dir(_BOX_CACHE_DIR)
    _PANEL_CACHE_READ_DIRS = [CACHE_DIR]

def _panel_cache_filename(crop_id:str, panel_key:str)->str:
    return f"{crop_id}__{panel_key}.pkl"

def panel_cache_path(crop_id:str, panel_key:str)->Path:
    return CACHE_DIR / _panel_cache_filename(crop_id, panel_key)

def panel_cache_existing_path(crop_id:str, panel_key:str):
    fn=_panel_cache_filename(crop_id, panel_key)
    for root in _PANEL_CACHE_READ_DIRS:
        p=Path(root)/fn
        if p.exists():
            return p
    return None

def save_panel_cache(crop_id:str, panel_key:str, cache_obj:dict)->Path:
    p = panel_cache_path(crop_id, panel_key)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(cache_obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)
    return p

def load_panel_cache(crop_id:str, panel_key:str):
    p = panel_cache_existing_path(crop_id, panel_key)
    if p is None:
        return None
    try:
        with open(p, "rb") as f:
            return pickle.load(f)
    except Exception:
        return None

def panel_cache_has_keys(cache_obj, required_keys)->bool:
    return isinstance(cache_obj, dict) and all(k in cache_obj for k in required_keys)

def panel_cache_stub(crop_id:str, panel_key:str)->dict:
    p = panel_cache_existing_path(crop_id, panel_key)
    if p is None:
        p = panel_cache_path(crop_id, panel_key)
    return {
        "__panel_cache_stub__": True,
        "crop_id": str(crop_id),
        "panel_key": str(panel_key),
        "cache_path": str(p),
    }

def panel_cache_is_stub(obj)->bool:
    return isinstance(obj, dict) and bool(obj.get("__panel_cache_stub__", False))

def get_panel_cache_for_use(crop_id:str, panel_key:str, required_keys=None):
    pmap = CROP_PANEL_CACHE.get(str(crop_id), {}) if "CROP_PANEL_CACHE" in globals() else {}
    obj = pmap.get(panel_key)
    if isinstance(obj, dict) and (not panel_cache_is_stub(obj)):
        cache = obj
    else:
        cache = load_panel_cache(crop_id, panel_key)
    if cache is None:
        return None
    if required_keys is not None and not panel_cache_has_keys(cache, required_keys):
        return None
    return cache

def release_panel_cache(crop_id:str, panel_key:str)->None:
    if "CROP_PANEL_CACHE" not in globals():
        return
    pmap = CROP_PANEL_CACHE.get(str(crop_id))
    if isinstance(pmap, dict) and panel_key in pmap:
        pmap[panel_key] = panel_cache_stub(crop_id, panel_key)

def progress_msg(done:int, total:int, t0:float, label:str)->str:
    elapsed = time.perf_counter() - t0
    rate = done / max(elapsed, 1e-6)
    eta = (total - done) / max(rate, 1e-6)
    return f"{label}: {done}/{total} done  elapsed={elapsed:.1f}s  ETA={eta:.0f}s"

BASE_PROV = {
    "run_id":                         RUN_ID,
    "project_config_version":         CFG["project_config_version"],
    "preprocessing_registry_version": CFG["preprocessing_registry_version"],
    "detector_registry_version":      CFG["detector_registry_version"],
    "feature_registry_version":       CFG["feature_registry_version"],
    "annotation_registry_version":    CFG["annotation_registry_version"],
    "matching_registry_version":      CFG["matching_registry_version"],
    "code_git_commit":                GIT_COMMIT,
    "merge_radius_px":                float(CFG["merge_radius_px"]),
    "truth_match_radius_px":          None,
    "created_at_utc":                 datetime.now(timezone.utc).isoformat(),
}
log(f"RUN_ID = {RUN_ID}")
log(f"Panel cache write dir = {CACHE_DIR}")
if len(_PANEL_CACHE_READ_DIRS) > 1:
    log(f"Panel cache read dirs = {[str(p) for p in _PANEL_CACHE_READ_DIRS]}")

In [ ]:
# ── Cell 5: Data root + QA crop registry + ACTUAL full-well registry discovery ──

def find_data_root()->Path:
    for c in CFG["data_root_candidates"]:
        p=Path(c)
        if not p.exists() or not p.is_dir():
            continue
        # verify it actually contains TIFFs (not just any dir)
        if any(True for _ in p.rglob("*.tiff") if True):
            return p
    raise FileNotFoundError(f"No data root with TIFFs found. Tried: {CFG['data_root_candidates']}")

def find_crop_registry()->Path:
    cands=[]
    for d in CFG["crop_registry_search_dirs"]:
        p=Path(d)
        if not p.exists():
            continue
        for ext in [".parquet",".csv",".yaml",".yml"]:
            cands.extend(p.glob(f"*crop_registry*{ext}"))
    if not cands:
        raise FileNotFoundError("No crop registry found")
    return sorted(cands)[-1]

def load_registry_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if str(path).endswith(".parquet"):
        return pd.read_parquet(path)
    if str(path).endswith(".csv"):
        return pd.read_csv(path)
    if str(path).endswith((".yaml", ".yml")):
        import yaml
        with open(path, "r", encoding="utf-8") as f:
            y = yaml.safe_load(f)
        if isinstance(y, dict) and "records" in y:
            return pd.DataFrame(y["records"])
        raise ValueError(f"Unexpected registry YAML structure: {path}")
    raise ValueError(f"Unsupported registry format: {path}")

def resolve_panel_path(data_root: Path, well_id: str, panel_key: str) -> Path:
    folder, wl = CFG["panel_to_folder_wl"][panel_key]
    p = Path(data_root) / str(well_id) / str(folder) / f"{well_id}_common_Fluorescence_{wl}_nm_Ex.tiff"
    return p

def read_tiff_shape(path: Path) -> tuple[int, int]:
    with tifffile.TiffFile(path) as tf:
        shp = tf.series[0].shape
    if len(shp) == 2:
        return int(shp[0]), int(shp[1])
    if len(shp) >= 3:
        return int(shp[-2]), int(shp[-1])
    raise ValueError(f"Unexpected TIFF shape for {path}: {shp}")

DATA_ROOT = find_data_root()
QA_CROP_REGISTRY_PATH = find_crop_registry()
log(f"DATA_ROOT = {DATA_ROOT}")
log(f"QA_CROP_REGISTRY = {QA_CROP_REGISTRY_PATH.name}")

qa_crop_registry = load_registry_table(QA_CROP_REGISTRY_PATH).copy()
if "dataset_id" not in qa_crop_registry.columns:
    qa_crop_registry["dataset_id"] = qa_crop_registry["well_id"].astype(str)

if CFG["well_allowlist"]:
    qa_crop_registry = qa_crop_registry[qa_crop_registry["well_id"].isin(CFG["well_allowlist"])].reset_index(drop=True)
if CFG["crop_id_allowlist"]:
    qa_crop_registry = qa_crop_registry[qa_crop_registry["crop_id"].isin(CFG["crop_id_allowlist"])].reset_index(drop=True)
if CFG["max_crops_total"]:
    qa_crop_registry = qa_crop_registry.head(CFG["max_crops_total"]).reset_index(drop=True)

for col in ["well_ymin_px","well_ymax_px","well_xmin_px","well_ymax_px","well_xmin_px","well_xmax_px"]:
    if col in qa_crop_registry.columns:
        qa_crop_registry[col] = qa_crop_registry[col].astype(int)

QA_CROP_LOOKUP = {str(r["crop_id"]): r for _, r in qa_crop_registry.iterrows()}
log(f"QA crops/windows: {len(qa_crop_registry)} across {sorted(qa_crop_registry['well_id'].astype(str).unique().tolist())}")

# Build ACTUAL full-well registry from the real TIFF sizes.
full_rows = []
shape_rows = []
for wid in CFG["well_allowlist"]:
    panel_used = None
    panel_path = None
    H = W = None
    last_err = None
    for pk in CFG["mentor_required_panels"]:
        try:
            p = resolve_panel_path(DATA_ROOT, str(wid), pk)
            if p.exists():
                H, W = read_tiff_shape(p)
                panel_used = pk
                panel_path = p
                break
        except Exception as e:
            last_err = e
    assert panel_used is not None, (
        f"No readable panel found for well {wid}. "
        f"Last error: {type(last_err).__name__ if last_err is not None else 'None'}: {last_err}"
    )
    full_rows.append({
        "crop_id": f"fullwell_{wid}",
        "dataset_id": str(wid),
        "well_id": str(wid),
        "well_ymin_px": 0,
        "well_ymax_px": int(H),
        "well_xmin_px": 0,
        "well_xmax_px": int(W),
    })
    shape_rows.append({
        "well_id": str(wid),
        "panel_used": panel_used,
        "panel_path": str(panel_path),
        "shape_y": int(H),
        "shape_x": int(W),
    })

fullwell_registry = pd.DataFrame(full_rows)
fullwell_shape_df = pd.DataFrame(shape_rows)

# Main pipeline registry: ACTUAL full wells only.
# `crop_registry` is kept as a compatibility alias for downstream cells,
# but every execution row is a REAL full-well extent, not a QA crop.
execution_registry = fullwell_registry.copy()
crop_registry = execution_registry.copy()
CROP_REGISTRY_PATH = QA_CROP_REGISTRY_PATH
CROP_LOOKUP = {str(r["crop_id"]): r for _, r in crop_registry.iterrows()}
log(f"Execution registry: {len(crop_registry)} ACTUAL full wells -> {crop_registry['crop_id'].tolist()}")
display(fullwell_shape_df)

# ── Annotations (NB02 TRUE_SPOT clicks) for QA only ──────────────────────────
annotations_df = None
_ann_candidates = [
    REPO_ROOT/CFG["tables_subdir"]/"annotations.parquet",
    REPO_ROOT/"annotations"/"clicked_truth",
]
for _ap in _ann_candidates:
    if _ap.is_file():
        annotations_df = pd.read_parquet(_ap)
        break
    if _ap.is_dir():
        fs = sorted(_ap.glob("*.parquet"))
        if fs:
            annotations_df = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)
            break

if annotations_df is not None:
    _n = int((annotations_df["label"]=="TRUE_SPOT").sum())
    _n_crops = int(annotations_df["crop_id"].nunique()) if "crop_id" in annotations_df.columns else np.nan
    log(f"Annotations: {len(annotations_df)} rows, {_n} TRUE_SPOT across {_n_crops} QA crops")
else:
    log("[warn] No annotations found — Spotiflow will be skipped")

# ── Cross-panel registration GT dicts for QA only (indexed by QA crop ids) ───
PANEL_OFFSETS = CFG.get("panel_registration_offsets", {})

GT_SPOTS_PER_PANEL = {}   # all panels, registration-corrected, QA crop ids
GT_SPOTS_ON_ONLY   = {}   # ON panels only, QA crop ids

if annotations_df is not None:
    _true = annotations_df[annotations_df["label"]=="TRUE_SPOT"].copy()
    _true["crop_id"] = _true["crop_id"].astype(str)

    for _, crow in qa_crop_registry.iterrows():
        cid = str(crow["crop_id"])
        wid = str(crow["well_id"])
        on_panels  = CFG["barcode_on_panels"].get(wid, CFG["barcode_panels"])
        woff = PANEL_OFFSETS.get(wid, {})
        sub  = _true[_true["crop_id"] == cid]

        for pk in CFG["barcode_panels"]:
            panel_rows = sub[sub["panel_key"] == pk] if "panel_key" in sub.columns else sub
            if len(panel_rows) == 0:
                GT_SPOTS_PER_PANEL[(cid, pk)] = np.zeros((0,2), dtype=np.float32)
                continue
            ys = panel_rows["refined_click_crop_y_px"].values.astype(np.float32)
            xs = panel_rows["refined_click_crop_x_px"].values.astype(np.float32)
            coords = np.stack([ys, xs], axis=1)
            _, idx = np.unique(np.round(coords, 1), axis=0, return_index=True)
            coords = coords[np.sort(idx)]
            dy, dx = woff.get(pk, (0.0, 0.0))
            coords = coords - np.array([[dy, dx]], dtype=np.float32)
            GT_SPOTS_PER_PANEL[(cid, pk)] = coords
            if pk in on_panels:
                GT_SPOTS_ON_ONLY[(cid, pk)] = coords

    n_gt = sum(len(v) for v in GT_SPOTS_ON_ONLY.values())
    log(f"GT_SPOTS_ON_ONLY (QA crops): {n_gt} coords across "
        f"{len(set(k[0] for k in GT_SPOTS_ON_ONLY))} QA crops")
    log("GT_SPOTS_PER_PANEL: QA-only dict with registration-corrected panel coordinates")
else:
    log("[warn] No annotations — GT_SPOTS dicts empty")


In [ ]:
# ── Cell 5B: BARCODE AUTO-DERIVATION ────────────────────────────────────────
# Derives barcode_on_panels / barcode_off_panels from raw ADU rankings at a
# quick panel-agnostic blob pass. Runs BEFORE any proposer or feature code.
# Falls back to CFG hardcoded values if derivation fails or confidence is low.
#
# Method:
#   1. For each well, run DoG blob detection on each panel independently
#   2. For each detected candidate, rank panels by raw patch-mean ADU
#   3. The top-K panels (K = expected n_on from CFG) form the predicted barcode
#   4. Aggregate across candidates by majority vote
#   5. If confidence >= threshold, override CFG barcode dicts; else keep hardcoded
#
# This makes the pipeline robust to future datasets where barcoding is unknown,
# and serves as a per-run QC check that barcoding matches expectation.

import numpy as np
from scipy.ndimage import gaussian_filter as _bc_gf
from skimage.feature import peak_local_max as _bc_plm

_BARCODE_DERIVE_MIN_CONFIDENCE = 0.80  # fraction of spots agreeing on assignment
_BARCODE_DERIVE_PATCH_R        = 3     # patch radius for raw ADU mean (pixels)
_BARCODE_DERIVE_DOG_SIGMA1     = 1.0
_BARCODE_DERIVE_DOG_SIGMA2     = 3.0
_BARCODE_DERIVE_DOG_THR        = 1500.0  # raw ADU threshold for quick blobs
_BARCODE_DERIVE_MIN_DISTANCE   = 5
_BARCODE_DERIVE_MAX_CANDIDATES = 500     # cap for speed

BARCODE_DERIVATION_RESULTS = {}  # {well_id: {on_panels, off_panels, confidence, method}}

def _derive_barcode_for_well(wid: str, panels: list, n_on_expected: int) -> dict:
    """Return dict with on_panels, off_panels, confidence, n_candidates."""
    panel_imgs = {}
    for pk in panels:
        try:
            p = get_tiff_path(wid, pk)
            if p.exists():
                panel_imgs[pk] = read_image_2d(p)
        except Exception:
            pass

    if len(panel_imgs) < 2:
        return {"on_panels": None, "off_panels": None, "confidence": 0.0,
                "n_candidates": 0, "method": "failed_no_images"}

    # Union of DoG candidates across all panels
    all_cands = []
    for pk, img in panel_imgs.items():
        resp = np.maximum(_bc_gf(img, _BARCODE_DERIVE_DOG_SIGMA1) -
                          _bc_gf(img, _BARCODE_DERIVE_DOG_SIGMA2), 0.0).astype(np.float32)
        pts = _bc_plm(resp, min_distance=_BARCODE_DERIVE_MIN_DISTANCE,
                      threshold_abs=_BARCODE_DERIVE_DOG_THR, exclude_border=True)
        all_cands.extend(pts.tolist())

    if not all_cands:
        return {"on_panels": None, "off_panels": None, "confidence": 0.0,
                "n_candidates": 0, "method": "failed_no_candidates"}

    # Deduplicate candidates across panels
    all_cands = np.array(all_cands, dtype=np.float32)
    from scipy.spatial import cKDTree as _cKDT
    tree = _cKDT(all_cands)
    used = np.zeros(len(all_cands), dtype=bool)
    deduped = []
    for i in range(len(all_cands)):
        if used[i]: continue
        idxs = tree.query_ball_point(all_cands[i], r=float(_BARCODE_DERIVE_MIN_DISTANCE))
        deduped.append(all_cands[i])
        for j in idxs:
            used[j] = True
    deduped = np.array(deduped, dtype=np.float32)

    # Cap for speed
    if len(deduped) > _BARCODE_DERIVE_MAX_CANDIDATES:
        deduped = deduped[:_BARCODE_DERIVE_MAX_CANDIDATES]

    # For each candidate, rank panels by patch-mean raw ADU
    panel_vote_counts = {pk: 0 for pk in panels}
    n_valid = 0

    for cy, cx in deduped:
        vals = {}
        for pk, img in panel_imgs.items():
            H, W = img.shape
            iy = min(max(int(round(cy)), 0), H - 1)
            ix = min(max(int(round(cx)), 0), W - 1)
            y0 = max(0, iy - _BARCODE_DERIVE_PATCH_R)
            y1 = min(H, iy + _BARCODE_DERIVE_PATCH_R + 1)
            x0 = max(0, ix - _BARCODE_DERIVE_PATCH_R)
            x1 = min(W, ix + _BARCODE_DERIVE_PATCH_R + 1)
            vals[pk] = float(img[y0:y1, x0:x1].mean())

        ranked = sorted(vals.keys(), key=lambda k: vals[k], reverse=True)
        predicted_on = set(ranked[:n_on_expected])
        for pk in predicted_on:
            panel_vote_counts[pk] += 1
        n_valid += 1

    if n_valid == 0:
        return {"on_panels": None, "off_panels": None, "confidence": 0.0,
                "n_candidates": 0, "method": "failed_no_valid"}

    # Assign ON panels = those with vote_count > 50% of candidates
    on_threshold = n_valid * 0.5
    predicted_on_panels = sorted([pk for pk, cnt in panel_vote_counts.items()
                                   if cnt > on_threshold])
    predicted_off_panels = sorted([pk for pk in panels if pk not in predicted_on_panels])

    # Confidence = fraction of candidates where top-K exactly matches predicted_on
    n_correct = 0
    predicted_on_set = set(predicted_on_panels)
    for cy, cx in deduped[:n_valid]:
        vals = {}
        for pk, img in panel_imgs.items():
            H, W = img.shape
            iy = min(max(int(round(cy)), 0), H - 1)
            ix = min(max(int(round(cx)), 0), W - 1)
            y0 = max(0, iy - _BARCODE_DERIVE_PATCH_R)
            y1 = min(H, iy + _BARCODE_DERIVE_PATCH_R + 1)
            x0 = max(0, ix - _BARCODE_DERIVE_PATCH_R)
            x1 = min(W, ix + _BARCODE_DERIVE_PATCH_R + 1)
            vals[pk] = float(img[y0:y1, x0:x1].mean())
        ranked = sorted(vals.keys(), key=lambda k: vals[k], reverse=True)
        if set(ranked[:n_on_expected]) == predicted_on_set:
            n_correct += 1

    confidence = n_correct / max(n_valid, 1)
    return {
        "on_panels":    predicted_on_panels,
        "off_panels":   predicted_off_panels,
        "confidence":   float(confidence),
        "n_candidates": int(n_valid),
        "method":       "raw_adu_patch_mean_ranking",
        "vote_counts":  panel_vote_counts,
    }

log("=" * 70)
log("BARCODE AUTO-DERIVATION")
all_panels = CFG["barcode_panels"]
for wid in CFG["well_allowlist"]:
    n_on_expected = len(CFG["barcode_on_panels"].get(wid, []))
    if n_on_expected == 0:
        log(f"  {wid}: no expected ON panels in CFG — skipping derivation")
        continue

    log(f"  {wid}: deriving barcode from raw ADU (n_on_expected={n_on_expected}) ...")
    t0_bc = time.perf_counter()
    result = _derive_barcode_for_well(wid, all_panels, n_on_expected)
    result["elapsed_sec"] = time.perf_counter() - t0_bc
    BARCODE_DERIVATION_RESULTS[wid] = result

    cfg_on  = sorted(CFG["barcode_on_panels"].get(wid, []))
    cfg_off = sorted(CFG["barcode_off_panels"].get(wid, []))

    if result["on_panels"] is not None and result["confidence"] >= _BARCODE_DERIVE_MIN_CONFIDENCE:
        derived_on  = sorted(result["on_panels"])
        derived_off = sorted(result["off_panels"])
        matches_cfg = (derived_on == cfg_on and derived_off == cfg_off)
        log(f"  {wid}: confidence={result['confidence']:.3f}  n={result['n_candidates']}  "
            f"{'MATCHES CFG' if matches_cfg else 'DIFFERS FROM CFG'}")
        log(f"    derived ON : {derived_on}")
        log(f"    cfg     ON : {cfg_on}")
        if not matches_cfg:
            log(f"  [warn] {wid}: derived barcoding differs from CFG hardcoded values!")
            log(f"    Overriding CFG with derived values. Check your experiment design.")
        # Always override with derived (even if matching — makes pipeline self-consistent)
        CFG["barcode_on_panels"][wid]  = derived_on
        CFG["barcode_off_panels"][wid] = derived_off
    else:
        conf = result.get("confidence", 0.0)
        log(f"  {wid}: derivation confidence={conf:.3f} < threshold={_BARCODE_DERIVE_MIN_CONFIDENCE} "
            f"OR failed (method={result.get('method','?')}) — keeping CFG hardcoded values")
        log(f"    keeping ON : {cfg_on}")

log("Barcode derivation complete.")
log(f"Final barcode_on_panels  = {CFG['barcode_on_panels']}")
log(f"Final barcode_off_panels = {CFG['barcode_off_panels']}")


In [ ]:
# ── Cell 6: Image IO helpers ─────────────────────────────────────────────────

def get_tiff_path(well_id:str, panel_key:str)->Path:
    folder,wl = CFG["panel_to_folder_wl"][panel_key]
    return DATA_ROOT/well_id/folder/f"{well_id}_common_Fluorescence_{wl}_nm_Ex.tiff"

def read_image_2d(path)->np.ndarray:
    arr=np.squeeze(tifffile.imread(str(path)))
    if arr.ndim==3: arr=arr.max(0)
    elif arr.ndim!=2: arr=arr.reshape(arr.shape[-2],arr.shape[-1])
    return arr.astype(np.float32,copy=False)

def load_crop_raw(well_id,panel_key,ymin,ymax,xmin,xmax)->np.ndarray:
    return read_image_2d(get_tiff_path(well_id,panel_key))[ymin:ymax,xmin:xmax].astype(np.float32)

def norm01_fn(img,p_lo=1,p_hi=99):
    lo=float(np.percentile(img,p_lo)); hi=float(np.percentile(img,p_hi))
    if hi<=lo: hi=lo+1.0
    return np.clip((img-lo)/(hi-lo),0,1).astype(np.float32),lo,hi

# verify path structure
log("Verifying panel paths...")
for w in CFG["well_allowlist"][:1]:
    for pk in CFG["mentor_required_panels"][:2]:
        p=get_tiff_path(w,pk)
        log(f"  {w}/{pk}: {'OK' if p.exists() else 'MISSING'} — {p.name}")


In [ ]:
# ── Cell 7: Core detection helpers ───────────────────────────────────────────

def minmax01(v):
    v=np.asarray(v,dtype=np.float32)
    if v.size==0: return v
    lo,hi=float(v.min()),float(v.max())
    if hi<=lo: return np.ones_like(v)
    return ((v-lo)/(hi-lo)).astype(np.float32)

def sample_scores(resp,coords):
    if len(coords)==0: return np.array([],dtype=np.float32)
    yy=np.clip(np.round(coords[:,0]).astype(int),0,resp.shape[0]-1)
    xx=np.clip(np.round(coords[:,1]).astype(int),0,resp.shape[1]-1)
    return resp[yy,xx].astype(np.float32)

def robust_quantile(resp,q):
    f=resp[np.isfinite(resp)].astype(np.float32)
    return float(np.quantile(f,q)) if f.size else np.inf

def local_max_points(resp,min_distance=3,threshold_abs=None):
    c=peak_local_max(resp,min_distance=int(min_distance),
                     threshold_abs=threshold_abs,exclude_border=False)
    return c.astype(float) if c.size else np.zeros((0,2),dtype=float)

def nms_greedy(pts,radius):
    if len(pts)<=1: return pts
    keep=[]; used=np.zeros(len(pts),bool)
    for i in range(len(pts)):
        if used[i]: continue
        keep.append(i)
        used|=np.sqrt(np.sum((pts-pts[i])**2,axis=1))<radius
    return pts[keep]

def safe_div(a,b,fallback=np.nan):
    try:
        if b==0 or not np.isfinite(b): return fallback
        return float(a)/float(b)
    except: return fallback

def make_detection_df(
    coords_yx,scores_raw,*,
    run_id,dataset_id,well_id,crop_id,
    method_id,method_family,source_view_id,
    score_type,score_is_calibrated,sigma_px,
    timing_sec,crop_origin_well_y_px,crop_origin_well_x_px,
    image_shape_y,image_shape_x,coord_semantics,
    source_image_fingerprint,
):
    coords=np.asarray(coords_yx,dtype=np.float32).reshape(-1,2)
    scores=np.asarray(scores_raw,dtype=np.float32).reshape(-1)
    if len(coords)!=len(scores):
        raise ValueError(f"coords/scores length mismatch {len(coords)} vs {len(scores)}")
    n = len(coords)
    if n == 0:
        return pd.DataFrame(columns=[
            "raw_detection_id","run_id","dataset_id","well_id","crop_id",
            "method_id","method_family","source_view_id","score_type",
            "score_is_calibrated","detection_score_raw","detection_score_norm",
            "crop_y_px","crop_x_px","well_y_px","well_x_px",
            "coord_frame_primary","coord_units","sigma_px","timing_sec",
            "crop_origin_well_y_px","crop_origin_well_x_px",
            "image_shape_y","image_shape_x","coordinate_semantics",
            "source_image_fingerprint",
            *BASE_PROV.keys(),
        ])
    snorm = minmax01(scores)
    # Vectorized raw_detection_id generation
    prefix = f"{run_id}|{dataset_id}|{well_id}|{crop_id}|{method_id}|{source_view_id}|"
    raw_ids = [sha1_text(f"{prefix}{i:08d}") for i in range(n)]
    df = pd.DataFrame({
        "raw_detection_id":      raw_ids,
        "run_id":                run_id,
        "dataset_id":            dataset_id,
        "well_id":               well_id,
        "crop_id":               crop_id,
        "method_id":             method_id,
        "method_family":         method_family,
        "source_view_id":        source_view_id,
        "score_type":            score_type,
        "score_is_calibrated":   bool(score_is_calibrated),
        "detection_score_raw":   scores.astype(np.float32),
        "detection_score_norm":  snorm.astype(np.float32),
        "crop_y_px":             coords[:,0].astype(np.float32),
        "crop_x_px":             coords[:,1].astype(np.float32),
        "well_y_px":             (crop_origin_well_y_px + coords[:,0]).astype(np.float32),
        "well_x_px":             (crop_origin_well_x_px + coords[:,1]).astype(np.float32),
        "coord_frame_primary":   "well",
        "coord_units":           "pixel",
        "sigma_px":              None if sigma_px is None else float(sigma_px),
        "timing_sec":            float(timing_sec),
        "crop_origin_well_y_px": int(crop_origin_well_y_px),
        "crop_origin_well_x_px": int(crop_origin_well_x_px),
        "image_shape_y":         int(image_shape_y),
        "image_shape_x":         int(image_shape_x),
        "coordinate_semantics":  coord_semantics,
        "source_image_fingerprint": source_image_fingerprint,
    })
    for k, v in BASE_PROV.items():
        df[k] = v
    return df

log("Core helpers defined.")


In [ ]:
# ── Cell 8: Preprocessing map functions (used by mentor methods + features) ───

def zscore_local(img,sigma):
    mu=ndi.gaussian_filter(img,sigma=sigma)
    mu2=ndi.gaussian_filter(img*img,sigma=sigma)
    return ((img-mu)/np.sqrt(np.maximum(mu2-mu*mu,1e-6))).astype(np.float32)

def white_local_fn(img,sigma):
    mu=ndi.gaussian_filter(img,sigma=sigma); r=img-mu
    return (r/np.sqrt(np.maximum(ndi.gaussian_filter(r*r,sigma=sigma),1e-6))).astype(np.float32)

def peak_bg_fn(img,ss,sl):
    return (ndi.gaussian_filter(img,ss)-ndi.gaussian_filter(img,sl)).astype(np.float32)

def highpass_fn(img,sigma):
    return (img-ndi.gaussian_filter(img,sigma=sigma)).astype(np.float32)

def local_mad_map_fn(img,radius=5):
    s=2*radius+1; med=ndi.median_filter(img,size=s)
    return (ndi.median_filter(np.abs(img-med),size=s)/0.6745+1e-6).astype(np.float32)

def white_transform_fn(img,sigma):
    hp=highpass_fn(img,sigma); mad=local_mad_map_fn(hp,max(2,int(round(sigma))))
    return np.clip(hp/mad,-8,8).astype(np.float32)

def opening_recon_residual_fn(img,radius):
    se=disk(int(radius)); er=morphology.erosion(img,se)
    return (img-reconstruction(er,img,method="dilation")).astype(np.float32)

def rolling_ball_residual_fn(img,radius):
    return (img-rolling_ball(img,radius=radius)).astype(np.float32)

def atrous_wavelet_bandpass_fn(img,sigmas):
    acc=np.zeros_like(img,dtype=np.float32); prev=img.astype(np.float32)
    for s in sigmas:
        blur=ndi.gaussian_filter(img,sigma=s).astype(np.float32)
        acc+=np.maximum(prev-blur,0); prev=blur
    return acc

def hessian_blobness_map_fn(img,sigma):
    sm=ndi.gaussian_filter(img.astype(np.float32),sigma=sigma)
    g=np.gradient
    Ixx=g(g(sm,axis=1),axis=1); Iyy=g(g(sm,axis=0),axis=0)
    Ixy=g(g(sm,axis=0),axis=1)
    tr=Ixx+Iyy; det=Ixx*Iyy-Ixy*Ixy
    disc=np.maximum(tr*tr-4*det,0.0)
    return np.abs(0.5*(tr+np.sqrt(disc))*0.5*(tr-np.sqrt(disc))).astype(np.float32)

def radial_symmetry_map_fn(img,radii=(1,2,3,4)):
    img=img.astype(np.float32)
    gy,gx=np.gradient(ndi.gaussian_filter(img,sigma=1.0))
    mag=np.sqrt(gx*gx+gy*gy)+1e-6; gxn=gx/mag; gyn=gy/mag
    H,W=img.shape; ys,xs=np.indices((H,W))
    accum=np.zeros((H,W),dtype=np.float32)
    for r in radii:
        py=np.clip(np.round(ys+r*gyn).astype(int),0,H-1)
        px=np.clip(np.round(xs+r*gxn).astype(int),0,W-1)
        np.add.at(accum,(py.ravel(),px.ravel()),mag.ravel())
    return ndi.gaussian_filter(accum,sigma=1.0).astype(np.float32)

def normalized_log_fn(img,sigma):
    return (-(sigma**2)*ndi.gaussian_laplace(img.astype(np.float32),sigma=sigma)).astype(np.float32)

def matched_filter_response_fn(img,sigma):
    rad=int(max(3,round(4*sigma))); yy,xx=np.mgrid[-rad:rad+1,-rad:rad+1]
    g=np.exp(-(xx*xx+yy*yy)/(2*sigma*sigma)).astype(np.float32)
    g/=np.sqrt(np.sum(g*g))+1e-6
    return ndi.correlate((img.astype(np.float32)-float(np.mean(img))),g,mode="nearest").astype(np.float32)

def compute_preprocessing_cache(crop:np.ndarray)->dict:
    plo,phi=CFG["norm_percentiles"]
    norm_img,lo,hi=norm01_fn(crop,plo,phi)
    return {
        "raw":                    crop,
        "norm01":                 norm_img,
        "norm_limits":            (lo,hi),
        "white_local":            white_local_fn(norm_img,CFG["white_local_sigma_px"]),
        "peak_bg":                peak_bg_fn(norm_img,CFG["peak_bg_sigma_small_px"],CFG["peak_bg_sigma_large_px"]),
        "z_local":                zscore_local(norm_img,CFG["z_local_sigma_px"]),
        "highpass":               highpass_fn(norm_img,CFG["highpass_sigma_px"]),
        "local_mad":              local_mad_map_fn(norm_img,radius=5),
        "white":                  white_transform_fn(norm_img,CFG["white_local_sigma_px"]),
        "rolling_ball_residual":  rolling_ball_residual_fn(norm_img,CFG["rolling_ball_radius_px"]),
        "opening_recon_residual": opening_recon_residual_fn(norm_img,CFG["opening_recon_radius_px"]),
        "atrous_wavelet_bandpass":atrous_wavelet_bandpass_fn(norm_img,CFG["wavelet_sigmas_px"]),
        "radial_symmetry_map":    radial_symmetry_map_fn(norm_img,CFG["radial_symmetry_radius_px"]),
        "hessian_blobness":       hessian_blobness_map_fn(norm_img,CFG["hessian_sigma_px"]),
        "log_normalized":         normalized_log_fn(norm_img,2.0),
        "matched_filter":         matched_filter_response_fn(norm_img,CFG["matched_filter_sigma_px"]),
        "image_shape_y":          crop.shape[0],
        "image_shape_x":          crop.shape[1],
    }

log("Preprocessing functions defined.")


In [ ]:
# ── Cell 9: Mentor method helpers (FROZEN — verbatim from NB03 v9) ────────────
# DO NOT MODIFY. These reproduce Yang's original detection logic exactly.

def quick_percentile_limits_np(img,p_lo=1,p_hi=99,step=16):
    s=img[::step,::step]; vmin=float(np.percentile(s,p_lo)); vmax=float(np.percentile(s,p_hi))
    if vmax<=vmin: vmax=vmin+1.0
    return vmin,vmax

def log_spot_mask_mentor_v1(img,sigma=3.0,min_distance=12,threshold=0.5,
    threshold_space="normalized",p_lo=1,p_hi=99,
    log_threshold=None,log_threshold_percentile=90,min_area=0):
    img=img.astype(np.float32,copy=False)
    if threshold_space=="normalized":
        work,_,_=norm01_fn(img,p_lo=p_lo,p_hi=p_hi); thr=float(threshold)
    else:
        work=img; thr=float(threshold)
    clog=-ndi.gaussian_laplace(work,sigma=sigma)
    size=2*int(min_distance)+1
    maxf=ndi.maximum_filter(clog,size=size,mode="nearest")
    peaks=(clog==maxf)&(work>thr)
    if not peaks.any(): return np.zeros_like(img,dtype=bool),clog.astype(np.float32)
    log_thr=(np.percentile(clog[peaks],log_threshold_percentile)
             if log_threshold is None else float(log_threshold))
    mask=peaks&(clog>=log_thr)
    if min_area>0:
        lbl=measure.label(mask); km=np.zeros_like(mask,dtype=bool)
        for r in measure.regionprops(lbl):
            if r.area>=min_area: km[lbl==r.label]=True
        mask=km
    return mask,clog.astype(np.float32)

def log_peaks_2d_mentor_v2(img2d,sigma=3.0,min_distance=12,threshold=None,
    exclude_border=True,*,threshold_space="raw",norm_limits=None,auto_norm_percentiles=(1,99)):
    img2d=np.asarray(img2d,dtype=np.float32)
    if threshold_space.lower()=="normalized":
        vmin,vmax=(quick_percentile_limits_np(img2d,*auto_norm_percentiles)
                   if norm_limits is None else norm_limits)
        work=np.clip((img2d-vmin)/max(vmax-vmin,1e-6),0,1).astype(np.float32)
    else:
        work=img2d
    size=2*int(min_distance)+1; selem=np.ones((size,size),dtype=bool)
    clog=-ndi.gaussian_laplace(work,sigma=sigma)
    pm=(clog==ndi.maximum_filter(clog,footprint=selem,mode="nearest"))
    thr=(np.percentile(work[::max(1,int(min_distance)),::max(1,int(min_distance))],75)
         if threshold is None else float(threshold))
    pm=pm&(work>thr)
    if exclude_border and min(pm.shape)>2*int(min_distance):
        b=int(min_distance); pm[:b,:]=False; pm[-b:,:]=False; pm[:,:b]=False; pm[:,-b:]=False
    return pm.astype(bool),clog.astype(np.float32)

def consensus_from_masks(masks,tolerance=2,min_hits=4):
    se=disk(int(tolerance)) if tolerance and tolerance>0 else None
    dil=[binary_dilation(m,se) if se is not None else m for m in masks]
    count=np.zeros_like(masks[0],dtype=np.uint16)
    for m in dil: count+=m.astype(np.uint16)
    lbl=measure.label(count>=int(min_hits))
    pts=[(float(r.centroid[0]),float(r.centroid[1])) for r in measure.regionprops(lbl)]
    return (count>=min_hits),np.asarray(pts,dtype=np.float32).reshape(-1,2),count.astype(np.float32)

def shape_gate_filter(img,coords,scores,sg):
    if not sg["enabled"] or len(coords)==0: return coords,scores
    H,W=img.shape; keep=np.ones(len(coords),bool)
    ism=ndi.gaussian_filter(img.astype(np.float64),sg["sigma_gradient"])
    gy,gx=np.gradient(ism)
    Jxx=ndi.gaussian_filter(gx*gx,sg["sigma_tensor"])
    Jyy=ndi.gaussian_filter(gy*gy,sg["sigma_tensor"])
    Jxy=ndi.gaussian_filter(gx*gy,sg["sigma_tensor"])
    for i,(cy,cx) in enumerate(coords):
        iy,ix=int(round(cy)),int(round(cx))
        y0,y1=max(0,iy-sg["patch_radius"]),min(H,iy+sg["patch_radius"]+1)
        x0,x1=max(0,ix-sg["patch_radius"]),min(W,ix+sg["patch_radius"]+1)
        if (y1-y0)<3 or (x1-x0)<3: keep[i]=False; continue
        jxx=float(np.mean(Jxx[y0:y1,x0:x1])); jyy=float(np.mean(Jyy[y0:y1,x0:x1]))
        jxy=float(np.mean(Jxy[y0:y1,x0:x1]))
        tr=jxx+jyy; det=jxx*jyy-jxy*jxy; disc=max(tr*tr-4*det,0.0)
        lm=max(abs(0.5*(tr+math.sqrt(disc))),abs(0.5*(tr-math.sqrt(disc))))
        ln=min(abs(0.5*(tr+math.sqrt(disc))),abs(0.5*(tr-math.sqrt(disc))))
        if lm<1e-12 or ln/lm<sg["min_blobness"]: keep[i]=False; continue
        p=img[y0:y1,x0:x1]; bg=float(np.median(p)); pv=float(img[min(iy,H-1),min(ix,W-1)])
        if bg>0 and pv/bg<sg["min_peak_snr"]: keep[i]=False
    return coords[keep],scores[keep]

def run_mentor_v1_cfg(cache_by_panel,vcfg):
    masks=[]
    for pk in CFG["mentor_required_panels"]:
        c=cache_by_panel.get(pk)
        if c is None: continue
        thr=vcfg["threshold_561"] if pk.endswith("561") else vcfg["threshold_638"]
        m,_=log_spot_mask_mentor_v1(
            c["raw"],sigma=vcfg["sigma"],min_distance=vcfg["min_distance"],
            threshold=thr,threshold_space=vcfg["threshold_space"],
            p_lo=CFG["norm_percentiles"][0],p_hi=CFG["norm_percentiles"][1],
            log_threshold=vcfg.get("log_threshold"),
            log_threshold_percentile=vcfg.get("log_threshold_percentile",70),
            min_area=vcfg.get("min_area",0))
        masks.append(m)
    if len(masks)<vcfg["consensus_k"]:
        return np.zeros((0,2),dtype=np.float32),np.array([],dtype=np.float32),None,None
    cm,coords,count=consensus_from_masks(masks,tolerance=vcfg["consensus_radius"],min_hits=vcfg["consensus_k"])
    scores=sample_scores(count,coords) if len(coords) else np.array([],dtype=np.float32)
    return coords,scores,count,cm

def run_mentor_v2_cfg(cache_by_panel,vcfg):
    masks=[]
    for pk in CFG["mentor_required_panels"]:
        c=cache_by_panel.get(pk)
        if c is None: continue
        thr=vcfg["threshold_561"] if pk.endswith("561") else vcfg["threshold_638"]
        nl=c["norm_limits"] if vcfg["threshold_space"]=="normalized" else None
        m,_=log_peaks_2d_mentor_v2(
            c["raw"],sigma=vcfg["sigma"],min_distance=vcfg["min_distance"],
            threshold=thr,exclude_border=vcfg.get("exclude_border",False),
            threshold_space=vcfg["threshold_space"],norm_limits=nl,
            auto_norm_percentiles=vcfg.get("auto_norm_percentiles",(1,99)))
        masks.append(m)
    if len(masks)<vcfg["consensus_k"]:
        return np.zeros((0,2),dtype=np.float32),np.array([],dtype=np.float32),None,None
    cm,coords,count=consensus_from_masks(masks,tolerance=vcfg["consensus_radius"],min_hits=vcfg["consensus_k"])
    scores=sample_scores(count,coords) if len(coords) else np.array([],dtype=np.float32)
    return coords,scores,count,cm

log("Mentor helpers defined (FROZEN).")


In [ ]:
# ── Cell 10: New raw-ADU proposer functions (self-contained) ─────────────────
# Each function loads its own images from disk.
# No external files, no shared state, no per-crop percentile normalization.

def _raw_imgs(well_id,panel_keys,ymin,ymax,xmin,xmax):
    out={}
    for pk in panel_keys:
        try: out[pk]=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax)
        except Exception as e: pass
    return out

def _local_max_above(resp,abs_thr,min_dist,max_pts=2000):
    thr=abs_thr if abs_thr>0 else None
    coords=local_max_points(resp,min_distance=min_dist,threshold_abs=thr)
    if len(coords)==0: return coords,np.array([],dtype=np.float32)
    scores=sample_scores(resp,coords)
    if len(coords)>max_pts:
        idx=np.argsort(-scores)[:max_pts]; coords,scores=coords[idx],scores[idx]
    return coords,scores

def _panel_union_nms(pts_list,radius):
    if not pts_list: return np.zeros((0,2),dtype=np.float32)
    all_pts=np.vstack(pts_list).astype(np.float32)
    return nms_greedy(all_pts,radius)

# ── P03: Perturb-CODEX multi-channel consensus ──────────────────────────────
# ON-mean − alpha*OFF-mean after Gaussian smoothing. Operationalises the
# multi-channel barcode specificity concept (Chen et al. 2015 MERFISH).
def detect_p03_perturb(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); off_p=CFG["barcode_off_panels"].get(well_id,[])
    sm=cfg["smooth_sigma"]; alpha=cfg["alpha"]
    on_imgs=[]; off_imgs=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax)
            on_imgs.append(gaussian_filter(r,sm) if sm>0 else r)
        except: pass
    for pk in off_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax)
            off_imgs.append(gaussian_filter(r,sm) if sm>0 else r)
        except: pass
    if not on_imgs: return np.zeros((0,2),dtype=np.float32),np.array([],dtype=np.float32)
    on_m=np.mean(np.stack(on_imgs),axis=0).astype(np.float32)
    off_m=(np.mean(np.stack(off_imgs),axis=0).astype(np.float32) if off_imgs else np.zeros_like(on_m))
    score=on_m-alpha*off_m
    return _local_max_above(score,cfg["abs_threshold"],cfg["min_distance"])

# ── P04: Difference of Gaussians — single sigma pair ───────────────────────
# Marr & Hildreth 1980; FISH-QUANT Mueller 2013.
def detect_p04_dog(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); s1,s2=cfg["sigma1"],cfg["sigma2"]
    pts_list=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            resp=np.maximum(gaussian_filter(r,s1)-gaussian_filter(r,s2),0.0)
            c,_=_local_max_above(resp,cfg["abs_threshold"],cfg["min_distance"])
            if len(c): pts_list.append(c)
        except: pass
    pts=_panel_union_nms(pts_list,cfg["min_distance"])
    return pts,(np.ones(len(pts),dtype=np.float32) if len(pts) else np.array([],dtype=np.float32))

# ── P05: LoG multiscale maximum ─────────────────────────────────────────────
# Lindeberg 1994 IJCV; FISH-QUANT v2.
def detect_p05_log(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); pts_list=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            resp=np.max(np.stack([np.maximum(-(s**2)*gaussian_laplace(r,s),0.0)
                                  for s in cfg["sigmas"]],axis=0),axis=0)
            c,_=_local_max_above(resp,cfg["abs_threshold"],cfg["min_distance"])
            if len(c): pts_list.append(c)
        except: pass
    pts=_panel_union_nms(pts_list,cfg["min_distance"])
    return pts,(np.ones(len(pts),dtype=np.float32) if len(pts) else np.array([],dtype=np.float32))

# ── P06: Gaussian highpass local max ────────────────────────────────────────
def detect_p06_highpass(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); pts_list=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            resp=np.maximum(r-gaussian_filter(r,cfg["bg_sigma"]),0.0)
            c,_=_local_max_above(resp,cfg["abs_threshold"],cfg["min_distance"])
            if len(c): pts_list.append(c)
        except: pass
    pts=_panel_union_nms(pts_list,cfg["min_distance"])
    return pts,(np.ones(len(pts),dtype=np.float32) if len(pts) else np.array([],dtype=np.float32))

# ── P10: Bright rescue — raw intensity local max ─────────────────────────────
def detect_p10_bright_rescue(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); pts_list=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            resp=np.log1p(np.maximum(r,0.0)) if cfg["use_log1p"] else r
            c,_=_local_max_above(resp,cfg["abs_threshold"],cfg["min_distance"])
            if len(c): pts_list.append(c)
        except: pass
    pts=_panel_union_nms(pts_list,cfg["min_distance"])
    return pts,(np.ones(len(pts),dtype=np.float32) if len(pts) else np.array([],dtype=np.float32))

# ── P24: ON-channel max-projection + raw local max ───────────────────────────
# Eng et al. 2019 seqFISH+.
# ── P24: ON-channel max-projection + DoG filter ──────────────────────────────
# Eng et al. 2019 seqFISH+. Max-proj across ON panels, then DoG(1,2) filter.
# Threshold tuned on DoG response in testing notebooks — must match here.
def detect_p24_onmax_raw(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); maxp=None
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            maxp=r if maxp is None else np.maximum(maxp,r)
        except: pass
    if maxp is None: return np.zeros((0,2),dtype=np.float32),np.array([],dtype=np.float32)
    resp=np.maximum(gaussian_filter(maxp,1.0)-gaussian_filter(maxp,2.0),0.0)
    return _local_max_above(resp,cfg["abs_threshold"],cfg["min_distance"])

# ── P27: Per-channel minimum gate (operationalises v13 best feature) ─────────
# Real spots must be bright above background in EVERY ON channel.
# Xia et al. 2019 PNAS; v13 per-channel min signal finding.
def detect_p27_perch_min(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); resids=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            resids.append(np.maximum(r-gaussian_filter(r,cfg["bg_sigma"]),0.0))
        except: pass
    if not resids: return np.zeros((0,2),dtype=np.float32),np.array([],dtype=np.float32)
    score=np.min(np.stack(resids),axis=0).astype(np.float32)
    return _local_max_above(score,cfg["abs_threshold"],cfg["min_distance"])

# ── P29: DoG multiscale scale-space maximum ──────────────────────────────────
# Lindeberg 1994; Lowe 2004 SIFT. Best of all 03B-shortlisted pairs.
def detect_p29_dog_multiscale(well_id,ymin,ymax,xmin,xmax,cfg):
    on_p=CFG["barcode_on_panels"].get(well_id,[]); pts_list=[]
    for pk in on_p:
        try:
            r=load_crop_raw(well_id,pk,ymin,ymax,xmin,xmax).astype(np.float32)
            resp=np.max(np.stack([np.maximum(gaussian_filter(r,s1)-gaussian_filter(r,s2),0.0)
                                   for s1,s2 in cfg["dog_pairs"]],axis=0),axis=0)
            c,_=_local_max_above(resp,cfg["abs_threshold"],cfg["min_distance"])
            if len(c): pts_list.append(c)
        except: pass
    pts=_panel_union_nms(pts_list,cfg["min_distance"])
    return pts,(np.ones(len(pts),dtype=np.float32) if len(pts) else np.array([],dtype=np.float32))

NEW_PROPOSER_DISPATCH = {
    "p03_perturb":      detect_p03_perturb,
    "p04_dog_1_2":      detect_p04_dog,
    "p04_dog_1_3":      detect_p04_dog,
    "p04_dog_1_4":      detect_p04_dog,
    "p05_log":          detect_p05_log,
    "p06_highpass":     detect_p06_highpass,
    "p10_bright_rescue":detect_p10_bright_rescue,
    "p24_onmax_raw":    detect_p24_onmax_raw,
    "p27_perch_min":    detect_p27_perch_min,
    "p29_dog_multiscale":detect_p29_dog_multiscale,
}
DETECTOR_FAMILY = {
    "mentor_v1":"mentor","mentor_v2":"mentor","mentor_v1_k2":"mentor",
    "mentor_v1_k4":"mentor","mentor_v1_tight":"mentor",
    "p03_perturb":"multi_channel","p04_dog_1_2":"dog","p04_dog_1_3":"dog",
    "p04_dog_1_4":"dog","p05_log":"log","p06_highpass":"highpass",
    "p10_bright_rescue":"localmax","p24_onmax_raw":"multi_channel",
    "p27_perch_min":"multi_channel","p29_dog_multiscale":"dog",
    "spotiflow_finetuned":"deep_learning",
    CFG["sf_method_A"]["name"]:"starfish_joint",
    CFG["sf_method_D"]["name"]:"starfish_joint",
    CFG["sf_method_G"]["name"]:"starfish_joint",
}
log("New proposer functions defined.")


In [ ]:
# ── Cell 11: Build preprocessing cache for ACTUAL full wells ─────────────────
# The mentor methods and feature engineering need per-panel maps in the same
# full-well context that will later be used by NB07. We do NOT compute these on
# small QA crops. Crop windows are imposed only later in NB05/NB06.

CROP_PANEL_CACHE        = {}   # {fullwell_crop_id: {panel_key: lightweight_stub}}
CROP_PANEL_FINGERPRINTS = {}   # {(crop_id,panel_key): sha1}
PREPROCESS_TIMINGS      = []

log(f"Building preprocessing cache for {len(crop_registry)} ACTUAL full wells × {len(CFG['mentor_required_panels'])} panels...")

t0_all = time.perf_counter()
for wi, (_, crow) in enumerate(crop_registry.iterrows(), start=1):
    cid = str(crow["crop_id"])
    wid = str(crow["well_id"])
    ymin, ymax = int(crow["well_ymin_px"]), int(crow["well_ymax_px"])
    xmin, xmax = int(crow["well_xmin_px"]), int(crow["well_xmax_px"])
    panel_cache = {}
    log(f"[well {wi}/{len(crop_registry)}] {cid} -> building/loading base preprocessing cache")

    for pj, pk in enumerate(CFG["mentor_required_panels"], start=1):
        t0 = time.perf_counter()
        try:
            cached = load_panel_cache(cid, pk)
            required = ["raw","norm01","white_local","peak_bg","z_local","highpass",
                        "local_mad","white","rolling_ball_residual","opening_recon_residual",
                        "atrous_wavelet_bandpass","radial_symmetry_map","hessian_blobness",
                        "log_normalized","matched_filter","panel_key","file_path","context_mode"]
            if panel_cache_has_keys(cached, required):
                panel_cache[pk] = panel_cache_stub(cid, pk)
                fp = cached.get("file_path")
                if fp is not None and Path(fp).exists():
                    CROP_PANEL_FINGERPRINTS[(cid, pk)] = sha1_file(fp)
                PREPROCESS_TIMINGS.append({
                    "crop_id": cid, "well_id": wid, "panel_key": pk,
                    "context_mode": "full_well", "timing_sec": time.perf_counter() - t0,
                    "cache_status": "disk_hit",
                })
                log(f"  [{pj}/{len(CFG['mentor_required_panels'])}] {pk}: disk cache hit")
                del cached
                gc.collect()
                continue

            p = get_tiff_path(wid, pk)
            if not p.exists():
                log(f"  [{pj}/{len(CFG['mentor_required_panels'])}] {pk}: [skip] file not found")
                continue

            log(f"  [{pj}/{len(CFG['mentor_required_panels'])}] {pk}: reading TIFF + computing base maps")
            full = read_image_2d(p)
            crop = full[ymin:ymax, xmin:xmax].astype(np.float32)
            c = compute_preprocessing_cache(crop)
            c["panel_key"] = pk
            c["file_path"] = str(p)
            c["context_mode"] = "full_well"
            save_panel_cache(cid, pk, c)
            panel_cache[pk] = panel_cache_stub(cid, pk)
            CROP_PANEL_FINGERPRINTS[(cid, pk)] = sha1_file(p)
            del full, crop, c
            gc.collect()

            PREPROCESS_TIMINGS.append({
                "crop_id": cid,
                "well_id": wid,
                "panel_key": pk,
                "context_mode": "full_well",
                "timing_sec": time.perf_counter() - t0,
                "cache_status": "computed_and_saved",
            })
            log(f"     saved -> {panel_cache_path(cid, pk).name}  sec={PREPROCESS_TIMINGS[-1]['timing_sec']:.1f}")
        except Exception as e:
            log(f"  [warn] {cid}/{pk}: {type(e).__name__}: {e}")
            PREPROCESS_TIMINGS.append({
                "crop_id": cid, "well_id": wid, "panel_key": pk,
                "context_mode": "full_well", "timing_sec": time.perf_counter() - t0,
                "cache_status": f"error:{type(e).__name__}",
            })

    CROP_PANEL_CACHE[cid] = panel_cache
    gc.collect()
    log(progress_msg(wi, len(crop_registry), t0_all, f"base preprocessing cache"))

n_cached = sum(len(v) for v in CROP_PANEL_CACHE.values())
log(f"Cache built: {n_cached} fullwell-panels across {len(CROP_PANEL_CACHE)} wells")

In [ ]:

# Reusable detector-cache short-circuit for Starfish methods.
_STARFISH_METHOD_IDS = [
    CFG["sf_method_A"]["name"],
    CFG["sf_method_D"]["name"],
    CFG["sf_method_G"]["name"],
]
_STARFISH_CACHE_READY = all(has_table_cache(DETECTOR_CACHE_DIR, "method_candidates", _mid) for _mid in _STARFISH_METHOD_IDS)

if _STARFISH_CACHE_READY:
    _sf_parts = [read_table_cache(DETECTOR_CACHE_DIR, "method_candidates", _mid) for _mid in _STARFISH_METHOD_IDS]
    starfish_df = pd.concat(_sf_parts, ignore_index=True) if _sf_parts else pd.DataFrame()
    log(f"Starfish detector cache hit: loaded {len(starfish_df)} rows from per-method cache and skipped recomputation.")
else:
    # ── Cell 12: Joint-panel Starfish proposers (Methods A, D, G) ────────────────
    # Replaces the per-panel norm01 approach with three tuned joint-panel methods.
    # All methods:
    #   1. Build sum_norm projection over ON panels from raw ADU (no percentile norm)
    #   2. Run blob_log once on the projection
    #   3. Verify candidates differently per method:
    #      A: per-panel local SNR >= snr_k in >= min_panels ON panels
    #      D: mean pairwise NCC across ON panels >= ncc_thresh (barcode-agnostic)
    #      G: union of A+D survivors, re-gated at loose SNR=regate_snr
    # Detections are strictly from ON-panel joint projection.
    # OFF-channel negatives are injected separately at the end of Cell 14.

    from scipy.signal import correlate2d as _correlate2d

    def _nms_kdtree(pts, radius):
        """O(N log N) NMS. Replaces nms_greedy for large candidate sets."""
        if len(pts) <= 1:
            return pts
        tree = cKDTree(pts)
        kept = np.ones(len(pts), dtype=bool)
        for i in range(len(pts)):
            if not kept[i]:
                continue
            for j in tree.query_ball_point(pts[i], radius):
                if j != i:
                    kept[j] = False
        return pts[kept]

    def _sum_norm_proj(raw_imgs, on_panels):
        """sum_norm joint projection over ON panels. No percentile normalisation.
        Each panel normalised by its own p99 (a scale factor, not cross-crop)."""
        imgs = [raw_imgs[pk] for pk in on_panels if pk in raw_imgs]
        if not imgs:
            return None
        normed = []
        for img in imgs:
            p99 = float(np.percentile(img, 99))
            if p99 <= 0:
                p99 = 1.0
            normed.append(np.clip(img / p99, 0.0, None).astype(np.float32))
        return np.stack(normed, axis=0).sum(axis=0)

    def _local_snr_vec(raw_img, yx, spot_r=2, bg_inner=4, bg_outer=9):
        """Local SNR = (spot_mean - bg_median) / bg_MAD on raw ADU."""
        if len(yx) == 0:
            return np.array([], dtype=np.float32)
        H, W = raw_img.shape
        dy, dx = np.meshgrid(
            np.arange(-bg_outer, bg_outer+1),
            np.arange(-bg_outer, bg_outer+1), indexing="ij")
        dist  = np.sqrt(dy**2 + dx**2)
        smask = dist <= spot_r
        rmask = (dist > bg_inner) & (dist <= bg_outer)
        out   = np.empty(len(yx), dtype=np.float32)
        for i, (y, x) in enumerate(yx):
            yi, xi = int(round(y)), int(round(x))
            y0=max(0,yi-bg_outer); y1=min(H,yi+bg_outer+1)
            x0=max(0,xi-bg_outer); x1=min(W,xi+bg_outer+1)
            patch = raw_img[y0:y1, x0:x1]
            dyo=yi-bg_outer-y0; dxo=xi-bg_outer-x0
            ph, pw = patch.shape
            sm = smask[dyo:dyo+ph, dxo:dxo+pw]
            rm = rmask[dyo:dyo+ph, dxo:dxo+pw]
            sp = patch[sm] if sm.any() else patch.ravel()
            rp = patch[rm] if rm.any() else patch.ravel()
            bg  = float(np.median(rp))
            mad = float(np.median(np.abs(rp - bg))) / 0.6745 + 1e-3
            out[i] = (float(np.mean(sp)) - bg) / mad
        return out

    def _cross_panel_ncc(raw_imgs, on_panels, yx, patch_r=3, search_r=2):
        """Mean pairwise NCC across ON panels. Tolerates residual registration offsets."""
        if len(yx) == 0:
            return np.array([], dtype=np.float32)
        keys = [pk for pk in on_panels if pk in raw_imgs]
        if len(keys) < 2:
            return np.ones(len(yx), dtype=np.float32)
        H = list(raw_imgs.values())[0].shape[0]
        W = list(raw_imgs.values())[0].shape[1]
        r = patch_r + search_r
        out = np.empty(len(yx), dtype=np.float32)
        for i, (y, x) in enumerate(yx):
            yi, xi = int(round(y)), int(round(x))
            y0=max(0,yi-r); y1=min(H,yi+r+1)
            x0=max(0,xi-r); x1=min(W,xi+r+1)
            patches = []
            for pk in keys:
                p = raw_imgs[pk][y0:y1, x0:x1].astype(np.float64)
                s = p.std()
                patches.append((p-p.mean())/s if s>1e-6 else np.zeros_like(p))
            pairs = []
            for a in range(len(patches)):
                for b in range(a+1, len(patches)):
                    cc = _correlate2d(patches[a], patches[b], mode="valid")
                    pairs.append(float(cc.max())/max(patches[a].size,1) if cc.size else 0.0)
            out[i] = float(np.mean(pairs)) if pairs else 0.0
        return out

    def _run_blob_log_joint(proj, blob_cfg):
        blobs = feature.blob_log(
            proj,
            min_sigma=blob_cfg["min_sigma"], max_sigma=blob_cfg["max_sigma"],
            num_sigma=blob_cfg["num_sigma"], threshold=blob_cfg["threshold"],
            overlap=blob_cfg["overlap"],    log_scale=blob_cfg["log_scale"],
        )
        if blobs is None or len(blobs)==0:
            return np.zeros((0,2), dtype=np.float32)
        return _nms_kdtree(np.asarray(blobs, dtype=np.float32)[:,:2],
                           blob_cfg["min_sigma"]*2.0)

    # ── Run all three methods ─────────────────────────────────────────────────────
    _SF_DFS     = []
    _blob_cfg   = CFG["starfish_blob_cfg"]
    _cfg_A      = CFG["sf_method_A"]
    _cfg_D      = CFG["sf_method_D"]
    _cfg_G      = CFG["sf_method_G"]

    log("Running joint-panel Starfish proposers (Methods A, D, G)...")
    _t_sf = time.perf_counter()

    for _, crow in crop_registry.iterrows():
        cid = str(crow["crop_id"]); wid = str(crow["well_id"])
        did = str(crow.get("dataset_id", wid))
        ymin,ymax = int(crow["well_ymin_px"]),int(crow["well_ymax_px"])
        xmin,xmax = int(crow["well_xmin_px"]),int(crow["well_xmax_px"])
        H,W = ymax-ymin,xmax-xmin; oy,ox = ymin,xmin
        on_panels = CFG["barcode_on_panels"].get(wid, CFG["barcode_panels"])
        pmap = CROP_PANEL_CACHE.get(cid, {})

        # Use get_panel_cache_for_use to handle stubs correctly
        raw_imgs = {}
        for pk in on_panels:
            if pk not in pmap: continue
            _c = get_panel_cache_for_use(cid, pk)
            if _c is not None and "raw" in _c:
                raw_imgs[pk] = _c["raw"]
                release_panel_cache(cid, pk)
        if not raw_imgs:
            continue
        proj = _sum_norm_proj(raw_imgs, on_panels)
        if proj is None:
            continue
        fp = CROP_PANEL_FINGERPRINTS.get(
            (cid, on_panels[0] if on_panels else ""), "unknown")

        try:
            candidates = _run_blob_log_joint(proj, _blob_cfg)
        except Exception as exc:
            log(f"  [warn] sf blob_log {cid}: {exc}"); continue

        if len(candidates) == 0:
            continue

        surv_A = np.zeros((0,2), dtype=np.float32)
        surv_D = np.zeros((0,2), dtype=np.float32)

        # ── Method A: per-panel SNR gate ─────────────────────────────────────────
        t0_A = time.perf_counter()
        try:
            on_pass = np.zeros(len(candidates), dtype=int)
            for pk in on_panels:
                raw = raw_imgs.get(pk)
                if raw is None: continue
                on_pass += (_local_snr_vec(raw, candidates) >= _cfg_A["snr_k"]).astype(int)
            surv_A = candidates[on_pass >= _cfg_A["min_panels"]]
            if len(surv_A):
                _SF_DFS.append(make_detection_df(
                    surv_A,
                    np.ones(len(surv_A), dtype=np.float32)*float(_cfg_A["snr_k"]),
                    run_id=RUN_ID, dataset_id=did, well_id=wid, crop_id=cid,
                    method_id=_cfg_A["name"], method_family="starfish_joint",
                    source_view_id=f"ON_joint__{_cfg_A['name']}",
                    score_type="local_snr", score_is_calibrated=False,
                    sigma_px=float(_blob_cfg["min_sigma"]),
                    timing_sec=time.perf_counter()-t0_A,
                    crop_origin_well_y_px=oy, crop_origin_well_x_px=ox,
                    image_shape_y=H, image_shape_x=W,
                    coord_semantics="blob_centroid",
                    source_image_fingerprint=fp))
        except Exception as exc:
            log(f"  [warn] sf_A {cid}: {exc}")

        # ── Method D: cross-panel NCC gate ───────────────────────────────────────
        t0_D = time.perf_counter()
        try:
            ncc_vals = _cross_panel_ncc(raw_imgs, on_panels, candidates,
                                        patch_r=_cfg_D["patch_r"],
                                        search_r=_cfg_D["search_r"])
            mask_D   = ncc_vals >= _cfg_D["ncc_thresh"]
            surv_D   = candidates[mask_D]
            if len(surv_D):
                _SF_DFS.append(make_detection_df(
                    surv_D,
                    ncc_vals[mask_D].astype(np.float32),
                    run_id=RUN_ID, dataset_id=did, well_id=wid, crop_id=cid,
                    method_id=_cfg_D["name"], method_family="starfish_joint",
                    source_view_id=f"ON_joint__{_cfg_D['name']}",
                    score_type="cross_panel_ncc", score_is_calibrated=False,
                    sigma_px=float(_blob_cfg["min_sigma"]),
                    timing_sec=time.perf_counter()-t0_D,
                    crop_origin_well_y_px=oy, crop_origin_well_x_px=ox,
                    image_shape_y=H, image_shape_x=W,
                    coord_semantics="blob_centroid",
                    source_image_fingerprint=fp))
        except Exception as exc:
            log(f"  [warn] sf_D {cid}: {exc}")

        # ── Method G: union of A+D, re-gated ─────────────────────────────────────
        t0_G = time.perf_counter()
        try:
            parts = [p for p in [surv_A, surv_D] if len(p)>0]
            if parts:
                union = _nms_kdtree(np.concatenate(parts, axis=0),
                                    float(_cfg_G["merge_r"]))
                if len(union):
                    on_pass2 = np.zeros(len(union), dtype=int)
                    for pk in on_panels:
                        raw = raw_imgs.get(pk)
                        if raw is None: continue
                        on_pass2 += (_local_snr_vec(raw, union) >= _cfg_G["regate_snr"]).astype(int)
                    surv_G = union[on_pass2 >= 2]
                    if len(surv_G):
                        _SF_DFS.append(make_detection_df(
                            surv_G,
                            np.ones(len(surv_G), dtype=np.float32)*float(_cfg_G["regate_snr"]),
                            run_id=RUN_ID, dataset_id=did, well_id=wid, crop_id=cid,
                            method_id=_cfg_G["name"], method_family="starfish_joint",
                            source_view_id=f"ON_joint__{_cfg_G['name']}",
                            score_type="union_regate_snr", score_is_calibrated=False,
                            sigma_px=float(_blob_cfg["min_sigma"]),
                            timing_sec=time.perf_counter()-t0_G,
                            crop_origin_well_y_px=oy, crop_origin_well_x_px=ox,
                            image_shape_y=H, image_shape_x=W,
                            coord_semantics="blob_centroid",
                            source_image_fingerprint=fp))
        except Exception as exc:
            log(f"  [warn] sf_G {cid}: {exc}")

    log(f"Starfish joint done in {time.perf_counter()-_t_sf:.1f}s")
    starfish_df = (pd.concat(_SF_DFS, ignore_index=True) if _SF_DFS else pd.DataFrame())
    if len(starfish_df):
        log(f"Starfish: {len(starfish_df)} detections across "
            f"{starfish_df['method_id'].nunique()} method variants")
        for mid, grp in starfish_df.groupby("method_id"):
            log(f"  {mid}: {len(grp)}")
    else:
        log("Starfish: 0 detections")

In [ ]:
# ── Cell [Spotiflow STUB] — spotiflow removed, stub keeps downstream intact ──
# Spotiflow proposer was removed. Define an empty spotiflow_df with the correct
# schema so the main detector loop and all downstream cells work unchanged.

spotiflow_df = pd.DataFrame(columns=[
    "raw_detection_id", "run_id", "dataset_id", "well_id", "crop_id",
    "method_id", "method_family", "source_view_id",
    "score_type", "score_is_calibrated",
    "detection_score_raw", "detection_score_norm",
    "well_y_px", "well_x_px",
    "crop_origin_well_y_px", "crop_origin_well_x_px",
    "image_shape_y", "image_shape_x",
    "coord_frame_primary", "coord_semantics",
    "sigma_px", "timing_sec", "source_image_fingerprint",
])

HAS_SPOTIFLOW = False
log("[spotiflow] Proposer disabled — spotiflow_df is empty (0 rows). Downstream unaffected.")

In [ ]:
# ── Cell [Spotiflow STUB] — spotiflow removed, stub keeps downstream intact ──
# Spotiflow proposer was removed. Define an empty spotiflow_df with the correct
# schema so the main detector loop and all downstream cells work unchanged.

spotiflow_df = pd.DataFrame(columns=[
    "raw_detection_id", "run_id", "dataset_id", "well_id", "crop_id",
    "method_id", "method_family", "source_view_id",
    "score_type", "score_is_calibrated",
    "detection_score_raw", "detection_score_norm",
    "well_y_px", "well_x_px",
    "crop_origin_well_y_px", "crop_origin_well_x_px",
    "image_shape_y", "image_shape_x",
    "coord_frame_primary", "coord_semantics",
    "sigma_px", "timing_sec", "source_image_fingerprint",
])

HAS_SPOTIFLOW = False
log("[spotiflow] Proposer disabled — spotiflow_df is empty (0 rows). Downstream unaffected.")

In [ ]:

# Reusable detector-cache short-circuit for all proposer outputs + side tables.
_CACHED_METHOD_IDS = [
    CFG["sf_method_A"]["name"],
    CFG["sf_method_D"]["name"],
    CFG["sf_method_G"]["name"],
    "mentor_v1",
    "mentor_v2",
    "mentor_v1_k2",
    "mentor_v1_k4",
    "mentor_v1_tight",
    *list(NEW_PROPOSER_DISPATCH.keys()),
]
_DETECTOR_CACHE_READY = (
    all(has_table_cache(DETECTOR_CACHE_DIR, "method_candidates", _mid) for _mid in _CACHED_METHOD_IDS) and
    has_table_cache(DETECTOR_CACHE_DIR, "detector_run_summary") and
    has_table_cache(DETECTOR_CACHE_DIR, "off_channel_negative_candidates")
)

if _DETECTOR_CACHE_READY:
    candidate_raw_dfs = [read_table_cache(DETECTOR_CACHE_DIR, "method_candidates", _mid) for _mid in _CACHED_METHOD_IDS]
    candidate_raw = pd.concat(candidate_raw_dfs, ignore_index=True) if candidate_raw_dfs else pd.DataFrame()
    detector_summary_df = read_table_cache(DETECTOR_CACHE_DIR, "detector_run_summary")
    off_neg_df = read_table_cache(DETECTOR_CACHE_DIR, "off_channel_negative_candidates")
    DETECTOR_RUN_SUMMARY = detector_summary_df.to_dict("records") if len(detector_summary_df) else []
    log(
        f"Detector cache hit: loaded {len(candidate_raw)} candidate rows, "
        f"{len(detector_summary_df)} detector-summary rows, and {len(off_neg_df)} OFF-negative rows."
    )
else:
    # ── Cell 14: Main detector loop — all proposers on every crop ────────────────

    DETECTOR_RUN_SUMMARY=[]
    candidate_raw_dfs=[]
    if len(spotiflow_df): candidate_raw_dfs.append(spotiflow_df)
    if len(starfish_df):  candidate_raw_dfs.append(starfish_df)

    MENTOR_VARIANTS=[
        ("mentor_v1",      run_mentor_v1_cfg, CFG["mentor_v1"]),
        ("mentor_v2",      run_mentor_v2_cfg, CFG["mentor_v2"]),
        ("mentor_v1_k2",   run_mentor_v1_cfg, CFG["mentor_v1_k2"]),
        ("mentor_v1_k4",   run_mentor_v1_cfg, CFG["mentor_v1_k4"]),
        ("mentor_v1_tight",run_mentor_v1_cfg, CFG["mentor_v1_tight"]),
    ]

    log(f"Main loop: {len(crop_registry)} ACTUAL full wells, {len(MENTOR_VARIANTS)+len(NEW_PROPOSER_DISPATCH)} methods")
    t_loop=time.perf_counter()

    for ci,(_, crow) in enumerate(crop_registry.iterrows()):
        cid=str(crow["crop_id"]); wid=str(crow["well_id"]); did=str(crow.get("dataset_id",wid))
        ymin,ymax=int(crow["well_ymin_px"]),int(crow["well_ymax_px"])
        xmin,xmax=int(crow["well_xmin_px"]),int(crow["well_xmax_px"])
        H,W=ymax-ymin,xmax-xmin; oy,ox=ymin,xmin
        cbp=CROP_PANEL_CACHE.get(cid,{})
        fp_ref=CROP_PANEL_FINGERPRINTS.get((cid,list(cbp.keys())[0] if cbp else ""),"unknown")

        # ── QA GT count summary for this well (from QA crop windows only) ─────────
        _gt_n = 0
        if annotations_df is not None and "well_id" in annotations_df.columns:
            _gt_n = int((annotations_df[(annotations_df["well_id"]==wid) &
                                        (annotations_df["label"]=="TRUE_SPOT")]).shape[0])
        log(f"[{ci+1}/{len(crop_registry)}] {cid} well={wid} ({H}x{W}) QA_TRUE_SPOT={_gt_n}")
        crop_dfs=[]

        # ── mentor methods — lightweight: only raw+norm01 needed ──────────────
        real_cbp = {}
        try:
            for pk in CFG["mentor_required_panels"]:
                p = get_tiff_path(wid, pk)
                if not p.exists():
                    continue
                img = read_image_2d(p)[ymin:ymax, xmin:xmax].astype(np.float32)
                norm, lo, hi = norm01_fn(img, *CFG["norm_percentiles"])
                real_cbp[pk] = {
                    "raw":        img,
                    "norm01":     norm,
                    "norm_limits":(lo, hi),
                }
                del norm
            if not real_cbp:
                for mid, _, _ in MENTOR_VARIANTS:
                    DETECTOR_RUN_SUMMARY.append({"method_id":mid,"crop_id":cid,"n_points":0,"status":"no_cache","timing_sec":0})
            else:
                for mid, runner, vcfg in MENTOR_VARIANTS:
                    t0 = time.perf_counter()
                    try:
                        coords, scores, _, _ = runner(real_cbp, vcfg)
                        timing = time.perf_counter() - t0
                        if len(coords):
                            sc = scores if len(scores)==len(coords) else np.ones(len(coords),dtype=np.float32)
                            crop_dfs.append(make_detection_df(
                                coords,sc,run_id=RUN_ID,dataset_id=did,well_id=wid,crop_id=cid,
                                method_id=mid,method_family=DETECTOR_FAMILY.get(mid,"mentor"),
                                source_view_id=f"all_panels__{mid}",score_type="consensus_count",
                                score_is_calibrated=False,sigma_px=None,timing_sec=timing,
                                crop_origin_well_y_px=oy,crop_origin_well_x_px=ox,
                                image_shape_y=H,image_shape_x=W,coord_semantics="consensus_centroid",
                                source_image_fingerprint=fp_ref))
                        DETECTOR_RUN_SUMMARY.append({"method_id":mid,"crop_id":cid,"n_points":len(coords),"status":"ok","timing_sec":timing})
                        log(f"  {mid}: {len(coords)}")
                    except Exception as exc:
                        DETECTOR_RUN_SUMMARY.append({"method_id":mid,"crop_id":cid,"n_points":0,"status":f"error:{type(exc).__name__}","timing_sec":time.perf_counter()-t0})
                        log(f"  [warn] {mid}: {exc}")
        finally:
            del real_cbp
            gc.collect()
        # ── new raw-ADU proposers ─────────────────────────────────────────────────
        for mid,fn in NEW_PROPOSER_DISPATCH.items():
            pcfg=CFG["new_proposers"][mid]; t0=time.perf_counter()
            try:
                coords,scores=fn(wid,ymin,ymax,xmin,xmax,pcfg)
                timing=time.perf_counter()-t0
                if len(coords):
                    sc=scores if len(scores)==len(coords) else np.ones(len(coords),dtype=np.float32)
                    crop_dfs.append(make_detection_df(
                        coords,sc,run_id=RUN_ID,dataset_id=did,well_id=wid,crop_id=cid,
                        method_id=mid,method_family=DETECTOR_FAMILY.get(mid,"raw_adu"),
                        source_view_id=f"{wid}__{mid}",score_type="raw_adu_response",
                        score_is_calibrated=False,sigma_px=None,timing_sec=timing,
                        crop_origin_well_y_px=oy,crop_origin_well_x_px=ox,
                        image_shape_y=H,image_shape_x=W,coord_semantics="local_maximum",
                        source_image_fingerprint="raw_adu"))
                DETECTOR_RUN_SUMMARY.append({"method_id":mid,"crop_id":cid,"n_points":len(coords),"status":"ok","timing_sec":timing})
                log(f"  {mid}: {len(coords)}")
            except Exception as exc:
                DETECTOR_RUN_SUMMARY.append({"method_id":mid,"crop_id":cid,"n_points":0,"status":f"error:{type(exc).__name__}","timing_sec":time.perf_counter()-t0})
                log(f"  [warn] {mid}: {exc}")

        if crop_dfs: candidate_raw_dfs.extend(crop_dfs)

    log(f"Loop done in {time.perf_counter()-t_loop:.1f}s")
    candidate_raw=(pd.concat(candidate_raw_dfs,ignore_index=True) if candidate_raw_dfs else pd.DataFrame())
    detector_summary_df=pd.DataFrame(DETECTOR_RUN_SUMMARY)
    log(f"Total raw detections: {len(candidate_raw)}")

    if len(detector_summary_df):
        _ok=detector_summary_df[detector_summary_df["status"]=="ok"]
        if len(_ok):
            _s=_ok.groupby("method_id")["n_points"].agg(["sum","mean","max"]).rename(columns={"sum":"total","mean":"avg"})
            print(_s.sort_values("total",ascending=False).to_string())

    # ── OFF-channel negative candidate injection ──────────────────────────────────
    # All existing proposers operate on ON panels only, so the main candidate union
    # contains zero OFF-channel locations. NB05/NB06 still need explicit OFF-channel
    # negatives for supervised learning, but they must NOT enter the main union-ID
    # namespace or they will contaminate real candidate unions.
    #
    # Therefore we generate them here as a separate side table only:
    #   1. Same (y,x) as ON-panel detections: real spots that are OFF in this channel
    #   2. Background local maxima in the OFF panel: bright OFF-channel noise
    #
    # These rows are saved separately and should be joined later in NB05 as explicit
    # negatives. They are NOT concatenated into candidate_raw.

    _OFF_NEG_DFS = []
    _OFF_NEG_METHOD = "off_channel_negative_sampler"
    _OFF_NEG_MAX_BG = 150   # background local maxima cap per OFF panel per crop

    log("Generating OFF-channel negative side-table candidates...")
    _t_off = time.perf_counter()

    # Collect ON-panel candidate well coords per crop for paired negatives
    _on_locs_by_crop = {}
    if len(candidate_raw):
        _on_only = candidate_raw.copy()
        for _cid_g, _grp in _on_only.groupby("crop_id"):
            _on_locs_by_crop[str(_cid_g)] = _grp[["crop_y_px", "crop_x_px"]].values.astype(np.float32)

    for _, crow in crop_registry.iterrows():
        cid = str(crow["crop_id"]); wid = str(crow["well_id"])
        did = str(crow.get("dataset_id", wid))
        ymin, ymax = int(crow["well_ymin_px"]), int(crow["well_ymax_px"])
        xmin, xmax = int(crow["well_xmin_px"]), int(crow["well_xmax_px"])
        H, W = ymax - ymin, xmax - xmin; oy, ox = ymin, xmin

        off_panels = CFG["barcode_off_panels"].get(wid, [])
        if not off_panels:
            continue

        on_locs = _on_locs_by_crop.get(cid, np.zeros((0, 2), dtype=np.float32))

        for pk in off_panels:
            try:
                raw = load_crop_raw(wid, pk, ymin, ymax, xmin, xmax).astype(np.float32)
            except Exception:
                continue

            neg_parts = []

            # Part 1: same locations as ON-panel detections (hard negatives)
            if len(on_locs):
                neg_parts.append(on_locs.copy())

            # Part 2: background local maxima in this OFF panel
            try:
                hp_off = np.maximum(raw - gaussian_filter(raw, 2.0), 0.0)
                bg_lm = peak_local_max(
                    hp_off,
                    min_distance=4,
                    exclude_border=True,
                    threshold_abs=200.0,
                )
                if len(bg_lm):
                    bg_c = bg_lm.astype(np.float32)
                    idx = np.random.permutation(len(bg_c))[:_OFF_NEG_MAX_BG]
                    neg_parts.append(bg_c[idx])
            except Exception:
                pass

            if not neg_parts:
                continue

            all_neg = np.vstack(neg_parts).astype(np.float32)
            all_neg[:, 0] = np.clip(all_neg[:, 0], 0, H - 1)
            all_neg[:, 1] = np.clip(all_neg[:, 1], 0, W - 1)
            all_neg = nms_greedy(all_neg, 3.0)   # small-N dedup only within the negative pool

            if len(all_neg) == 0:
                continue

            yy = np.clip(np.round(all_neg[:, 0]).astype(int), 0, H - 1)
            xx = np.clip(np.round(all_neg[:, 1]).astype(int), 0, W - 1)
            scores_neg = raw[yy, xx].astype(np.float32)

            df_neg = make_detection_df(
                all_neg, scores_neg,
                run_id=RUN_ID, dataset_id=did, well_id=wid, crop_id=cid,
                method_id=_OFF_NEG_METHOD, method_family="negative_sampler",
                source_view_id=f"{pk}__{_OFF_NEG_METHOD}",
                score_type="raw_adu_pixel", score_is_calibrated=False,
                sigma_px=None, timing_sec=0.0,
                crop_origin_well_y_px=oy, crop_origin_well_x_px=ox,
                image_shape_y=H, image_shape_x=W,
                coord_semantics="local_maximum",
                source_image_fingerprint="raw_adu",
            )
            df_neg["is_off_channel_negative"] = True
            df_neg["off_channel_panel"] = pk
            _OFF_NEG_DFS.append(df_neg)

    log(f"OFF-channel side-table generation done in {time.perf_counter()-_t_off:.1f}s")

    # Keep main candidate_raw schema stable, but do NOT inject negatives into it.
    if "is_off_channel_negative" not in candidate_raw.columns:
        candidate_raw["is_off_channel_negative"] = False
    if "off_channel_panel" not in candidate_raw.columns:
        candidate_raw["off_channel_panel"] = None

    if _OFF_NEG_DFS:
        off_neg_df = pd.concat(_OFF_NEG_DFS, ignore_index=True)
        log(
            f"OFF-channel negatives side table: {len(off_neg_df)} candidates across "
            f"{off_neg_df['crop_id'].nunique()} crops"
        )
    else:
        off_neg_df = pd.DataFrame(columns=list(candidate_raw.columns))
        if "is_off_channel_negative" not in off_neg_df.columns:
            off_neg_df["is_off_channel_negative"] = pd.Series(dtype=bool)
        if "off_channel_panel" not in off_neg_df.columns:
            off_neg_df["off_channel_panel"] = pd.Series(dtype=object)
        log("[warn] No OFF-channel negatives generated — check barcode_off_panels config")

    log(f"Main candidate_raw remains union-safe: {len(candidate_raw)} rows")


# ── Persist reusable detector caches for reruns of NB03/04 ───────────────────
if len(candidate_raw):
    for _mid, _grp in candidate_raw.groupby("method_id", sort=False):
        write_table_cache(_grp, DETECTOR_CACHE_DIR, "method_candidates", _mid)
if 'detector_summary_df' in globals() and detector_summary_df is not None:
    write_table_cache(detector_summary_df, DETECTOR_CACHE_DIR, "detector_run_summary")
if 'off_neg_df' in globals() and off_neg_df is not None:
    write_table_cache(off_neg_df, DETECTOR_CACHE_DIR, "off_channel_negative_candidates")
log(f"Reusable detector caches refreshed under {DETECTOR_CACHE_DIR}")


In [ ]:
# ── Cell 15: Candidate union + clusters ──────────────────────────────────────

def cc_radius(coords,radius):
    n=len(coords)
    if n==0: return np.array([],dtype=int)
    if n==1: return np.array([0],dtype=int)
    tree=cKDTree(coords); pairs=list(tree.query_pairs(r=radius))
    if not pairs: return np.arange(n,dtype=int)
    rows,cols=[],[]
    for i,j in pairs: rows.extend([i,j]); cols.extend([j,i])
    data=np.ones(len(rows),dtype=np.uint8)
    g=coo_matrix((data,(rows,cols)),shape=(n,n))
    _,labels=connected_components(g,directed=False)
    return labels.astype(int)

def build_union_tables(craw,merge_r):
    if len(craw)==0: return pd.DataFrame(),pd.DataFrame()
    n_methods=craw["method_id"].nunique()
    urows,mrows=[],[]
    for keys,g in craw.groupby(["dataset_id","well_id","crop_id"],sort=True):
        did_g,wid_g,cid_g=keys
        g=g.sort_values(["method_id","source_view_id","raw_detection_id"]).reset_index(drop=True)
        coords=g[["well_y_px","well_x_px"]].to_numpy(dtype=np.float32)
        labels=cc_radius(coords,merge_r)
        for comp in np.unique(labels):
            idx=np.flatnonzero(labels==comp)
            sub=g.iloc[idx].copy().reset_index(drop=True)
            mids=sorted(sub["raw_detection_id"].tolist())
            uid=sha1_text("|".join(mids))
            cc=coords[idx]; cen=cc.mean(0)
            d2c=np.sqrt(((cc-cen)**2).sum(1)); med_i=int(np.argmin(d2c)); med=cc[med_i]
            pset=sorted(sub["method_id"].astype(str).unique().tolist())
            fset=sorted(sub["method_family"].astype(str).unique().tolist())
            top=sub.sort_values(["detection_score_norm","detection_score_raw","raw_detection_id"],
                                ascending=[False,False,True]).iloc[0]
            urows.append({
                "union_id":uid,"run_id":top["run_id"],
                "dataset_id":top["dataset_id"],"well_id":top["well_id"],"crop_id":top["crop_id"],
                "union_n_members":int(len(sub)),
                "union_centroid_well_y_px":float(cen[0]),"union_centroid_well_x_px":float(cen[1]),
                "union_medoid_well_y_px":float(med[0]),"union_medoid_well_x_px":float(med[1]),
                "union_radius_px":float(d2c.max() if len(d2c) else 0.0),
                "union_bbox_ymin_px":float(cc[:,0].min()),"union_bbox_xmin_px":float(cc[:,1].min()),
                "union_bbox_ymax_px":float(cc[:,0].max()),"union_bbox_xmax_px":float(cc[:,1].max()),
                "top_method_id":str(top["method_id"]),
                "top_method_score_raw":float(top["detection_score_raw"]),
                "top_method_score_norm":float(top["detection_score_norm"]),
                "proposer_support_count":int(len(pset)),
                "proposer_support_fraction":float(len(pset)/max(n_methods,1)),
                "proposer_set_canonical":"|".join(pset),
                "family_support_count":int(len(fset)),
                "family_set_canonical":"|".join(fset),
                "cluster_id":None,
                **{k:top[k] for k in BASE_PROV if k in top},
            })
            ranked=sub.sort_values(["detection_score_norm","detection_score_raw","raw_detection_id"],
                                   ascending=[False,False,True]).reset_index(drop=True)
            yr=ranked[["well_y_px","well_x_px"]].to_numpy(dtype=np.float32)
            dc=np.sqrt(((yr-cen)**2).sum(1)); dm=np.sqrt(((yr-med)**2).sum(1))
            med_rid=sub.iloc[med_i]["raw_detection_id"]
            for rank in range(len(ranked)):
                mrows.append({"union_id":uid,"raw_detection_id":ranked.iloc[rank]["raw_detection_id"],
                               "member_rank_by_score":rank+1,
                               "member_is_medoid":bool(ranked.iloc[rank]["raw_detection_id"]==med_rid),
                               "member_distance_to_centroid_px":float(dc[rank]),
                               "member_distance_to_medoid_px":float(dm[rank])})
    return pd.DataFrame(urows),pd.DataFrame(mrows)

def build_cluster_table(cu,cluster_r):
    if len(cu)==0: return pd.DataFrame()
    rows=[]
    for keys,g in cu.groupby(["dataset_id","well_id","crop_id"],sort=True):
        did_g,wid_g,cid_g=keys; g=g.copy().reset_index(drop=True)
        coords=g[["union_medoid_well_y_px","union_medoid_well_x_px"]].to_numpy(dtype=np.float32)
        labels=cc_radius(coords,cluster_r)
        for comp in np.unique(labels):
            idx=np.flatnonzero(labels==comp); sub=g.iloc[idx]; cc=coords[idx]
            cen=cc.mean(0)
            clid=sha1_text("cluster|"+("|".join(sorted(sub["union_id"].tolist()))))
            rows.append({"cluster_id":clid,"run_id":RUN_ID,"dataset_id":did_g,"well_id":wid_g,"crop_id":cid_g,
                         "n_union_candidates":int(len(idx)),
                         "cluster_centroid_well_y_px":float(cen[0]),"cluster_centroid_well_x_px":float(cen[1]),
                         "cluster_radius_px":float(np.sqrt(((cc-cen)**2).sum(1)).max() if len(idx)>1 else 0.0)})
            cu.loc[g.iloc[idx].index,"cluster_id"]=clid
    return pd.DataFrame(rows)

log("Building union tables...")
candidate_union,candidate_union_membership=build_union_tables(candidate_raw,CFG["merge_radius_px"])
log(f"Union: {len(candidate_union)} candidates")
if len(candidate_raw) and len(candidate_union_membership):
    candidate_raw=candidate_raw.merge(
        candidate_union_membership[["union_id","raw_detection_id"]],on="raw_detection_id",how="left")

log("Building cluster table...")
candidate_clusters=build_cluster_table(candidate_union,CFG["cluster_radius_px"])
log(f"Clusters: {len(candidate_clusters)}")


In [ ]:
# ── Cell 16: Feature Engineering (NB04 inline, RAW-LOCAL rewrite) ────────────
# RAW-domain only. No crop percentiles. No crop ranks. No GT-derived reference
# profiles. All features must be computable identically anywhere in the well.

from scipy.optimize import curve_fit as _curve_fit

# ── patch geometry helpers ────────────────────────────────────────────────────
_IR   = int(CFG["patch_inner_radius_px"])
_R1I  = int(CFG["patch_ring1_inner_px"])
_R1O  = int(CFG["patch_ring1_outer_px"])
_R2I  = int(CFG["patch_ring2_inner_px"])
_R2O  = int(CFG["patch_ring2_outer_px"])

def _make_annulus_mask(inner_r,outer_r,R=20):
    yy,xx=np.mgrid[-R:R+1,-R:R+1]
    d2=yy*yy+xx*xx
    return (d2<=outer_r*outer_r)&(d2>inner_r*inner_r)

_INNER_MASK = _make_annulus_mask(0,_IR)
_RING1_MASK = _make_annulus_mask(_R1I,_R1O)
_RING2_MASK = _make_annulus_mask(_R2I,_R2O)
_R_MAX = max(_R2O,_IR)+2

def bilinear_sample(img,y,x):
    H,W=img.shape
    y=float(y); x=float(x)
    y0=max(0,min(H-2,int(y))); x0=max(0,min(W-2,int(x)))
    fy=y-y0; fx=x-x0
    return float((1-fy)*(1-fx)*img[y0,x0] + (1-fy)*fx*img[y0,x0+1] +
                 fy*(1-fx)*img[y0+1,x0] + fy*fx*img[y0+1,x0+1])

def extract_annulus(img,cy,cx,mask,R=20):
    H,W=img.shape
    iy,ix=int(round(cy)),int(round(cx))
    y0=max(0,iy-R); y1=min(H,iy+R+1)
    x0=max(0,ix-R); x1=min(W,ix+R+1)
    if (y1-y0)<3 or (x1-x0)<3:
        return np.array([],dtype=np.float32),0.0
    patch=img[y0:y1,x0:x1]
    dy0,dx0=iy-R-y0,ix-R-x0
    pH,pW=patch.shape
    mH,mW=mask.shape
    my0=max(0,-dy0); mx0=max(0,-dx0)
    py0=max(0,dy0);  px0=max(0,dx0)
    my1=min(mH,my0+(pH-py0)); mx1=min(mW,mx0+(pW-px0))
    py1=py0+(my1-my0); px1=px0+(mx1-mx0)
    if py1<=py0 or px1<=px0 or my1<=my0 or mx1<=mx0:
        return np.array([],dtype=np.float32),0.0
    m_sub=mask[my0:my1,mx0:mx1]
    p_sub=patch[py0:py1,px0:px1]
    if m_sub.shape != p_sub.shape:
        return np.array([],dtype=np.float32),0.0
    vals=p_sub[m_sub].astype(np.float32)
    cov=float(m_sub.sum())/float(mask.sum()+1e-9)
    return vals,cov

def patch_stats(v):
    if len(v)==0:
        return {"mean":np.nan,"std":np.nan,"max":np.nan,"min":np.nan,"median":np.nan}
    return {"mean":float(np.mean(v)),
            "std":float(np.std(v)),
            "max":float(np.max(v)),
            "min":float(np.min(v)),
            "median":float(np.median(v))}

def robust_mad(v):
    v=np.asarray(v,dtype=np.float32)
    if v.size==0:
        return np.nan
    med=float(np.median(v))
    return float(np.median(np.abs(v-med))/0.67448975)

def safe_skew(v):
    v=np.asarray(v,dtype=np.float32)
    if v.size<3:
        return np.nan
    mu=float(np.mean(v)); sd=float(np.std(v))
    if sd<1e-12:
        return 0.0
    z=(v-mu)/sd
    return float(np.mean(z**3))

def safe_kurt(v):
    v=np.asarray(v,dtype=np.float32)
    if v.size<4:
        return np.nan
    mu=float(np.mean(v)); sd=float(np.std(v))
    if sd<1e-12:
        return 0.0
    z=(v-mu)/sd
    return float(np.mean(z**4)-3.0)

def radial_profile_means(img,cy,cx,radii=(1,2,3,4,5,6)):
    H,W=img.shape
    iy,ix=int(round(cy)),int(round(cx))
    R=max(radii)+1
    y0=max(0,iy-R); y1=min(H,iy+R+1)
    x0=max(0,ix-R); x1=min(W,ix+R+1)
    patch=img[y0:y1,x0:x1]
    yy,xx=np.mgrid[y0:y1,x0:x1]
    rr=np.sqrt((yy-cy)**2+(xx-cx)**2)
    out={}
    prev=0.0
    for r in radii:
        m=(rr<=r) & (rr>prev)
        out[r]=float(np.mean(patch[m])) if np.any(m) else np.nan
        prev=float(r)
    return out

def local_plane_fit(img,cy,cx,r=10,inner_excl=3):
    H,W=img.shape
    iy,ix=int(round(cy)),int(round(cx))
    y0=max(0,iy-r); y1=min(H,iy+r+1)
    x0=max(0,ix-r); x1=min(W,ix+r+1)
    if (y1-y0)<5 or (x1-x0)<5:
        return {"plane_a":np.nan,"plane_b":np.nan,"plane_c":np.nan,
                "plane_resid_std":np.nan,"plane_center_pred":np.nan}
    yy,xx=np.mgrid[y0:y1,x0:x1]
    rr2=(yy-cy)**2+(xx-cx)**2
    use=rr2>(inner_excl*inner_excl)
    vals=img[y0:y1,x0:x1]
    A=np.column_stack([yy[use].ravel(),xx[use].ravel(),np.ones(int(use.sum()))])
    b=vals[use].ravel()
    if len(b)<6:
        return {"plane_a":np.nan,"plane_b":np.nan,"plane_c":np.nan,
                "plane_resid_std":np.nan,"plane_center_pred":np.nan}
    coefs,_,_,_=np.linalg.lstsq(A,b,rcond=None)
    pred=(A@coefs)
    resid=b-pred
    center_pred=float(coefs[0]*cy + coefs[1]*cx + coefs[2])
    return {"plane_a":float(coefs[0]),
            "plane_b":float(coefs[1]),
            "plane_c":float(coefs[2]),
            "plane_resid_std":float(np.std(resid)),
            "plane_center_pred":center_pred}

def quadratic_bowl_fit(img,cy,cx,r=8,inner_excl=0):
    H,W=img.shape
    iy,ix=int(round(cy)),int(round(cx))
    y0=max(0,iy-r); y1=min(H,iy+r+1)
    x0=max(0,ix-r); x1=min(W,ix+r+1)
    if (y1-y0)<5 or (x1-x0)<5:
        return {"quad_resid_std":np.nan,"quad_center_pred":np.nan,
                "quad_trace":np.nan,"quad_det":np.nan}
    yy,xx=np.mgrid[y0:y1,x0:x1]
    yy2=(yy-cy).astype(np.float32)
    xx2=(xx-cx).astype(np.float32)
    rr2=yy2*yy2+xx2*xx2
    use=rr2>float(inner_excl*inner_excl)
    vals=img[y0:y1,x0:x1].astype(np.float32)
    A=np.column_stack([
        yy2[use].ravel()**2,
        xx2[use].ravel()**2,
        (yy2[use]*xx2[use]).ravel(),
        yy2[use].ravel(),
        xx2[use].ravel(),
        np.ones(int(use.sum()))
    ])
    b=vals[use].ravel()
    if len(b)<10:
        return {"quad_resid_std":np.nan,"quad_center_pred":np.nan,
                "quad_trace":np.nan,"quad_det":np.nan}
    coefs,_,_,_=np.linalg.lstsq(A,b,rcond=None)
    pred=(A@coefs)
    resid=b-pred
    a,b2,c,d,e,f0=coefs
    Hm=np.array([[2*a,c],[c,2*b2]],dtype=np.float64)
    return {"quad_resid_std":float(np.std(resid)),
            "quad_center_pred":float(f0),
            "quad_trace":float(np.trace(Hm)),
            "quad_det":float(np.linalg.det(Hm))}

def patch_entropy_raw(v,bins=32):
    v=np.asarray(v,dtype=np.float32)
    if v.size==0:
        return np.nan
    vmin=float(np.min(v)); vmax=float(np.max(v))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax<=vmin:
        return 0.0
    h,_=np.histogram(v,bins=bins,range=(vmin,vmax))
    p=h.astype(np.float64)/max(int(h.sum()),1)
    p=p[p>0]
    return float(-np.sum(p*np.log2(p)))

def _gauss2d(coords,amp,y0,x0,sy,sx,off):
    yy,xx=coords
    return (amp*np.exp(-(((yy-y0)**2)/(2*sy*sy)+((xx-x0)**2)/(2*sx*sx)))+off).ravel()


def bilinear_sample_batch(img: np.ndarray, ys: np.ndarray, xs: np.ndarray) -> np.ndarray:
    """Vectorized bilinear sampling of img at multiple (y, x) coordinates."""
    H, W = img.shape
    ys = np.asarray(ys, dtype=np.float32)
    xs = np.asarray(xs, dtype=np.float32)
    y0 = np.clip(ys.astype(int), 0, H - 2)
    x0 = np.clip(xs.astype(int), 0, W - 2)
    fy = ys - y0
    fx = xs - x0
    return (
        (1 - fy) * (1 - fx) * img[y0, x0] +
        (1 - fy) * fx       * img[y0, x0 + 1] +
        fy       * (1 - fx) * img[y0 + 1, x0] +
        fy       * fx       * img[y0 + 1, x0 + 1]
    ).astype(np.float32)

log("Feature engineering helpers defined for RAW-LOCAL rewrite.")

In [ ]:
# ── Cell 17: Family 1 — RAW local photometry + local-noise physics ───────────
# No norm01. No crop-global scaling. No percentile features.
# Batched by crop: cache loaded once per crop, all candidates processed, then released.

log("="*70 + "\nFAMILY 1: RAW local photometry + local-noise physics")
t0_f1 = time.perf_counter()
F1_ROWS = []
MAP_KEYS = ["raw", "highpass", "white", "z_local", "peak_bg"]

for cid, cu_crop in candidate_union.groupby("crop_id", sort=True):
    cid = str(cid)
    crow = CROP_LOOKUP.get(cid)
    if crow is None:
        for _, urow in cu_crop.iterrows():
            F1_ROWS.append({"union_id": urow["union_id"]})
        continue

    pmap = CROP_PANEL_CACHE.get(cid, {})
    primary = next((pk for pk in CFG["barcode_panels"] if pk in pmap), None)
    if primary is None:
        for _, urow in cu_crop.iterrows():
            F1_ROWS.append({"union_id": urow["union_id"]})
        continue

    cache = get_panel_cache_for_use(cid, primary)
    if cache is None:
        for _, urow in cu_crop.iterrows():
            F1_ROWS.append({"union_id": urow["union_id"]})
        continue

    try:
        # preload all map arrays once
        map_arrs = {mk: cache.get(mk) for mk in MAP_KEYS}
        raw_img  = cache.get("raw")

        oy = float(crow["well_ymin_px"])
        ox = float(crow["well_xmin_px"])

        for _, urow in cu_crop.iterrows():
            uid = urow["union_id"]
            wy  = float(urow["union_medoid_well_y_px"])
            wx  = float(urow["union_medoid_well_x_px"])
            cy  = wy - oy
            cx  = wx - ox
            f   = {"union_id": uid}

            for map_key in MAP_KEYS:
                img = map_arrs.get(map_key)
                if img is None:
                    continue

                center = bilinear_sample(img, cy, cx)
                vi,  ci_  = extract_annulus(img, cy, cx, _INNER_MASK, _R_MAX)
                vr1, c1_  = extract_annulus(img, cy, cx, _RING1_MASK, _R_MAX)
                vr2, c2_  = extract_annulus(img, cy, cx, _RING2_MASK, _R_MAX)

                f[f"photo_{map_key}_center"] = center

                if ci_ >= 0.5 and len(vi) > 0:
                    s = patch_stats(vi)
                    for k, v in s.items():
                        f[f"photo_{map_key}_inner_{k}"] = v
                    f[f"photo_{map_key}_inner_mad"] = robust_mad(vi)
                else:
                    for k in ["mean","std","max","min","median","mad"]:
                        f[f"photo_{map_key}_inner_{k}"] = np.nan

                if c1_ >= 0.3 and len(vr1) > 0:
                    rs1 = patch_stats(vr1)
                    f[f"photo_{map_key}_ring1_mean"] = rs1["mean"]
                    f[f"photo_{map_key}_ring1_std"]  = rs1["std"]
                    f[f"photo_{map_key}_ring1_mad"]  = robust_mad(vr1)
                else:
                    f[f"photo_{map_key}_ring1_mean"] = np.nan
                    f[f"photo_{map_key}_ring1_std"]  = np.nan
                    f[f"photo_{map_key}_ring1_mad"]  = np.nan

                if c2_ >= 0.3 and len(vr2) > 0:
                    rs2 = patch_stats(vr2)
                    f[f"photo_{map_key}_ring2_mean"] = rs2["mean"]
                    f[f"photo_{map_key}_ring2_std"]  = rs2["std"]
                    f[f"photo_{map_key}_ring2_mad"]  = robust_mad(vr2)
                else:
                    f[f"photo_{map_key}_ring2_mean"] = np.nan
                    f[f"photo_{map_key}_ring2_std"]  = np.nan
                    f[f"photo_{map_key}_ring2_mad"]  = np.nan

                if len(vi) > 0 and len(vr1) > 0:
                    mu_i  = float(np.mean(vi))
                    mu_r1 = float(np.mean(vr1))
                    sd_r1 = float(np.std(vr1)) + 1e-9
                    mad_r1 = robust_mad(vr1)
                    f[f"photo_{map_key}_contrast_r1"]  = mu_i - mu_r1
                    f[f"photo_{map_key}_snr_r1_std"]   = (mu_i - mu_r1) / sd_r1
                    f[f"photo_{map_key}_snr_r1_mad"]   = (mu_i - mu_r1) / max(mad_r1, 1e-9)
                else:
                    f[f"photo_{map_key}_contrast_r1"] = np.nan
                    f[f"photo_{map_key}_snr_r1_std"]  = np.nan
                    f[f"photo_{map_key}_snr_r1_mad"]  = np.nan

                if len(vi) > 0 and len(vr2) > 0:
                    mu_i  = float(np.mean(vi))
                    mu_r2 = float(np.mean(vr2))
                    sd_r2 = float(np.std(vr2)) + 1e-9
                    mad_r2 = robust_mad(vr2)
                    f[f"photo_{map_key}_contrast_r2"]  = mu_i - mu_r2
                    f[f"photo_{map_key}_snr_r2_std"]   = (mu_i - mu_r2) / sd_r2
                    f[f"photo_{map_key}_snr_r2_mad"]   = (mu_i - mu_r2) / max(mad_r2, 1e-9)
                else:
                    f[f"photo_{map_key}_contrast_r2"] = np.nan
                    f[f"photo_{map_key}_snr_r2_std"]  = np.nan
                    f[f"photo_{map_key}_snr_r2_mad"]  = np.nan

                rp = radial_profile_means(img, cy, cx, radii=(1,2,3,4,5,6))
                for r, v in rp.items():
                    f[f"photo_{map_key}_radial_bin{r}"] = v
                f[f"photo_{map_key}_radial_drop_1_3"] = (rp[1]-rp[3]) if np.isfinite(rp.get(1,np.nan)) and np.isfinite(rp.get(3,np.nan)) else np.nan
                f[f"photo_{map_key}_radial_drop_2_5"] = (rp[2]-rp[5]) if np.isfinite(rp.get(2,np.nan)) and np.isfinite(rp.get(5,np.nan)) else np.nan

            if raw_img is not None:
                pf = local_plane_fit(raw_img, cy, cx, r=10, inner_excl=_IR)
                qf = quadratic_bowl_fit(raw_img, cy, cx, r=8, inner_excl=0)
                for k, v in pf.items(): f[f"raw_{k}"] = v
                for k, v in qf.items(): f[f"raw_{k}"] = v
                center_raw = bilinear_sample(raw_img, cy, cx)
                f["raw_center_minus_plane"] = center_raw - pf["plane_center_pred"] if np.isfinite(pf["plane_center_pred"]) else np.nan
                f["raw_center_minus_quad"]  = center_raw - qf["quad_center_pred"]  if np.isfinite(qf["quad_center_pred"])  else np.nan

            F1_ROWS.append(f)

        log(f"  F1 crop {cid}: {len(cu_crop)} candidates done")

    finally:
        release_panel_cache(cid, primary)
        del cache
        gc.collect()

f1_df = pd.DataFrame(F1_ROWS)
log(f"Family 1 done: {len([c for c in f1_df.columns if c!='union_id'])} cols in {time.perf_counter()-t0_f1:.1f}s")

In [ ]:
# ── Cell 20: Pre-compute RAW-domain response maps for Family 2 ────────────────

log("Adding RAW-domain multiscale response maps to preprocessing cache...")
t0_log = time.perf_counter()
n_done = 0
total = len(CROP_PANEL_CACHE)

REQ_F2_KEYS = [
    "log_raw_s1", "log_raw_s2", "log_raw_s3", "log_raw_s5",
    "dog_raw_s1", "dog_raw_s2", "dog_raw_s3", "dog_raw_s5",
    "hess_raw_trace", "hess_raw_det", "hess_raw_l1", "hess_raw_l2",
    "hess_raw_blobness", "hp_local_std9",
]

for cid, panel_cache in CROP_PANEL_CACHE.items():
    log(f"[Family 2 precompute] {cid}: start")

    for pk in list(panel_cache.keys()):
        cache = get_panel_cache_for_use(cid, pk)
        if not isinstance(cache, dict):
            continue

        if panel_cache_has_keys(cache, REQ_F2_KEYS):
            release_panel_cache(cid, pk)
            continue

        raw_img = cache.get("raw")
        if raw_img is None:
            release_panel_cache(cid, pk)
            continue

        raw32 = raw_img.astype(np.float32, copy=False)

        for sig in (1.0, 2.0, 3.0, 5.0):
            cache[f"log_raw_s{sig:.0f}"] = (
                -(sig ** 2) * ndi.gaussian_laplace(raw32, sigma=sig)
            ).astype(np.float32)

            g1 = ndi.gaussian_filter(raw32, sigma=max(sig - 0.5, 0.5)).astype(np.float32)
            g2 = ndi.gaussian_filter(raw32, sigma=sig + 0.5).astype(np.float32)
            dog = (g1 - g2).astype(np.float32)
            dog[dog < 0] = 0.0
            cache[f"dog_raw_s{sig:.0f}"] = dog
            del g1, g2, dog

        sm = ndi.gaussian_filter(raw32, sigma=2.0).astype(np.float32)
        del raw32

        gy = np.gradient(sm, axis=0).astype(np.float32)
        gx = np.gradient(sm, axis=1).astype(np.float32)
        del sm

        gyy = np.gradient(gy, axis=0).astype(np.float32)
        gyx = np.gradient(gy, axis=1).astype(np.float32)
        del gy

        gxy = np.gradient(gx, axis=0).astype(np.float32)
        gxx = np.gradient(gx, axis=1).astype(np.float32)
        del gx

        gxy_sym = (0.5 * (gxy + gyx)).astype(np.float32)
        del gxy, gyx

        Htr = (gxx + gyy).astype(np.float32)
        Hdet = (gxx * gyy - gxy_sym * gxy_sym).astype(np.float32)
        del gxx, gyy, gxy_sym

        disc = np.maximum(Htr * Htr - 4.0 * Hdet, 0.0).astype(np.float32)
        sqrt_disc = np.sqrt(disc).astype(np.float32)
        del disc

        l1 = (0.5 * (Htr + sqrt_disc)).astype(np.float32)
        l2 = (0.5 * (Htr - sqrt_disc)).astype(np.float32)
        del sqrt_disc

        cache["hess_raw_trace"] = Htr
        cache["hess_raw_det"] = Hdet
        cache["hess_raw_l1"] = l1
        cache["hess_raw_l2"] = l2
        cache["hess_raw_blobness"] = np.maximum(np.minimum(-l1, -l2), 0.0).astype(np.float32)

        del Htr, Hdet, l1, l2

        hp = cache.get("highpass")
        if hp is not None:
            hp32 = hp.astype(np.float32, copy=False)
            mu = ndi.uniform_filter(hp32, size=9).astype(np.float32)
            mu2 = ndi.uniform_filter(hp32 * hp32, size=9).astype(np.float32)
            var = np.maximum(mu2 - mu * mu, 0.0).astype(np.float32)
            cache["hp_local_std9"] = np.sqrt(var).astype(np.float32)
            del hp32, mu, mu2, var

        save_panel_cache(cid, pk, cache)
        release_panel_cache(cid, pk)
        del cache, raw_img
        gc.collect()

    n_done += 1
    log(progress_msg(n_done, total, t0_log, "Family 2 precompute full wells"))
    gc.collect()

log(f"RAW-domain response maps added to cache in {time.perf_counter() - t0_log:.1f}s")

In [ ]:
# ── Cell 21: Family 2 — RAW-domain detector responses + scale persistence ────
# Batched by crop: cache loaded once per crop.

log("="*70 + "\nFAMILY 2: RAW-domain detector responses + scale persistence")
t0_f2 = time.perf_counter()
F2_ROWS = []
RAW_SIGMAS = CFG.get("log_sigmas_feat", [1,2,3,4,5,6])

for cid, cu_crop in candidate_union.groupby("crop_id", sort=True):
    cid = str(cid)
    crow = CROP_LOOKUP.get(cid)
    if crow is None:
        for _, urow in cu_crop.iterrows():
            F2_ROWS.append({"union_id": urow["union_id"]})
        continue

    pmap = CROP_PANEL_CACHE.get(cid, {})
    primary = next((pk for pk in CFG["barcode_panels"] if pk in pmap), None)
    if primary is None:
        for _, urow in cu_crop.iterrows():
            F2_ROWS.append({"union_id": urow["union_id"]})
        continue

    cache = get_panel_cache_for_use(cid, primary)
    if cache is None:
        for _, urow in cu_crop.iterrows():
            F2_ROWS.append({"union_id": urow["union_id"]})
        continue

    try:
        raw_img = cache.get("raw")
        if raw_img is None:
            for _, urow in cu_crop.iterrows():
                F2_ROWS.append({"union_id": urow["union_id"]})
            continue

        H, W = raw_img.shape
        oy = float(crow["well_ymin_px"])
        ox = float(crow["well_xmin_px"])

        for _, urow in cu_crop.iterrows():
            uid = urow["union_id"]
            wy  = float(urow["union_medoid_well_y_px"])
            wx  = float(urow["union_medoid_well_x_px"])
            cy  = wy - oy
            cx  = wx - ox
            iy  = min(max(int(round(cy)), 0), H-1)
            ix  = min(max(int(round(cx)), 0), W-1)
            f   = {"union_id": uid}

            log_vals = []
            dog_vals = []
            for sig in RAW_SIGMAS:
                log_map = cache.get(f"log_raw_s{sig:.0f}")
                dog_map = cache.get(f"dog_raw_s{sig:.0f}")
                lv = bilinear_sample(log_map, cy, cx) if log_map is not None else np.nan
                dv = bilinear_sample(dog_map, cy, cx) if dog_map is not None else np.nan
                f[f"resp_log_raw_s{sig:.0f}_center"] = lv
                f[f"resp_dog_raw_s{sig:.0f}_center"] = dv
                log_vals.append(lv)
                dog_vals.append(dv)

            log_arr = np.array(log_vals, dtype=np.float32)
            dog_arr = np.array(dog_vals, dtype=np.float32)
            lf = log_arr[np.isfinite(log_arr)]
            df = dog_arr[np.isfinite(dog_arr)]

            f["resp_log_raw_max"]              = float(np.max(lf))  if len(lf) > 0 else np.nan
            f["resp_log_raw_mean"]             = float(np.mean(lf)) if len(lf) > 0 else np.nan
            f["resp_log_raw_argmax_sigma"]     = float(RAW_SIGMAS[int(np.nanargmax(log_arr))]) if np.isfinite(log_arr).any() else np.nan
            f["resp_log_raw_scale_cv"]         = float(np.std(lf)/max(np.mean(np.abs(lf)),1e-9)) if len(lf)>=2 else np.nan
            f["resp_log_raw_persistence_npos"] = int(np.sum(lf>0)) if len(lf)>0 else 0

            f["resp_dog_raw_max"]          = float(np.max(df))  if len(df) > 0 else np.nan
            f["resp_dog_raw_mean"]         = float(np.mean(df)) if len(df) > 0 else np.nan
            f["resp_dog_raw_argmax_sigma"] = float(RAW_SIGMAS[int(np.nanargmax(dog_arr))]) if np.isfinite(dog_arr).any() else np.nan
            f["resp_dog_raw_scale_cv"]     = float(np.std(df)/max(np.mean(np.abs(df)),1e-9)) if len(df)>=2 else np.nan

            for mk in ["matched_filter","radial_symmetry_map","hessian_blobness","highpass","peak_bg"]:
                arr = cache.get(mk)
                if arr is not None:
                    f[f"resp_{mk}_center"] = bilinear_sample(arr, cy, cx)
                    y0=max(0,iy-3); y1=min(H,iy+4); x0=max(0,ix-3); x1=min(W,ix+4)
                    patch = arr[y0:y1, x0:x1]
                    f[f"resp_{mk}_max3"]  = float(np.max(patch))  if patch.size > 0 else np.nan
                    f[f"resp_{mk}_mean3"] = float(np.mean(patch)) if patch.size > 0 else np.nan

            for mk in ["hess_raw_trace","hess_raw_det","hess_raw_l1","hess_raw_l2","hess_raw_blobness"]:
                arr = cache.get(mk)
                f[f"resp_{mk}_center"] = bilinear_sample(arr, cy, cx) if arr is not None else np.nan

            hpstd = cache.get("hp_local_std9")
            hp    = cache.get("highpass")
            if hpstd is not None and hp is not None:
                hp_center = bilinear_sample(hp, cy, cx)
                hp_std    = bilinear_sample(hpstd, cy, cx)
                f["resp_highpass_local_snr"] = hp_center / max(hp_std, 1e-9)
            else:
                f["resp_highpass_local_snr"] = np.nan

            rad = CFG["psf_fit_radius_px"]
            y0=max(0,iy-rad); y1=min(H,iy+rad+1); x0=max(0,ix-rad); x1=min(W,ix+rad+1)
            pp = raw_img[y0:y1, x0:x1]
            if pp.shape[0] >= 5 and pp.shape[1] >= 5:
                yy, xx = np.indices(pp.shape)
                p0 = [float(pp.max()-pp.min()), float(cy-y0), float(cx-x0), 2.5, 2.5, float(pp.min())]
                bounds = ([0,0,0,0.5,0.5,-np.inf],[np.inf,pp.shape[0]-1,pp.shape[1]-1,12.0,12.0,np.inf])
                try:
                    popt, _ = _curve_fit(_gauss2d, (yy,xx), pp.ravel(), p0=p0, bounds=bounds, maxfev=2000)
                    amp,fy,fx,sy,sx,off = popt
                    fitted = _gauss2d((yy,xx), *popt).reshape(pp.shape)
                    f["psf_raw_amplitude"]     = float(amp)
                    f["psf_raw_sigma_y"]        = float(sy)
                    f["psf_raw_sigma_x"]        = float(sx)
                    f["psf_raw_sigma_mean"]     = float((sy+sx)/2)
                    f["psf_raw_aspect_ratio"]   = float(max(sy,sx)/max(min(sy,sx),0.01))
                    f["psf_raw_residual_norm"]  = float(np.sqrt(np.mean((pp-fitted)**2))/max(abs(amp),1e-6))
                    f["psf_raw_fit_success"]    = 1.0
                except:
                    f["psf_raw_fit_success"] = 0.0
            else:
                f["psf_raw_fit_success"] = 0.0

            F2_ROWS.append(f)

        log(f"  F2 crop {cid}: {len(cu_crop)} candidates done")

    finally:
        release_panel_cache(cid, primary)
        del cache
        gc.collect()

f2_df = pd.DataFrame(F2_ROWS)
log(f"Family 2 done: {len([c for c in f2_df.columns if c!='union_id'])} cols in {time.perf_counter()-t0_f2:.1f}s")

In [ ]:
# ── Cell 19: Family 3 — Spatial / geometric / cluster (~40 features) ─────────

log("="*70+"\nFAMILY 3: Spatial / geometric")
t0_f3=time.perf_counter(); F3_PARTS=[]

for cid in candidate_union["crop_id"].unique():
    cu=candidate_union[candidate_union["crop_id"]==cid].copy()
    cc=candidate_clusters[candidate_clusters["crop_id"]==cid].copy() if "crop_id" in candidate_clusters.columns else pd.DataFrame()
    crow=CROP_LOOKUP.get(cid)
    if crow is None or len(cu)==0: continue
    coords=cu[["union_medoid_well_y_px","union_medoid_well_x_px"]].to_numpy(dtype=np.float64)
    scores=cu["top_method_score_norm"].to_numpy(dtype=np.float64)
    uids=cu["union_id"].tolist(); n=len(coords)
    cH=float(crow["well_ymax_px"]-crow["well_ymin_px"])
    cW=float(crow["well_xmax_px"]-crow["well_xmin_px"])
    cymin,cxmin=float(crow["well_ymin_px"]),float(crow["well_xmin_px"])
    cymax,cxmax=float(crow["well_ymax_px"]),float(crow["well_xmax_px"])
    tree=cKDTree(coords) if n>=2 else None
    rows=[]
    for i in range(n):
        f={"union_id":uids[i]}; y,x=coords[i]
        for r in CFG["neighbor_radii_px"]:
            f[f"n_neighbors_{int(r)}px"]=(len(tree.query_ball_point(coords[i],r))-1) if tree else 0
        if tree and n>=CFG["nn_k"]+1:
            dists,_=tree.query(coords[i],k=CFG["nn_k"]+1)
            for ki in range(CFG["nn_k"]): f[f"nn{ki+1}_dist_px"]=float(dists[ki+1])
        else:
            for ki in range(CFG["nn_k"]): f[f"nn{ki+1}_dist_px"]=np.nan
        f["local_density_10px"]=f.get("n_neighbors_10px",0)/(math.pi*100)
        stronger=scores>scores[i]
        if stronger.any():
            sc=coords[stronger]; ds=np.sqrt(np.sum((sc-coords[i])**2,axis=1))
            ni=np.argmin(ds)
            f["nearest_stronger_dist_px"]=float(ds[ni])
            f["nearest_stronger_score_ratio"]=safe_div(scores[i],scores[stronger][ni])
        else:
            f["nearest_stronger_dist_px"]=np.nan; f["nearest_stronger_score_ratio"]=np.nan; f["is_crop_peak"]=1.0
        f["border_dist_y_px"]=min(y-cymin,cymax-y); f["border_dist_x_px"]=min(x-cxmin,cxmax-x)
        f["border_distance_px"]=max(min(f["border_dist_y_px"],f["border_dist_x_px"]),0.0)
        f["border_distance_frac"]=safe_div(f["border_distance_px"],min(cH,cW)/2)
        rows.append(f)
    part=pd.DataFrame(rows)
    # cluster features
    if len(cc)>0 and "cluster_id" in cu.columns:
        cl_feats=cc[["cluster_id","n_union_candidates","cluster_radius_px"]].copy()
        part=part.merge(cu[["union_id","cluster_id"]],on="union_id",how="left")
        part=part.merge(cl_feats,on="cluster_id",how="left")
        part=part.rename(columns={"n_union_candidates":"cluster_n_union","cluster_radius_px":"cluster_r_px"})
    part["crop_n_candidates"]=n; part["crop_candidate_density"]=n/(cH*cW)
    F3_PARTS.append(part)

f3_df=pd.concat(F3_PARTS,ignore_index=True) if F3_PARTS else pd.DataFrame(columns=["union_id"])
log(f"Family 3 done: {len([c for c in f3_df.columns if c!='union_id'])} cols in {time.perf_counter()-t0_f3:.1f}s")


In [ ]:
print("candidate_raw shape:", candidate_raw.shape)
print("candidate_raw cols:", candidate_raw.columns.tolist()[:20])

In [ ]:
# ── Cell 23: Family 4 — Multi-proposer consensus (expanded topology) ─────────

log("="*70+"\nFAMILY 4: Multi-proposer consensus")
t0_f4=time.perf_counter(); F4_ROWS=[]; N_union=len(candidate_union)

ALL_METHODS=sorted(candidate_raw["method_id"].unique().tolist()) if len(candidate_raw) else []
ALL_FAMILIES=sorted(candidate_raw["method_family"].unique().tolist()) if len(candidate_raw) else []

_mem=candidate_union_membership.merge(
    candidate_raw[["raw_detection_id","method_id","method_family","detection_score_norm"]],
    on="raw_detection_id",how="left"
) if len(candidate_union_membership) else pd.DataFrame()

_msets={}; _fsets={}; _mscores={}; _fscores={}; _spread={}
if len(_mem):
    for uid,grp in _mem.groupby("union_id"):
        _msets[uid]=set(grp["method_id"].dropna().unique())
        _fsets[uid]=set(grp["method_family"].dropna().unique())
        _mscores[uid]={m:float(g["detection_score_norm"].max()) for m,g in grp.groupby("method_id")}
        _fscores[uid]={fam:float(g["detection_score_norm"].max()) for fam,g in grp.groupby("method_family")}
    if "member_distance_to_centroid_px" in candidate_union_membership.columns:
        for uid,grp in candidate_union_membership.groupby("union_id"):
            d=grp["member_distance_to_centroid_px"].dropna().to_numpy(np.float32)
            _spread[uid]=float(np.std(d)) if len(d)>1 else 0.0

for _,(_, urow) in enumerate(candidate_union.iterrows()):
    uid=urow["union_id"]; f={"union_id":uid}
    mp=_msets.get(uid,set()); fp_=_fsets.get(uid,set())

    f["proposer_support_count"]=int(urow.get("proposer_support_count",0))
    f["proposer_support_fraction"]=float(urow.get("proposer_support_fraction",0))
    f["family_support_count"]=int(urow.get("family_support_count",0))
    f["union_n_members"]=int(urow.get("union_n_members",0))
    f["union_radius_px"]=float(urow.get("union_radius_px",0))
    f["consensus_spread_std_px"]=_spread.get(uid,np.nan)

    for m in ALL_METHODS:
        f[f"vote_{m}"]=1.0 if m in mp else 0.0
    for fam in ALL_FAMILIES:
        f[f"vote_fam_{fam}"]=1.0 if fam in fp_ else 0.0

    msc=_mscores.get(uid,{})
    fsc=_fscores.get(uid,{})
    for m in ALL_METHODS:
        f[f"mscore_{m}"]=msc.get(m,np.nan)
    for fam in ALL_FAMILIES:
        f[f"fscore_{fam}"]=fsc.get(fam,np.nan)

    # topology summaries
    f["vote_any_mentor"]       =1.0 if any("mentor" in m for m in mp) else 0.0
    f["vote_any_multi_channel"]=1.0 if any("multi_channel" in m for m in mp) else 0.0
    f["vote_any_dog"]          =1.0 if any("dog" in m for m in mp) else 0.0
    f["vote_any_log"]          =1.0 if any(m in ("p05_log","log") for m in mp) else 0.0
    f["vote_any_starfish"]     =1.0 if any(m.startswith("sf_") for m in mp) else 0.0
    f["vote_any_perturb"]      =1.0 if any("perturb" in m for m in mp) else 0.0
    f["vote_any_perch"]        =1.0 if any("perch" in m for m in mp) else 0.0

    fam_n=max(len(fp_),1)
    f["family_diversity_frac"]=len(fp_)/max(len(ALL_FAMILIES),1)
    f["method_diversity_frac"]=len(mp)/max(len(ALL_METHODS),1)
    f["consensus_density"]    =safe_div(f["union_n_members"],max(f["union_radius_px"],1e-6))

    # independent-family agreements
    f["agree_dog_and_perturb"]     =1.0 if (f["vote_any_dog"] and f["vote_any_perturb"]) else 0.0
    f["agree_starfish_and_dog"]    =1.0 if (f["vote_any_starfish"] and f["vote_any_dog"]) else 0.0
    f["agree_starfish_and_mentor"] =1.0 if (f["vote_any_starfish"] and f["vote_any_mentor"]) else 0.0
    f["agree_perch_and_perturb"]   =1.0 if (f["vote_any_perch"] and f["vote_any_perturb"]) else 0.0

    F4_ROWS.append(f)

    if _n % 250 == 0 or _n == N_union:
        log(progress_msg(_n, N_union, t0_f4, "F4 candidates"))

f4_df=pd.DataFrame(F4_ROWS)
log(f"Family 4 done: {len([c for c in f4_df.columns if c!='union_id'])} cols in {time.perf_counter()-t0_f4:.1f}s")

In [ ]:
# ── Cell 24: Lightweight per-panel HP / local-noise prep for Family 5 ────────
# Family 5 now reuses base preprocessing whenever possible.
# We only ensure that per-panel highpass and hp_local_std9 exist if missing.
# No duplicate raw cache, no f5_hp_mean9.

log("Ensuring lightweight per-panel highpass/local-noise maps for Family 5...")
t0_hp = time.perf_counter()
n_crops_done = 0
total = len(CROP_PANEL_CACHE)

for cid, panel_cache in CROP_PANEL_CACHE.items():
    crow = CROP_LOOKUP.get(cid)
    if crow is None:
        continue

    log(f"[Family 5 prep] {cid}: start")

    for pk in CFG["barcode_panels"]:
        cache = get_panel_cache_for_use(cid, pk)
        if not isinstance(cache, dict):
            continue

        try:
            # If Family 5-compatible keys already exist, nothing to do
            if panel_cache_has_keys(cache, ["f5_hp", "f5_hp_std9"]):
                continue

            hp = cache.get("f5_hp", cache.get("highpass"))
            sd = cache.get("f5_hp_std9", cache.get("hp_local_std9"))

            # Reuse existing base-cache highpass if present
            if hp is None:
                raw = cache.get("raw")
                if raw is not None:
                    raw = np.asarray(raw, dtype=np.float32)
                    hp = np.maximum(raw - ndi.gaussian_filter(raw, 4.0), 0.0).astype(np.float32)
                else:
                    hp = None

            # Compute local std only if still missing
            if hp is not None and sd is None:
                hp32 = np.asarray(hp, dtype=np.float32)
                mu  = ndi.uniform_filter(hp32, size=9)
                mu2 = ndi.uniform_filter(hp32 * hp32, size=9)
                sd  = np.sqrt(np.maximum(mu2 - mu * mu, 0.0)).astype(np.float32)

            changed = False
            if hp is not None and "f5_hp" not in cache:
                cache["f5_hp"] = np.asarray(hp, dtype=np.float32)
                changed = True
            if sd is not None and "f5_hp_std9" not in cache:
                cache["f5_hp_std9"] = np.asarray(sd, dtype=np.float32)
                changed = True

            if changed:
                save_panel_cache(cid, pk, cache)

        except Exception as e:
            log(f"  [warn] {cid}/{pk}: {type(e).__name__}: {e}")

        finally:
            release_panel_cache(cid, pk)
            gc.collect()

    n_crops_done += 1
    log(progress_msg(n_crops_done, total, t0_hp, "Family 5 lightweight prep"))

log(f"Family 5 lightweight prep done in {time.perf_counter() - t0_hp:.1f}s")

In [ ]:
# ── Cell 25: Family 5 — Barcode / multichannel RAW coherence + local noise ───
# Exact same feature semantics as before, but vectorized and lower-RAM:
# - one panel cache at a time
# - exact clipped rectangle means via integral images
# - no per-candidate iterrows loop

log("="*70 + "\nFAMILY 5: Barcode / multichannel RAW coherence + local noise [vectorized]")
t0_f5 = time.perf_counter()

import gc
import numpy as np
import pandas as pd

# ---------------------------
# Helpers
# ---------------------------
def _integral_image(arr):
    """
    Summed-area table with 1-pixel zero padding.
    Output shape = (H+1, W+1)
    Uses float64 for numerical stability.
    """
    arr64 = np.asarray(arr, dtype=np.float64)
    ii = np.empty((arr64.shape[0] + 1, arr64.shape[1] + 1), dtype=np.float64)
    ii[0, :] = 0.0
    ii[:, 0] = 0.0
    ii[1:, 1:] = arr64.cumsum(axis=0).cumsum(axis=1)
    return ii

def _rect_means_from_integral(ii, y0, y1, x0, x1):
    """
    Exact means over rectangles [y0:y1, x0:x1] with exclusive upper bounds.
    y0,y1,x0,x1 are int arrays of same length.
    Matches np.mean(arr[y0:y1, x0:x1]) exactly up to FP precision.
    """
    sums = ii[y1, x1] - ii[y0, x1] - ii[y1, x0] + ii[y0, x0]
    counts = (y1 - y0) * (x1 - x0)
    out = np.full(len(counts), np.nan, dtype=np.float64)
    valid = counts > 0
    out[valid] = sums[valid] / counts[valid]
    return out

def _safe_nan_ratio(num, den, floor=None):
    """
    Elementwise ratio with denominator floor and NaN propagation.
    """
    num = np.asarray(num, dtype=np.float64)
    den = np.asarray(den, dtype=np.float64).copy()
    out = np.full(num.shape, np.nan, dtype=np.float64)

    finite = np.isfinite(num) & np.isfinite(den)
    if floor is not None:
        den[finite] = np.maximum(den[finite], floor)
    nz = finite & (den != 0)
    out[nz] = num[nz] / den[nz]
    return out

def _nanmean_rows(a):
    with np.errstate(invalid="ignore"):
        return np.nanmean(a, axis=1)

def _nanmin_rows(a):
    with np.errstate(invalid="ignore"):
        return np.nanmin(a, axis=1)

def _nanmax_rows(a):
    with np.errstate(invalid="ignore"):
        return np.nanmax(a, axis=1)

def _nanstd_rows(a):
    with np.errstate(invalid="ignore"):
        return np.nanstd(a, axis=1)

# ---------------------------
# Main
# ---------------------------
F5_FRAMES = []
barcode_panels = list(CFG["barcode_panels"])

for cid, cu_crop in candidate_union.groupby("crop_id", sort=True):
    cid = str(cid)
    crow = CROP_LOOKUP.get(cid)

    # Preserve prior behavior when crop lookup is missing
    if crow is None:
        tmp = pd.DataFrame({"union_id": cu_crop["union_id"].to_numpy()})
        F5_FRAMES.append(tmp)
        log(f"  F5 crop {cid}: missing crop lookup -> emitted union_id only for {len(tmp)} rows")
        continue

    # Pull crop metadata
    wid = str(cu_crop.iloc[0]["well_id"])
    on_p = list(CFG["barcode_on_panels"].get(wid, []))
    off_p = list(CFG["barcode_off_panels"].get(wid, []))

    oy = float(crow["well_ymin_px"])
    ox = float(crow["well_xmin_px"])
    H = int(crow["well_ymax_px"]) - int(crow["well_ymin_px"])
    W = int(crow["well_xmax_px"]) - int(crow["well_xmin_px"])

    # Candidate coordinates for this crop
    union_ids = cu_crop["union_id"].to_numpy()
    wy = cu_crop["union_medoid_well_y_px"].to_numpy(dtype=np.float64)
    wx = cu_crop["union_medoid_well_x_px"].to_numpy(dtype=np.float64)

    cy = wy - oy
    cx = wx - ox
    iy = np.rint(cy).astype(np.int64)
    ix = np.rint(cx).astype(np.int64)
    iy = np.clip(iy, 0, H - 1)
    ix = np.clip(ix, 0, W - 1)

    n = len(cu_crop)

    # Output frame scaffold
    crop_out = pd.DataFrame({"union_id": union_ids})

    # Store per-panel center metrics for later ON/OFF aggregation
    panel_raw_center = {}
    panel_hp_center  = {}
    panel_snr_center = {}

    # Precompute rectangle bounds exactly matching old code
    r = 3
    yp0 = np.maximum(0, iy - r)
    yp1 = np.minimum(H, iy + r + 1)
    xp0 = np.maximum(0, ix - r)
    xp1 = np.minimum(W, ix + r + 1)

    rg0 = np.maximum(0, iy - 7)
    rg1 = np.minimum(H, iy + 8)
    xg0 = np.maximum(0, ix - 7)
    xg1 = np.minimum(W, ix + 8)

    # Process ONE panel at a time to keep RAM bounded
    for pk in barcode_panels:
        short = pk.replace("_", "").lower()
        cache = get_panel_cache_for_use(cid, pk)

        try:
            # Default all outputs for this panel to NaN
            raw_center   = np.full(n, np.nan, dtype=np.float64)
            raw_inner    = np.full(n, np.nan, dtype=np.float64)
            raw_ring1    = np.full(n, np.nan, dtype=np.float64)
            raw_contrast = np.full(n, np.nan, dtype=np.float64)

            hp_center    = np.full(n, np.nan, dtype=np.float64)
            hp_inner     = np.full(n, np.nan, dtype=np.float64)
            hp_ring1     = np.full(n, np.nan, dtype=np.float64)
            hp_contrast  = np.full(n, np.nan, dtype=np.float64)
            hp_local_snr = np.full(n, np.nan, dtype=np.float64)

            if cache is not None:
                raw = cache.get("f5_raw", cache.get("raw"))
                hp  = cache.get("f5_hp", cache.get("highpass"))
                sd  = cache.get("f5_hp_std9", cache.get("hp_local_std9"))

                if (raw is not None) and (hp is not None):
                    raw = np.asarray(raw)
                    hp  = np.asarray(hp)

                    # Centers
                    raw_center = raw[iy, ix].astype(np.float64, copy=False)
                    hp_center  = hp[iy, ix].astype(np.float64, copy=False)

                    # Exact clipped window means via integral images
                    raw_ii = _integral_image(raw)
                    hp_ii  = _integral_image(hp)

                    raw_inner = _rect_means_from_integral(raw_ii, yp0, yp1, xp0, xp1)
                    hp_inner  = _rect_means_from_integral(hp_ii,  yp0, yp1, xp0, xp1)

                    raw_ring1 = _rect_means_from_integral(raw_ii, rg0, rg1, xg0, xg1)
                    hp_ring1  = _rect_means_from_integral(hp_ii,  rg0, rg1, xg0, xg1)

                    raw_contrast = raw_inner - raw_ring1
                    hp_contrast  = hp_inner - hp_ring1

                    if sd is not None:
                        sd = np.asarray(sd)
                        sd_center = sd[iy, ix].astype(np.float64, copy=False)
                        den = np.maximum(sd_center, 1e-9)
                        hp_local_snr = hp_center / den

                    # Drop large temporaries promptly
                    del raw_ii, hp_ii

            crop_out[f"bc_{short}_raw_center"]   = raw_center
            crop_out[f"bc_{short}_raw_inner"]    = raw_inner
            crop_out[f"bc_{short}_raw_ring1"]    = raw_ring1
            crop_out[f"bc_{short}_raw_contrast"] = raw_contrast
            crop_out[f"bc_{short}_hp_center"]    = hp_center
            crop_out[f"bc_{short}_hp_inner"]     = hp_inner
            crop_out[f"bc_{short}_hp_ring1"]     = hp_ring1
            crop_out[f"bc_{short}_hp_contrast"]  = hp_contrast
            crop_out[f"bc_{short}_hp_local_snr"] = hp_local_snr

            panel_raw_center[pk] = raw_center
            panel_hp_center[pk]  = hp_center
            panel_snr_center[pk] = hp_local_snr

        finally:
            if cache is not None:
                release_panel_cache(cid, pk)
            gc.collect()

    # ---------------------------
    # Aggregate ON / OFF barcode stats
    # ---------------------------
    def _stack_metric(metric_dict, panel_list):
        if len(panel_list) == 0:
            return np.full((n, 0), np.nan, dtype=np.float64)
        return np.column_stack([
            metric_dict.get(pk, np.full(n, np.nan, dtype=np.float64))
            for pk in panel_list
        ])

    on_raw_mat  = _stack_metric(panel_raw_center, on_p)
    off_raw_mat = _stack_metric(panel_raw_center, off_p)
    on_hp_mat   = _stack_metric(panel_hp_center,  on_p)
    off_hp_mat  = _stack_metric(panel_hp_center,  off_p)
    on_snr_mat  = _stack_metric(panel_snr_center, on_p)
    off_snr_mat = _stack_metric(panel_snr_center, off_p)

    # These warnings are expected for all-NaN rows
    with np.errstate(all="ignore", invalid="ignore"):
        bc_on_raw_mean  = _nanmean_rows(on_raw_mat)  if on_raw_mat.shape[1]  else np.full(n, np.nan)
        bc_on_raw_min   = _nanmin_rows(on_raw_mat)   if on_raw_mat.shape[1]  else np.full(n, np.nan)
        bc_on_raw_max   = _nanmax_rows(on_raw_mat)   if on_raw_mat.shape[1]  else np.full(n, np.nan)
        bc_off_raw_mean = _nanmean_rows(off_raw_mat) if off_raw_mat.shape[1] else np.full(n, np.nan)
        bc_off_raw_max  = _nanmax_rows(off_raw_mat)  if off_raw_mat.shape[1] else np.full(n, np.nan)

        bc_on_hp_mean   = _nanmean_rows(on_hp_mat)   if on_hp_mat.shape[1]   else np.full(n, np.nan)
        bc_on_hp_min    = _nanmin_rows(on_hp_mat)    if on_hp_mat.shape[1]   else np.full(n, np.nan)
        bc_on_hp_max    = _nanmax_rows(on_hp_mat)    if on_hp_mat.shape[1]   else np.full(n, np.nan)
        bc_off_hp_mean  = _nanmean_rows(off_hp_mat)  if off_hp_mat.shape[1]  else np.full(n, np.nan)
        bc_off_hp_max   = _nanmax_rows(off_hp_mat)   if off_hp_mat.shape[1]  else np.full(n, np.nan)

        bc_on_snr_mean  = _nanmean_rows(on_snr_mat)  if on_snr_mat.shape[1]  else np.full(n, np.nan)
        bc_on_snr_min   = _nanmin_rows(on_snr_mat)   if on_snr_mat.shape[1]  else np.full(n, np.nan)
        bc_off_snr_max  = _nanmax_rows(off_snr_mat)  if off_snr_mat.shape[1] else np.full(n, np.nan)

    crop_out["bc_on_raw_mean"]  = bc_on_raw_mean
    crop_out["bc_on_raw_min"]   = bc_on_raw_min
    crop_out["bc_on_raw_max"]   = bc_on_raw_max
    crop_out["bc_off_raw_mean"] = bc_off_raw_mean
    crop_out["bc_off_raw_max"]  = bc_off_raw_max

    crop_out["bc_on_hp_mean"]   = bc_on_hp_mean
    crop_out["bc_on_hp_min"]    = bc_on_hp_min
    crop_out["bc_on_hp_max"]    = bc_on_hp_max
    crop_out["bc_off_hp_mean"]  = bc_off_hp_mean
    crop_out["bc_off_hp_max"]   = bc_off_hp_max

    crop_out["bc_on_snr_mean"]  = bc_on_snr_mean
    crop_out["bc_on_snr_min"]   = bc_on_snr_min
    crop_out["bc_off_snr_max"]  = bc_off_snr_max

    crop_out["bc_on_off_raw_ratio"] = _safe_nan_ratio(bc_on_raw_mean, bc_off_raw_max, floor=1.0)
    crop_out["bc_on_off_hp_ratio"]  = _safe_nan_ratio(bc_on_hp_mean,  bc_off_hp_max,  floor=1.0)
    crop_out["bc_on_off_snr_ratio"] = _safe_nan_ratio(bc_on_snr_mean, bc_off_snr_max, floor=1e-6)

    crop_out["bc_on_minus_off_raw"] = np.where(
        np.isfinite(bc_on_raw_mean) & np.isfinite(bc_off_raw_max),
        bc_on_raw_mean - bc_off_raw_max,
        np.nan
    )
    crop_out["bc_on_minus_off_hp"] = np.where(
        np.isfinite(bc_on_hp_mean) & np.isfinite(bc_off_hp_max),
        bc_on_hp_mean - bc_off_hp_max,
        np.nan
    )
    crop_out["bc_on_minus_off_snr"] = np.where(
        np.isfinite(bc_on_snr_mean) & np.isfinite(bc_off_snr_max),
        bc_on_snr_mean - bc_off_snr_max,
        np.nan
    )

    # Match original logic: CV only if at least 2 finite ON values
    if on_hp_mat.shape[1]:
        hp_counts = np.sum(np.isfinite(on_hp_mat), axis=1)
        hp_mean   = _nanmean_rows(on_hp_mat)
        hp_std    = _nanstd_rows(on_hp_mat)
        crop_out["bc_on_hp_cv"] = np.where(
            hp_counts >= 2,
            _safe_nan_ratio(hp_std, hp_mean + 1e-9),
            np.nan
        )
    else:
        crop_out["bc_on_hp_cv"] = np.nan

    if on_snr_mat.shape[1]:
        snr_counts = np.sum(np.isfinite(on_snr_mat), axis=1)
        snr_mean   = _nanmean_rows(on_snr_mat)
        snr_std    = _nanstd_rows(on_snr_mat)
        crop_out["bc_on_snr_cv"] = np.where(
            snr_counts >= 2,
            _safe_nan_ratio(snr_std, snr_mean + 1e-9),
            np.nan
        )
    else:
        crop_out["bc_on_snr_cv"] = np.nan

    # Profile L2 norm across all barcode panels using hp centers
    if len(barcode_panels):
        hp_all = np.column_stack([
            panel_hp_center.get(pk, np.full(n, np.nan, dtype=np.float64))
            for pk in barcode_panels
        ])
        hp_all_zeroed = np.where(np.isfinite(hp_all), hp_all, 0.0)
        any_finite = np.any(np.isfinite(hp_all), axis=1)
        l2 = np.sqrt(np.sum(hp_all_zeroed ** 2, axis=1))
        crop_out["bc_profile_l2_norm"] = np.where(any_finite, l2, np.nan)
    else:
        crop_out["bc_profile_l2_norm"] = np.nan

    # 561 / 638 ON means
    on_561_panels = [pk for pk in on_p if "561" in pk]
    on_638_panels = [pk for pk in on_p if "638" in pk]

    on_561_mat = _stack_metric(panel_hp_center, on_561_panels)
    on_638_mat = _stack_metric(panel_hp_center, on_638_panels)

    with np.errstate(all="ignore", invalid="ignore"):
        bc_on_561_hp_mean = _nanmean_rows(on_561_mat) if on_561_mat.shape[1] else np.full(n, np.nan)
        bc_on_638_hp_mean = _nanmean_rows(on_638_mat) if on_638_mat.shape[1] else np.full(n, np.nan)

    crop_out["bc_on_561_hp_mean"] = bc_on_561_hp_mean
    crop_out["bc_on_638_hp_mean"] = bc_on_638_hp_mean
    crop_out["bc_on_561_638_hp_ratio"] = _safe_nan_ratio(bc_on_561_hp_mean, bc_on_638_hp_mean, floor=None)

    F5_FRAMES.append(crop_out)
    log(f"  F5 crop {cid}: {n} candidates done")

    # Free crop-local arrays aggressively on 8 GB RAM
    del crop_out
    del panel_raw_center, panel_hp_center, panel_snr_center
    gc.collect()

f5_df = pd.concat(F5_FRAMES, ignore_index=True) if F5_FRAMES else pd.DataFrame(columns=["union_id"])
log(f"Family 5 done: {len([c for c in f5_df.columns if c != 'union_id'])} cols in {time.perf_counter() - t0_f5:.1f}s")

In [ ]:
# ── Cell 26: Family 6 removed in RAW-LOCAL rewrite ────────────────────────────
# Crop/image-regime features were deployment-invalid because they broadcast
# crop-global statistics into every candidate and force crop-specific inference.

log("="*70 + "\nFAMILY 6: REMOVED")
f6_df=pd.DataFrame({"union_id":candidate_union["union_id"].tolist()})
log("Family 6 intentionally removed: no crop-global regime features.")

In [ ]:
# ── Cell 27: Family 7 precompute removed (Option A) ───────────────────────────
# Structure-tensor maps are now computed on the fly inside Cell 28
# for the primary panel only. This avoids large redundant full-well caches.

log("Family 7 precompute removed: structure tensor will be computed on the fly in Cell 28.")

In [ ]:
# ── Cell 28: Family 7 — RAW-domain symmetry / shape / curvature ──────────────
# Option A rewrite: compute structure-tensor maps on the fly for the primary
# panel only. No Family 7 precompute cache.

log("=" * 70 + "\nFAMILY 7: RAW-domain symmetry / shape / curvature")
t0_f7 = time.perf_counter()
F7_ROWS = []

sig = float(CFG["struct_tensor_sigma"])

for cid, cu_crop in candidate_union.groupby("crop_id", sort=True):
    cid = str(cid)
    crow = CROP_LOOKUP.get(cid)

    if crow is None:
        for uid in cu_crop["union_id"].tolist():
            F7_ROWS.append({"union_id": uid})
        continue

    pmap = CROP_PANEL_CACHE.get(cid, {})
    primary = next((pk for pk in CFG["barcode_panels"] if pk in pmap), None)
    if primary is None:
        for uid in cu_crop["union_id"].tolist():
            F7_ROWS.append({"union_id": uid})
        continue

    cache = get_panel_cache_for_use(cid, primary)
    if not isinstance(cache, dict):
        for uid in cu_crop["union_id"].tolist():
            F7_ROWS.append({"union_id": uid})
        continue

    try:
        raw_img = cache.get("raw")
        hp_img  = cache.get("highpass")
        base    = hp_img if hp_img is not None else raw_img

        if base is None:
            for uid in cu_crop["union_id"].tolist():
                F7_ROWS.append({"union_id": uid})
            continue

        base = np.asarray(base, dtype=np.float32)
        H, W = base.shape
        oy = float(crow["well_ymin_px"])
        ox = float(crow["well_xmin_px"])

        rs  = cache.get("radial_symmetry_map")
        l1m = cache.get("hess_raw_l1")
        l2m = cache.get("hess_raw_l2")

        # Compute structure tensor on the fly for this one primary panel
        sm = ndi.gaussian_filter(base, sigma=sig)
        gy, gx = np.gradient(sm)
        Jxx = ndi.gaussian_filter(gx * gx, sigma=sig).astype(np.float32)
        Jyy = ndi.gaussian_filter(gy * gy, sigma=sig).astype(np.float32)
        Jxy = ndi.gaussian_filter(gx * gy, sigma=sig).astype(np.float32)

        for urow in cu_crop.itertuples(index=False):
            uid = urow.union_id
            wy  = float(urow.union_medoid_well_y_px)
            wx  = float(urow.union_medoid_well_x_px)
            cy  = wy - oy
            cx  = wx - ox
            iy  = min(max(int(round(cy)), 0), H - 1)
            ix  = min(max(int(round(cx)), 0), W - 1)

            f = {"union_id": uid}

            if rs is not None:
                f["shape_radial_symmetry"] = bilinear_sample(rs, cy, cx)
                vi, _ = extract_annulus(rs, cy, cx, _INNER_MASK, _R_MAX)
                vr, _ = extract_annulus(rs, cy, cx, _RING1_MASK, _R_MAX)
                f["shape_radial_sym_residual"] = (
                    float(np.mean(vi)) - float(np.mean(vr))
                    if len(vi) > 0 and len(vr) > 0 else np.nan
                )
            else:
                f["shape_radial_symmetry"] = np.nan
                f["shape_radial_sym_residual"] = np.nan

            jxx = float(Jxx[iy, ix])
            jyy = float(Jyy[iy, ix])
            jxy = float(Jxy[iy, ix])
            tr  = jxx + jyy
            det = jxx * jyy - jxy * jxy
            disc = max(tr * tr - 4.0 * det, 0.0)
            ev1 = 0.5 * (tr + math.sqrt(disc))
            ev2 = 0.5 * (tr - math.sqrt(disc))
            lm  = max(abs(ev1), abs(ev2))
            ln  = min(abs(ev1), abs(ev2))

            f["shape_eccentricity"] = safe_div(ln, lm) if lm > 1e-12 else np.nan
            f["shape_struct_trace"] = float(tr)
            f["shape_struct_det"]   = float(det)

            if l1m is not None and l2m is not None:
                l1 = float(bilinear_sample(l1m, cy, cx))
                l2 = float(bilinear_sample(l2m, cy, cx))
                f["shape_hess_l1"]    = l1
                f["shape_hess_l2"]    = l2
                f["shape_hess_ratio"] = safe_div(min(abs(l1), abs(l2)), max(abs(l1), abs(l2), 1e-9))
            else:
                f["shape_hess_l1"]    = np.nan
                f["shape_hess_l2"]    = np.nan
                f["shape_hess_ratio"] = np.nan

            rh = CFG["hu_patch_radius_px"]
            pp = base[max(0, iy - rh):min(H, iy + rh + 1), max(0, ix - rh):min(W, ix + rh + 1)]

            if pp.shape[0] >= 5 and pp.shape[1] >= 5:
                try:
                    thr = float(np.mean(pp))
                    bn  = (pp > thr).astype(np.uint8)
                    if bn.sum() > 0:
                        m  = measure.moments_central(bn.astype(float), order=3)
                        nm = measure.moments_normalized(m, order=3)
                        hu = measure.moments_hu(nm)
                        for k in range(min(CFG["n_hu_moments"], len(hu))):
                            v = hu[k]
                            f[f"shape_hu{k+1}"] = float(-np.sign(v) * np.log10(max(abs(v), 1e-30)))
                except Exception:
                    pass

            vi, cov = extract_annulus(base, cy, cx, _INNER_MASK, _R_MAX)
            if cov > 0.5 and len(vi) > 3:
                hm = (float(np.max(vi)) + float(np.min(vi))) / 2.0
                f["shape_circularity"] = float(np.sum(vi > hm)) / len(vi)
            else:
                f["shape_circularity"] = np.nan

            F7_ROWS.append(f)

        log(f"  F7 crop {cid}: {len(cu_crop)} candidates done")

    finally:
        release_panel_cache(cid, primary)
        del cache
        try:
            del base, sm, gy, gx, Jxx, Jyy, Jxy
        except Exception:
            pass
        gc.collect()

f7_df = pd.DataFrame(F7_ROWS)
log(f"Family 7 done: {len([c for c in f7_df.columns if c != 'union_id'])} cols in {time.perf_counter() - t0_f7:.1f}s")

In [ ]:
# ── Cell 30: Family 8 removed + Family 9 rewritten ────────────────────────────

# ── Family 8 removed ──────────────────────────────────────────────────────────
log("=" * 70 + "\nFAMILY 8: REMOVED")
f8_df = pd.DataFrame({"union_id": candidate_union["union_id"].tolist()})
log("Family 8 intentionally removed: no crop-relative ranks or crop-relative standardization.")

# ── Family 9: per-panel RAW evidence maps — streamed, no precompute cache ────
log("=" * 70 + "\nFAMILY 9: Per-panel RAW evidence maps [streamed]")
t0_f9 = time.perf_counter()

import gc
import numpy as np
import pandas as pd

def _bilinear_sample_vec(arr, ys, xs):
    """
    Vectorized bilinear sampling.
    ys, xs are float arrays in image coordinates.
    """
    if arr is None:
        return np.full(len(ys), np.nan, dtype=np.float64)

    a = np.asarray(arr, dtype=np.float64)
    H, W = a.shape

    ys = np.asarray(ys, dtype=np.float64)
    xs = np.asarray(xs, dtype=np.float64)

    valid = np.isfinite(ys) & np.isfinite(xs)
    out = np.full(ys.shape, np.nan, dtype=np.float64)
    if not np.any(valid):
        return out

    yv = np.clip(ys[valid], 0.0, H - 1.0)
    xv = np.clip(xs[valid], 0.0, W - 1.0)

    y0 = np.floor(yv).astype(np.int64)
    x0 = np.floor(xv).astype(np.int64)
    y1 = np.minimum(y0 + 1, H - 1)
    x1 = np.minimum(x0 + 1, W - 1)

    dy = yv - y0
    dx = xv - x0

    v00 = a[y0, x0]
    v01 = a[y0, x1]
    v10 = a[y1, x0]
    v11 = a[y1, x1]

    out_valid = (
        (1.0 - dy) * (1.0 - dx) * v00 +
        (1.0 - dy) * dx         * v01 +
        dy         * (1.0 - dx) * v10 +
        dy         * dx         * v11
    )
    out[valid] = out_valid
    return out

def _nanmean_rows_local(a):
    with np.errstate(invalid="ignore"):
        return np.nanmean(a, axis=1)

def _nanmax_rows_local(a):
    with np.errstate(invalid="ignore"):
        return np.nanmax(a, axis=1)

F9_FRAMES = []
barcode_panels = list(CFG["barcode_panels"])

for cid, cu_crop in candidate_union.groupby("crop_id", sort=True):
    cid = str(cid)
    crow = CROP_LOOKUP.get(cid)

    if crow is None:
        tmp = pd.DataFrame({"union_id": cu_crop["union_id"].to_numpy()})
        F9_FRAMES.append(tmp)
        log(f"  F9 crop {cid}: missing crop lookup -> emitted union_id only for {len(tmp)} rows")
        continue

    wid  = str(cu_crop.iloc[0]["well_id"])
    on_p = list(CFG["barcode_on_panels"].get(wid, []))
    off_p = list(CFG["barcode_off_panels"].get(wid, []))

    ymin = int(crow["well_ymin_px"])
    ymax = int(crow["well_ymax_px"])
    xmin = int(crow["well_xmin_px"])
    xmax = int(crow["well_xmax_px"])
    oy   = float(ymin)
    ox   = float(xmin)

    union_ids = cu_crop["union_id"].to_numpy()
    wy = cu_crop["union_medoid_well_y_px"].to_numpy(dtype=np.float64)
    wx = cu_crop["union_medoid_well_x_px"].to_numpy(dtype=np.float64)

    cy = wy - oy
    cx = wx - ox
    n  = len(cu_crop)

    crop_out = pd.DataFrame({"union_id": union_ids})

    panel_mf_center   = {}
    panel_hess_center = {}
    panel_rs_center   = {}

    for pk in barcode_panels:
        short = pk.replace("_", "").lower()
        cache = get_panel_cache_for_use(cid, pk)

        try:
            raw = None
            hp  = None
            hp_std9 = None

            if isinstance(cache, dict):
                raw = cache.get("raw")
                hp  = cache.get("highpass")
                hp_std9 = cache.get("hp_local_std9")

            if raw is None:
                try:
                    raw = load_crop_raw(wid, pk, ymin, ymax, xmin, xmax).astype(np.float32)
                except Exception as e:
                    log(f"  [warn] {cid}/{pk}: {type(e).__name__}: {e}")
                    raw = None

            if raw is None:
                for base_name in ["log_raw", "dog_raw", "mf_raw", "hess_raw", "rs_raw", "hp", "hp_std9"]:
                    crop_out[f"f9_{short}_{base_name}_center"] = np.nan
                crop_out[f"f9_{short}_hp_local_snr"] = np.nan
                panel_mf_center[pk]   = np.full(n, np.nan, dtype=np.float64)
                panel_hess_center[pk] = np.full(n, np.nan, dtype=np.float64)
                panel_rs_center[pk]   = np.full(n, np.nan, dtype=np.float64)
                continue

            raw = np.asarray(raw, dtype=np.float32)

            log_raw  = (-(4.0) * ndi.gaussian_laplace(raw, sigma=2.0)).astype(np.float32)
            dog_raw  = np.maximum(ndi.gaussian_filter(raw, 1.0) - ndi.gaussian_filter(raw, 3.0), 0.0).astype(np.float32)
            mf_raw   = matched_filter_response_fn(raw, CFG["matched_filter_sigma_px"])
            hess_raw = hessian_blobness_map_fn(raw, CFG["hessian_sigma_px"])
            rs_raw   = radial_symmetry_map_fn(raw, CFG["radial_symmetry_radius_px"])

            if hp is None:
                hp = np.maximum(raw - ndi.gaussian_filter(raw, 4.0), 0.0).astype(np.float32)
            else:
                hp = np.asarray(hp, dtype=np.float32)

            if hp_std9 is None:
                mu  = ndi.uniform_filter(hp, size=9)
                mu2 = ndi.uniform_filter(hp * hp, size=9)
                hp_std9 = np.sqrt(np.maximum(mu2 - mu * mu, 0.0)).astype(np.float32)
            else:
                hp_std9 = np.asarray(hp_std9, dtype=np.float32)

            log_center  = _bilinear_sample_vec(log_raw,  cy, cx)
            dog_center  = _bilinear_sample_vec(dog_raw,  cy, cx)
            mf_center   = _bilinear_sample_vec(mf_raw,   cy, cx)
            hess_center = _bilinear_sample_vec(hess_raw, cy, cx)
            rs_center   = _bilinear_sample_vec(rs_raw,   cy, cx)
            hp_center   = _bilinear_sample_vec(hp,       cy, cx)
            sd_center   = _bilinear_sample_vec(hp_std9,  cy, cx)

            hp_local_snr = np.full(n, np.nan, dtype=np.float64)
            valid_snr = np.isfinite(hp_center) & np.isfinite(sd_center)
            hp_local_snr[valid_snr] = hp_center[valid_snr] / np.maximum(sd_center[valid_snr], 1e-9)

            crop_out[f"f9_{short}_log_raw_center"]  = log_center
            crop_out[f"f9_{short}_dog_raw_center"]  = dog_center
            crop_out[f"f9_{short}_mf_raw_center"]   = mf_center
            crop_out[f"f9_{short}_hess_raw_center"] = hess_center
            crop_out[f"f9_{short}_rs_raw_center"]   = rs_center
            crop_out[f"f9_{short}_hp_center"]       = hp_center
            crop_out[f"f9_{short}_hp_std9_center"]  = sd_center
            crop_out[f"f9_{short}_hp_local_snr"]    = hp_local_snr

            panel_mf_center[pk]   = mf_center
            panel_hess_center[pk] = hess_center
            panel_rs_center[pk]   = rs_center

        finally:
            if cache is not None:
                release_panel_cache(cid, pk)
            try:
                del raw, log_raw, dog_raw, mf_raw, hess_raw, rs_raw, hp, hp_std9
            except Exception:
                pass
            gc.collect()

    def _stack_metric(metric_dict, panel_list):
        if len(panel_list) == 0:
            return np.full((n, 0), np.nan, dtype=np.float64)
        return np.column_stack([
            metric_dict.get(pk, np.full(n, np.nan, dtype=np.float64))
            for pk in panel_list
        ])

    on_mf_mat    = _stack_metric(panel_mf_center,   on_p)
    off_mf_mat   = _stack_metric(panel_mf_center,   off_p)
    on_hess_mat  = _stack_metric(panel_hess_center, on_p)
    off_hess_mat = _stack_metric(panel_hess_center, off_p)
    on_rs_mat    = _stack_metric(panel_rs_center,   on_p)
    off_rs_mat   = _stack_metric(panel_rs_center,   off_p)

    with np.errstate(all="ignore", invalid="ignore"):
        f9_on_mf_mean    = _nanmean_rows_local(on_mf_mat)    if on_mf_mat.shape[1]    else np.full(n, np.nan)
        f9_off_mf_max    = _nanmax_rows_local(off_mf_mat)    if off_mf_mat.shape[1]   else np.full(n, np.nan)
        f9_on_hess_mean  = _nanmean_rows_local(on_hess_mat)  if on_hess_mat.shape[1]  else np.full(n, np.nan)
        f9_off_hess_max  = _nanmax_rows_local(off_hess_mat)  if off_hess_mat.shape[1] else np.full(n, np.nan)
        f9_on_rs_mean    = _nanmean_rows_local(on_rs_mat)    if on_rs_mat.shape[1]    else np.full(n, np.nan)
        f9_off_rs_max    = _nanmax_rows_local(off_rs_mat)    if off_rs_mat.shape[1]   else np.full(n, np.nan)

    crop_out["f9_on_mf_mean"]   = f9_on_mf_mean
    crop_out["f9_off_mf_max"]   = f9_off_mf_max
    crop_out["f9_on_hess_mean"] = f9_on_hess_mean
    crop_out["f9_off_hess_max"] = f9_off_hess_max
    crop_out["f9_on_rs_mean"]   = f9_on_rs_mean
    crop_out["f9_off_rs_max"]   = f9_off_rs_max

    crop_out["f9_on_off_mf_ratio"] = np.where(
        np.isfinite(f9_on_mf_mean) & np.isfinite(f9_off_mf_max),
        f9_on_mf_mean / np.maximum(f9_off_mf_max, 1e-9),
        np.nan
    )
    crop_out["f9_on_off_hess_ratio"] = np.where(
        np.isfinite(f9_on_hess_mean) & np.isfinite(f9_off_hess_max),
        f9_on_hess_mean / np.maximum(f9_off_hess_max, 1e-9),
        np.nan
    )
    crop_out["f9_on_off_rs_ratio"] = np.where(
        np.isfinite(f9_on_rs_mean) & np.isfinite(f9_off_rs_max),
        f9_on_rs_mean / np.maximum(f9_off_rs_max, 1e-9),
        np.nan
    )

    F9_FRAMES.append(crop_out)
    log(f"  F9 crop {cid}: {n} candidates done")

    del crop_out
    del panel_mf_center, panel_hess_center, panel_rs_center
    gc.collect()

f9_df = pd.concat(F9_FRAMES, ignore_index=True) if F9_FRAMES else pd.DataFrame(columns=["union_id"])
log(f"Family 9 done: {len([c for c in f9_df.columns if c != 'union_id'])} cols in {time.perf_counter() - t0_f9:.1f}s")

In [ ]:
# ── Cell 31: Family 9B removed in RAW-LOCAL rewrite ───────────────────────────
# Removed because it used GT-derived per-well reference profiles, which are not
# deployment-safe and leak annotation-derived template information into features.

log("="*70 + "\nFAMILY 9B: REMOVED")
f9b_df=pd.DataFrame({"union_id":candidate_union["union_id"].tolist()})
log("Family 9B intentionally removed: no GT-derived barcode reference profiles.")

In [ ]:
# ── Cell 32: Family 10 — RAW local background / noise / clutter characterization ──
# Batched by crop: cache loaded once per crop.

log("="*70+"\nFAMILY 10: RAW local background / noise / clutter")
t0_f10 = time.perf_counter()
F10_ROWS = []

for cid, cu_crop in candidate_union.groupby("crop_id", sort=True):
    cid  = str(cid)
    crow = CROP_LOOKUP.get(cid)
    if crow is None:
        for _, urow in cu_crop.iterrows():
            F10_ROWS.append({"union_id": urow["union_id"]})
        continue

    pmap    = CROP_PANEL_CACHE.get(cid, {})
    primary = next((pk for pk in CFG["barcode_panels"] if pk in pmap), None)
    if primary is None:
        for _, urow in cu_crop.iterrows():
            F10_ROWS.append({"union_id": urow["union_id"]})
        continue

    cache = get_panel_cache_for_use(cid, primary)
    if cache is None:
        for _, urow in cu_crop.iterrows():
            F10_ROWS.append({"union_id": urow["union_id"]})
        continue

    try:
        raw_img = cache.get("raw")
        hp_img  = cache.get("highpass")
        hpsd    = cache.get("hp_local_std9")
        base    = raw_img
        if base is None:
            for _, urow in cu_crop.iterrows():
                F10_ROWS.append({"union_id": urow["union_id"]})
            continue

        H, W = base.shape
        oy = float(crow["well_ymin_px"])
        ox = float(crow["well_xmin_px"])

        for _, urow in cu_crop.iterrows():
            uid = urow["union_id"]
            wy  = float(urow["union_medoid_well_y_px"])
            wx  = float(urow["union_medoid_well_x_px"])
            cy  = wy - oy
            cx  = wx - ox
            iy  = min(max(int(round(cy)), 0), H-1)
            ix  = min(max(int(round(cx)), 0), W-1)
            f   = {"union_id": uid}

            vr2, cov2 = extract_annulus(base, cy, cx, _RING2_MASK, _R_MAX)
            if cov2 > 0.3 and len(vr2) > 5:
                f["bg_ring2_mean"]    = float(np.mean(vr2))
                f["bg_ring2_std"]     = float(np.std(vr2))
                f["bg_ring2_mad"]     = robust_mad(vr2)
                f["bg_ring2_skew"]    = safe_skew(vr2)
                f["bg_ring2_kurt"]    = safe_kurt(vr2)
                f["bg_ring2_entropy"] = patch_entropy_raw(vr2)
            else:
                for k in ["bg_ring2_mean","bg_ring2_std","bg_ring2_mad","bg_ring2_skew","bg_ring2_kurt","bg_ring2_entropy"]:
                    f[k] = np.nan

            pf = local_plane_fit(base, cy, cx, r=12, inner_excl=_IR)
            for k, v in pf.items():
                f[f"bg_{k}"] = v

            r2f = 12
            y0=max(0,iy-r2f); y1=min(H,iy+r2f+1)
            x0=max(0,ix-r2f); x1=min(W,ix+r2f+1)
            if (y1-y0) >= 5 and (x1-x0) >= 5:
                pp  = base[y0:y1, x0:x1].astype(np.float32)
                gy, gx = np.gradient(pp)
                gm  = np.sqrt(gy*gy + gx*gx)
                yy_g, xx_g = np.indices(pp.shape)
                cy_l, cx_l = iy-y0, ix-x0
                d2  = (yy_g-cy_l)**2 + (xx_g-cx_l)**2
                rm  = d2 > _IR**2
                vals = pp[rm]
                f["bg_patch_std"]     = float(np.std(vals))        if vals.size > 0 else np.nan
                f["bg_patch_mad"]     = robust_mad(vals)           if vals.size > 0 else np.nan
                f["bg_patch_entropy"] = patch_entropy_raw(vals)    if vals.size > 0 else np.nan
                f["bg_grad_mag_mean"] = float(np.mean(gm[rm]))     if rm.sum()  > 0 else np.nan
                f["bg_grad_mag_std"]  = float(np.std(gm[rm]))      if rm.sum()  > 0 else np.nan
            else:
                for k in ["bg_patch_std","bg_patch_mad","bg_patch_entropy","bg_grad_mag_mean","bg_grad_mag_std"]:
                    f[k] = np.nan

            center_raw = bilinear_sample(base, cy, cx)
            f["bg_center_minus_plane"] = center_raw - pf["plane_center_pred"] if np.isfinite(pf["plane_center_pred"]) else np.nan

            if hp_img is not None:
                hpv = bilinear_sample(hp_img, cy, cx)
                f["bg_highpass_local_snr"] = safe_div(hpv, max(bilinear_sample(hpsd,cy,cx),1e-9)) if hpsd is not None else np.nan
            else:
                f["bg_highpass_local_snr"] = np.nan

            F10_ROWS.append(f)

        log(f"  F10 crop {cid}: {len(cu_crop)} candidates done")

    finally:
        release_panel_cache(cid, primary)
        del cache
        gc.collect()

f10_df = pd.DataFrame(F10_ROWS)
log(f"Family 10 done: {len([c for c in f10_df.columns if c!='union_id'])} cols in {time.perf_counter()-t0_f10:.1f}s")

In [39]:
# ── Cell 26: Assemble full feature table and save all outputs ─────────────────
# Full-well-derived candidate/features are the deployment contract.
# NB05/NB06 should impose QA crop windows AFTER loading these saved tables.
# NB07 should load these saved full-well artifacts directly and must NOT rerun
# proposer/feature generation when the cache is present.

log("Assembling full feature table...")

feat_parts = [f1_df, f2_df, f3_df, f4_df, f5_df, f6_df, f7_df, f8_df, f9_df, f9b_df, f10_df]
features_df = feat_parts[0]

for pt in feat_parts[1:]:
    if len(pt) > 0 and "union_id" in pt.columns:
        features_df = features_df.merge(pt, on="union_id", how="left", suffixes=("", "_dup"))
        features_df = features_df.drop(columns=[c for c in features_df.columns if c.endswith("_dup")])


# ── Add well-level raw-domain background scale features ───────────────────────
# These are computed from ACTUAL full-well support and then persisted so that
# training rows (masked to QA crop windows later) and NB07 inference rows share
# identical denominators for bgscale_* and *_global_norm.
log("Computing well-level raw-domain background scales from ACTUAL full wells...")

def _iter_panel_cache_keys(pmap):
    if not isinstance(pmap, dict) or len(pmap) == 0:
        return []
    return [pk for pk in pmap.keys() if pk in CFG["barcode_panels"]]

_WELL_BG_SCALE = {}
_bg_acc = {}

for _, crow in crop_registry.iterrows():
    cid = str(crow["crop_id"])
    wid = str(crow["well_id"])
    pmap = CROP_PANEL_CACHE.get(cid, {})
    if not pmap:
        continue

    for pk in _iter_panel_cache_keys(pmap):
        cache = get_panel_cache_for_use(cid, pk)
        if not isinstance(cache, dict):
            continue
        try:
            raw = cache.get("raw")
            hp = cache.get("highpass")
            log_s3 = cache.get("log_raw_s3")

            key = (wid, pk)
            if key not in _bg_acc:
                _bg_acc[key] = {
                    "hp_sq_sum": 0.0,
                    "hp_n": 0,
                    "log_sq_sum": 0.0,
                    "log_n": 0,
                    "raw_sum": 0.0,
                    "raw_sq_sum": 0.0,
                    "raw_n": 0,
                }

            acc = _bg_acc[key]

            if raw is not None:
                raw_arr = np.asarray(raw, dtype=np.float32)
                m = np.isfinite(raw_arr)
                if np.any(m):
                    vals = raw_arr[m]
                    acc["raw_sum"] += float(np.sum(vals))
                    acc["raw_sq_sum"] += float(np.sum(vals * vals))
                    acc["raw_n"] += int(vals.size)

            if hp is not None:
                hp_arr = np.asarray(hp, dtype=np.float32)
                m = np.isfinite(hp_arr)
                if np.any(m):
                    vals = hp_arr[m]
                    acc["hp_sq_sum"] += float(np.sum(vals * vals))
                    acc["hp_n"] += int(vals.size)

            if log_s3 is not None:
                lg_arr = np.asarray(log_s3, dtype=np.float32)
                m = np.isfinite(lg_arr)
                if np.any(m):
                    vals = lg_arr[m]
                    acc["log_sq_sum"] += float(np.sum(vals * vals))
                    acc["log_n"] += int(vals.size)
        finally:
            release_panel_cache(cid, pk)
            del cache

for (wid, pk), acc in _bg_acc.items():
    if wid not in _WELL_BG_SCALE:
        _WELL_BG_SCALE[wid] = {}

    hp_energy = np.nan
    if acc["hp_n"] > 0:
        hp_energy = float(np.sqrt(acc["hp_sq_sum"] / max(acc["hp_n"], 1)))

    log_energy_s3 = np.nan
    if acc["log_n"] > 0:
        log_energy_s3 = float(np.sqrt(acc["log_sq_sum"] / max(acc["log_n"], 1)))

    raw_std = np.nan
    if acc["raw_n"] > 1:
        raw_mean = acc["raw_sum"] / acc["raw_n"]
        raw_var = (acc["raw_sq_sum"] / acc["raw_n"]) - (raw_mean * raw_mean)
        raw_std = float(np.sqrt(max(raw_var, 0.0)))

    _WELL_BG_SCALE[wid][pk] = {
        "hp_energy": hp_energy,
        "log_energy_s3": log_energy_s3,
        "raw_std": raw_std,
    }

if len(features_df) and len(candidate_union):
    log("Adding per-candidate global-normalization features from persisted full-well support...")

    union_meta = candidate_union[["union_id", "crop_id", "well_id"]].copy()

    crop_primary_panel = {}
    for _, crow in crop_registry.iterrows():
        cid = str(crow["crop_id"])
        pmap = CROP_PANEL_CACHE.get(cid, {})
        available_panels = _iter_panel_cache_keys(pmap)
        primary = next((pk for pk in CFG["barcode_panels"] if pk in available_panels), None)
        crop_primary_panel[cid] = primary

    union_meta["primary_panel"] = union_meta["crop_id"].astype(str).map(crop_primary_panel)

    hp_scales = []
    log_scales = []
    raw_stds = []

    for _, r in union_meta.iterrows():
        wid = str(r["well_id"])
        pk = r["primary_panel"]
        sc = _WELL_BG_SCALE.get(wid, {}).get(pk, {}) if pk is not None else {}
        hp_scales.append(float(sc.get("hp_energy", np.nan)) if sc else np.nan)
        log_scales.append(float(sc.get("log_energy_s3", np.nan)) if sc else np.nan)
        raw_stds.append(float(sc.get("raw_std", np.nan)) if sc else np.nan)

    union_meta["bgscale_hp_energy_primary"] = hp_scales
    union_meta["bgscale_log_energy_s3_primary"] = log_scales
    union_meta["bgscale_raw_std_primary"] = raw_stds

    features_df = features_df.merge(
        union_meta[
            [
                "union_id",
                "bgscale_hp_energy_primary",
                "bgscale_log_energy_s3_primary",
                "bgscale_raw_std_primary",
            ]
        ],
        on="union_id",
        how="left",
        suffixes=("", "_dup"),
    )
    features_df = features_df.drop(columns=[c for c in features_df.columns if c.endswith("_dup")])

    if "resp_highpass_center" in features_df.columns:
        features_df["resp_highpass_center_global_norm"] = (
            features_df["resp_highpass_center"].astype(float) /
            np.maximum(features_df["bgscale_hp_energy_primary"].astype(float), 1e-9)
        )

    if "resp_log_raw_s3_center" in features_df.columns:
        features_df["resp_log_raw_s3_center_global_norm"] = (
            features_df["resp_log_raw_s3_center"].astype(float) /
            np.maximum(features_df["bgscale_log_energy_s3_primary"].astype(float), 1e-9)
        )

    if "psf_raw_amplitude" in features_df.columns:
        features_df["psf_raw_amplitude_global_norm"] = (
            features_df["psf_raw_amplitude"].astype(float) /
            np.maximum(features_df["bgscale_raw_std_primary"].astype(float), 1e-9)
        )

n_feat_cols = len([c for c in features_df.columns if c != "union_id"])
log(f"Feature table: {len(features_df)} rows × {n_feat_cols} feature columns")

for nm, df in zip(
    ["F1_photometry", "F2_responses", "F3_spatial", "F4_consensus", "F5_barcode",
     "F6_regime_stub", "F7_shape", "F8_rank_stub", "F9_perchannel", "F9B_profile_stub", "F10_background"],
    feat_parts
):
    nc = len([c for c in df.columns if c != "union_id"])
    nn = df.drop(columns="union_id", errors="ignore").notna().mean().mean() if nc > 0 else np.nan
    log(f"  {nm}: {nc} cols, mean non-null={nn:.3f}")

_added_bg_cols = [
    c for c in [
        "bgscale_hp_energy_primary",
        "bgscale_log_energy_s3_primary",
        "bgscale_raw_std_primary",
        "resp_highpass_center_global_norm",
        "resp_log_raw_s3_center_global_norm",
        "psf_raw_amplitude_global_norm",
    ]
    if c in features_df.columns
]
log(f"  F26_global_bgscale: {len(_added_bg_cols)} cols -> {_added_bg_cols}")


# ── Persist per-family feature tables for debug/reuse ─────────────────────────
_feature_family_tables = {
    "F1_photometry": f1_df,
    "F2_responses": f2_df,
    "F3_spatial": f3_df,
    "F4_consensus": f4_df,
    "F5_barcode": f5_df,
    "F6_regime_stub": f6_df,
    "F7_shape": f7_df,
    "F8_rank_stub": f8_df,
    "F9_perchannel": f9_df,
    "F9B_profile_stub": f9b_df,
    "F10_background": f10_df,
}
_feature_family_paths = {}
for _fam_name, _fam_df in _feature_family_tables.items():
    _feature_family_paths[_fam_name] = str(write_table_cache(_fam_df, FEATURE_CACHE_DIR, _fam_name))
log(f"Per-family feature caches refreshed under {FEATURE_CACHE_DIR}")


# ── QA overlays on ACTUAL full wells ──────────────────────────────────────────
if CFG["write_overlays"] and len(candidate_union):
    log("Writing overlays...")
    for _, crow in crop_registry.iterrows():
        cid = str(crow["crop_id"])
        pmap = CROP_PANEL_CACHE.get(cid, {})
        available_panels = _iter_panel_cache_keys(pmap)
        if len(available_panels) == 0:
            continue

        rpk = "C1_638" if "C1_638" in available_panels else sorted(available_panels)[0]
        cache = get_panel_cache_for_use(cid, rpk)
        if not isinstance(cache, dict):
            continue

        try:
            base = cache.get("raw")
            if base is None:
                continue

            sub = candidate_union[candidate_union["crop_id"] == cid]
            if len(sub) == 0:
                continue

            fig, ax = plt.subplots(figsize=(7, 7))
            ax.imshow(
                base,
                cmap="gray",
                vmin=float(np.percentile(base, 1)),
                vmax=float(np.percentile(base, 99)),
                origin="upper",
            )
            py = sub["union_centroid_well_y_px"].to_numpy() - float(crow["well_ymin_px"])
            px = sub["union_centroid_well_x_px"].to_numpy() - float(crow["well_xmin_px"])
            ax.scatter(px, py, s=10, facecolors="none", edgecolors="yellow", linewidths=0.6)
            ax.set_title(f"{cid[:28]} | {len(sub)} candidates")
            ax.axis("off")
            fig.tight_layout()
            fig.savefig(OVERLAY_DIR / f"{cid}_union_overlay.png", dpi=140, bbox_inches="tight")
            plt.close(fig)
        finally:
            release_panel_cache(cid, rpk)
            del cache

    log(f"Overlays -> {OVERLAY_DIR}")


# ── Persist preprocessing cache manifest so future notebooks can trust disk artifacts ──
cache_manifest = {
    "run_id": RUN_ID,
    "execution_mode": CFG["execution_mode"],
    "cache_dir": str(CACHE_DIR),
    "fullwell_ids": [str(x) for x in crop_registry["crop_id"].tolist()],
    "mentor_required_panels": [str(x) for x in CFG["mentor_required_panels"]],
}
write_json(cache_manifest, MANIFEST_DIR / f"{RUN_ID}_fullwell_cache_manifest.json")


# ── Persist all outputs (deployment cache) ────────────────────────────────────
# These saved full-well tables are the contract for NB05/NB06/NB07. NB07 should
# load them directly and must not rerun proposer/feature generation when present.
paths = {}
paths["detector_cache_dir"] = str(DETECTOR_CACHE_DIR)
paths["feature_family_cache_dir"] = str(FEATURE_CACHE_DIR)

if len(candidate_raw):
    paths["candidate_raw"] = str(safe_parquet(candidate_raw, TABLES_DIR / f"{RUN_ID}_candidate_raw.parquet"))
if len(candidate_union):
    paths["candidate_union"] = str(safe_parquet(candidate_union, TABLES_DIR / f"{RUN_ID}_candidate_union.parquet"))
if len(candidate_union_membership):
    paths["candidate_union_membership"] = str(
        safe_parquet(candidate_union_membership, TABLES_DIR / f"{RUN_ID}_candidate_union_membership.parquet")
    )
if len(candidate_clusters):
    paths["candidate_clusters"] = str(safe_parquet(candidate_clusters, TABLES_DIR / f"{RUN_ID}_candidate_clusters.parquet"))
if len(features_df):
    paths["candidate_features"] = str(safe_parquet(features_df, TABLES_DIR / f"{RUN_ID}_candidate_features.parquet"))
if len(detector_summary_df):
    paths["detector_summary"] = str(
        safe_parquet(detector_summary_df, TABLES_DIR / f"{RUN_ID}_detector_run_summary.parquet")
    )
if len(off_neg_df):
    paths["off_channel_negative_candidates"] = str(
        safe_parquet(off_neg_df, TABLES_DIR / f"{RUN_ID}_off_channel_negative_candidates.parquet")
    )

paths["fullwell_registry"] = str(safe_parquet(crop_registry, TABLES_DIR / f"{RUN_ID}_fullwell_registry.parquet"))

# ── QA crop registry as "crop_registry" for NB05 manifest resolution ──────────
# NB05 looks for paths["crop_registry"] in the manifest. Point it to the QA
# crop registry (the 50-crop supervision territory), NOT the fullwell registry.
paths["crop_registry"] = str(safe_parquet(qa_crop_registry, TABLES_DIR / f"{RUN_ID}_crop_registry.parquet"))
paths["qa_crop_registry_snapshot"] = str(safe_parquet(qa_crop_registry, TABLES_DIR / f"{RUN_ID}_qa_crop_registry_snapshot.parquet"))

import json as _json
with open(TABLES_DIR / f"{RUN_ID}_well_bg_scales.json", "w") as _fp:
    _json.dump(_WELL_BG_SCALE, _fp, indent=2)
paths["well_bg_scales"] = str(TABLES_DIR / f"{RUN_ID}_well_bg_scales.json")
paths["feature_family_tables"] = _feature_family_paths

manifest = {
    "run_id": RUN_ID,
    "notebook_name": "nb03_nb04_candidate_universe_features_fullwell.ipynb",
    "execution_mode": "full_well",
    "repo_root": str(REPO_ROOT),
    "data_root": str(DATA_ROOT),
    "qa_crop_registry_path": str(QA_CROP_REGISTRY_PATH),
    "n_full_wells": int(len(crop_registry)),
    "fullwell_ids": [str(x) for x in crop_registry["crop_id"].tolist()],
    "n_qa_crops": int(len(qa_crop_registry)),
    "n_candidate_raw": int(len(candidate_raw)),
    "n_candidate_union": int(len(candidate_union)),
    "n_off_channel_negative_candidates": int(len(off_neg_df) if len(off_neg_df) else 0),
    "n_candidate_membership": int(len(candidate_union_membership) if len(candidate_union_membership) else 0),
    "n_candidate_clusters": int(len(candidate_clusters)),
    "n_features": int(n_feat_cols),
    "n_methods": int(candidate_raw["method_id"].nunique() if len(candidate_raw) else 0),
    "paths": paths,
    "config": {
        k: str(v) if not isinstance(v, (str, int, float, bool, list, dict, type(None))) else v
        for k, v in CFG.items()
    },
    "base_provenance": BASE_PROV,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
mp = write_json(manifest, MANIFEST_DIR / f"{RUN_ID}_run_manifest.json")
paths["run_manifest"] = str(mp)

log(f"Manifest -> {mp.name}")
log("All outputs saved.")
print("\nOutput files:")
print("\nOutput files:")
for k, v in paths.items():
    if k == "feature_family_tables":
        print(f"  {k:35s} ->")
        for sk, sv in v.items():
            print(f"      {sk:28s} -> {Path(str(sv)).name}")
    else:
        print(f"  {k:35s} -> {Path(str(v)).name}")

[2026-01-24 04:09:53 UTC] Assembling full feature table...
[2026-01-24 04:09:53 UTC] Computing well-level raw-domain background scales from ACTUAL full wells...
[2026-01-24 04:13:05 UTC] Adding per-candidate global-normalization features from persisted full-well support...
[2026-01-24 04:13:06 UTC] Feature table: 5955 rows × 433 feature columns
[2026-01-24 04:13:06 UTC]   F1_photometry: 146 cols, mean non-null=0.999
[2026-01-24 04:13:06 UTC]   F2_responses: 43 cols, mean non-null=0.857
[2026-01-24 04:13:06 UTC]   F3_spatial: 19 cols, mean non-null=0.810
[2026-01-24 04:13:06 UTC]   F4_consensus: 70 cols, mean non-null=0.819
[2026-01-24 04:13:06 UTC]   F5_barcode: 70 cols, mean non-null=0.999
[2026-01-24 04:13:06 UTC]   F6_regime_stub: 0 cols, mean non-null=nan
[2026-01-24 04:13:06 UTC]   F7_shape: 13 cols, mean non-null=0.998
[2026-01-24 04:13:06 UTC]   F8_rank_stub: 0 cols, mean non-null=nan
[2026-01-24 04:13:06 UTC]   F9_perchannel: 49 cols, mean non-null=1.000
[2026-01-24 04:13:06 UT

In [ ]:
# ── Cell 27: Exit checks ─────────────────────────────────────────────────────
req_raw={"raw_detection_id","run_id","dataset_id","well_id","method_id","method_family",
         "source_view_id","score_type","score_is_calibrated","detection_score_raw",
         "detection_score_norm","well_y_px","well_x_px","coord_frame_primary","timing_sec"}
req_union={"union_id","run_id","dataset_id","well_id","union_n_members",
           "union_centroid_well_y_px","union_centroid_well_x_px",
           "union_medoid_well_y_px","union_medoid_well_x_px",
           "union_radius_px","top_method_id","top_method_score_raw","top_method_score_norm",
           "proposer_support_count","proposer_support_fraction",
           "family_support_count","family_set_canonical"}

if len(candidate_raw):
    miss=sorted(req_raw-set(candidate_raw.columns))
    assert not miss, f"candidate_raw missing: {miss}"
    assert candidate_raw["raw_detection_id"].is_unique,"raw_detection_id not unique"
    assert set(candidate_raw["coord_frame_primary"].dropna().unique())=={"well"}

if len(candidate_union):
    miss=sorted(req_union-set(candidate_union.columns))
    assert not miss, f"candidate_union missing: {miss}"
    assert candidate_union["union_id"].is_unique,"union_id not unique"

if len(candidate_clusters):
    assert candidate_clusters["cluster_id"].is_unique,"cluster_id not unique"

if len(features_df):
    assert features_df["union_id"].is_unique,"features union_id not unique"
    missing_feats=set(candidate_union["union_id"])-set(features_df["union_id"])
    assert not missing_feats,f"{len(missing_feats)} union candidates missing features"

# exact parity support columns required for downstream NB06/NB07 reuse
for _req in ["bgscale_hp_energy_primary","bgscale_log_energy_s3_primary","bgscale_raw_std_primary"]:
    assert _req in features_df.columns, f"Missing required full-well support feature: {_req}"
for _req in ["resp_highpass_center_global_norm","resp_log_raw_s3_center_global_norm","psf_raw_amplitude_global_norm"]:
    if _req in features_df.columns:
        assert np.isfinite(features_df[_req].dropna()).all(), f"Non-finite values in {_req}"

log("All exit checks passed.")
print(f"candidate_raw:      {len(candidate_raw)}")
print(f"candidate_union:    {len(candidate_union)}")
print(f"candidate_clusters: {len(candidate_clusters)}")
print(f"features:           {len(features_df)} rows × {n_feat_cols} cols")
print(f"n_methods:          {candidate_raw['method_id'].nunique() if len(candidate_raw) else 0}")
print(f"mean_proposals/fullwell_unit:{len(candidate_union)/max(len(crop_registry),1):.1f}")
print("execution_registry_mode: ACTUAL full wells only; QA crop registry not used for proposer/feature computation")

print("\nFeature family sizes:")
for nm,df in zip(["F1","F2","F3","F4","F5_barcode","F6","F7","F8","F9","F9B","F10"],feat_parts):
    nc=len([c for c in df.columns if c!="union_id"])
    print(f"  {nm:20s}: {nc:4d} cols")
print(f"  {'TOTAL':20s}: {n_feat_cols:4d} cols")
